In [ ]:
# =============================================================================
# Cell 1 — Runtime and configuration
# =============================================================================
#
# Purpose:
#   1. Define the complete central training configuration.
#   2. Detect Google Colab, Python, CPU, RAM, disk, and NVIDIA GPU availability.
#   3. Protect the user from accidentally starting a full 50,000-sample run
#      without a supported GPU.
#   4. Establish deterministic random seeds.
#   5. Print an estimated resource plan.
#
# Important:
#   - This cell does NOT modify the system Python.
#   - Python 3.11 will be installed into /content/pingo-env in Cell 3.
#   - QUICK_TEST_MODE=True is the safe default.
#   - To run full training, explicitly change:
#
#         QUICK_TEST_MODE = False
#         ALLOW_FULL_CPU_RUN = False
#
#     A CUDA GPU is strongly recommended for the full run.
#
# =============================================================================

from __future__ import annotations

import dataclasses
import datetime as dt
import hashlib
import json
import os
import platform
import random
import shutil
import subprocess
import sys
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, Optional


# -----------------------------------------------------------------------------
# User-controlled safety switches
# -----------------------------------------------------------------------------

# Keep True for installation verification and a small smoke-test training run.
QUICK_TEST_MODE = True

# Full dataset generation target. This is used only when QUICK_TEST_MODE=False.
POSITIVE_SYNTHETIC_TARGET = 50_000

# Hard upper limit supported by this notebook configuration.
POSITIVE_SYNTHETIC_MAX = 100_000

# Existing completed data will be reused unless this is explicitly changed.
FORCE_REBUILD = False

# Resume feature extraction and training from valid checkpoints when available.
RESUME_TRAINING = True

# Ingest real Pingo recordings when supplied later.
USE_REAL_RECORDINGS = True

# Run false-positive mining after first-pass model training.
USE_HARD_NEGATIVE_MINING = True

# Enable optional VAD gating during deployment evaluation.
USE_VAD_DURING_INFERENCE = True

# Full training on CPU is intentionally blocked by default.
# Changing this to True is not recommended for Colab CPU runtimes.
ALLOW_FULL_CPU_RUN = False

# When True, later cells may copy checkpoints to Google Drive after Drive
# has been mounted by the user.
ENABLE_GOOGLE_DRIVE_BACKUP = False

# Optional Drive directory. It will not be created or mounted in this cell.
GOOGLE_DRIVE_BACKUP_DIR = (
    "/content/drive/MyDrive/pingo_training_backup"
)


# -----------------------------------------------------------------------------
# Helper functions
# -----------------------------------------------------------------------------

def run_command(
    command: list[str],
    *,
    check: bool = False,
    timeout: int = 30,
) -> subprocess.CompletedProcess[str]:
    """
    Run a command without shell=True.

    This avoids shell quoting problems and prevents accidental shell injection.
    """
    try:
        return subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=check,
            timeout=timeout,
            env=os.environ.copy(),
        )
    except FileNotFoundError as exc:
        return subprocess.CompletedProcess(
            args=command,
            returncode=127,
            stdout="",
            stderr=f"Command not found: {command[0]} ({exc})",
        )
    except subprocess.TimeoutExpired as exc:
        return subprocess.CompletedProcess(
            args=command,
            returncode=124,
            stdout=exc.stdout or "",
            stderr=f"Command timed out after {timeout} seconds.",
        )


def command_exists(command: str) -> bool:
    """Return True when an executable exists on PATH."""
    return shutil.which(command) is not None


def bytes_to_gib(value: int | float) -> float:
    """Convert bytes to GiB."""
    return float(value) / (1024**3)


def detect_colab() -> bool:
    """Detect whether the cell is running inside Google Colab."""
    return (
        "google.colab" in sys.modules
        or os.environ.get("COLAB_RELEASE_TAG") is not None
        or Path("/content").exists()
    )


def detect_gpu() -> Dict[str, Any]:
    """
    Detect NVIDIA GPU availability using nvidia-smi.

    PyTorch is not imported here because its compatible version will be
    resolved and installed inside the isolated environment in later cells.
    """
    result: Dict[str, Any] = {
        "available": False,
        "name": None,
        "driver_version": None,
        "memory_total_mib": None,
        "memory_free_mib": None,
        "compute_capability": None,
        "diagnostic": None,
    }

    if not command_exists("nvidia-smi"):
        result["diagnostic"] = "nvidia-smi is not installed or unavailable."
        return result

    query = run_command(
        [
            "nvidia-smi",
            "--query-gpu=name,driver_version,memory.total,memory.free,"
            "compute_cap",
            "--format=csv,noheader,nounits",
        ],
        timeout=20,
    )

    if query.returncode != 0 or not query.stdout.strip():
        result["diagnostic"] = (
            query.stderr.strip()
            or "nvidia-smi returned no GPU information."
        )
        return result

    first_gpu = query.stdout.strip().splitlines()[0]
    fields = [field.strip() for field in first_gpu.split(",")]

    if len(fields) < 5:
        result["diagnostic"] = (
            f"Unexpected nvidia-smi output: {first_gpu!r}"
        )
        return result

    try:
        result.update(
            {
                "available": True,
                "name": fields[0],
                "driver_version": fields[1],
                "memory_total_mib": int(float(fields[2])),
                "memory_free_mib": int(float(fields[3])),
                "compute_capability": fields[4],
                "diagnostic": None,
            }
        )
    except (TypeError, ValueError) as exc:
        result["diagnostic"] = (
            f"Could not parse nvidia-smi output: {exc}. "
            f"Raw output: {first_gpu!r}"
        )

    return result


def get_memory_info() -> Dict[str, float]:
    """Read host memory information from /proc/meminfo where available."""
    memory = {
        "total_gib": 0.0,
        "available_gib": 0.0,
    }

    meminfo_path = Path("/proc/meminfo")
    if not meminfo_path.exists():
        return memory

    values: Dict[str, int] = {}

    for line in meminfo_path.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines():
        if ":" not in line:
            continue

        key, raw_value = line.split(":", 1)
        parts = raw_value.strip().split()

        if not parts:
            continue

        try:
            # Linux reports these values in KiB.
            values[key] = int(parts[0]) * 1024
        except ValueError:
            continue

    memory["total_gib"] = bytes_to_gib(values.get("MemTotal", 0))
    memory["available_gib"] = bytes_to_gib(
        values.get("MemAvailable", 0)
    )

    return memory


def get_disk_info(path: str = "/content") -> Dict[str, float]:
    """Return total, used, and free disk information for a path."""
    target = Path(path)

    if not target.exists():
        target = Path.cwd()

    usage = shutil.disk_usage(target)

    return {
        "path": str(target.resolve()),
        "total_gib": bytes_to_gib(usage.total),
        "used_gib": bytes_to_gib(usage.used),
        "free_gib": bytes_to_gib(usage.free),
    }


def stable_config_hash(config_data: Dict[str, Any]) -> str:
    """Create a deterministic SHA-256 hash of the central configuration."""
    encoded = json.dumps(
        config_data,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    ).encode("utf-8")

    return hashlib.sha256(encoded).hexdigest()


# -----------------------------------------------------------------------------
# Runtime detection
# -----------------------------------------------------------------------------

IS_COLAB = detect_colab()
GPU_INFO = detect_gpu()
MEMORY_INFO = get_memory_info()
DISK_INFO = get_disk_info("/content")

CURRENT_UTC = dt.datetime.now(dt.timezone.utc)
CURRENT_UTC_ISO = CURRENT_UTC.isoformat()


# -----------------------------------------------------------------------------
# Central configuration dataclass
# -----------------------------------------------------------------------------

@dataclass(frozen=True)
class PingoTrainingConfig:
    # -------------------------------------------------------------------------
    # Model identity
    # -------------------------------------------------------------------------
    WAKE_WORD: str = "pingo"
    MODEL_NAME: str = "pingo"
    RANDOM_SEED: int = 42

    # -------------------------------------------------------------------------
    # Audio format
    # -------------------------------------------------------------------------
    SAMPLE_RATE: int = 16_000
    CHANNELS: int = 1
    SAMPLE_WIDTH: int = 2  # Signed 16-bit PCM = 2 bytes per sample

    # OpenWakeWord commonly evaluates audio using approximately 80 ms chunks,
    # but the classifier's actual feature context is repository-dependent.
    # Cell 4 and Cell 16 must inspect upstream model metadata and replace or
    # validate this value before feature generation.
    STREAM_CHUNK_SAMPLES: int = 1_280
    STREAM_CHUNK_DURATION_SECONDS: float = 0.080

    # Safe provisional clip duration for synthetic generation and normalization.
    # The exact classifier-compatible feature context will be resolved from the
    # checked-out OpenWakeWord repository before training.
    CLIP_DURATION_SECONDS: float = 2.0

    # -------------------------------------------------------------------------
    # Dataset scale
    # -------------------------------------------------------------------------
    POSITIVE_SYNTHETIC_TARGET: int = POSITIVE_SYNTHETIC_TARGET
    POSITIVE_SYNTHETIC_MAX: int = POSITIVE_SYNTHETIC_MAX

    # Small values used only in smoke-test mode.
    QUICK_POSITIVE_SYNTHETIC_TARGET: int = 96
    QUICK_HARD_NEGATIVE_SYNTHETIC_TARGET: int = 96
    QUICK_PUBLIC_NEGATIVE_TARGET: int = 192
    QUICK_MAX_TRAINING_STEPS: int = 100

    # Full-run dataset objectives. Actual counts are reported from manifests;
    # these numbers are configuration objectives, not fabricated results.
    FULL_HARD_NEGATIVE_SYNTHETIC_TARGET: int = 20_000
    FULL_GENERAL_NEGATIVE_CLIP_TARGET: int = 100_000
    FULL_FINAL_TEST_CASE_TARGET: int = 50_000

    # -------------------------------------------------------------------------
    # Dataset splits
    # -------------------------------------------------------------------------
    TRAIN_FRACTION: float = 0.80
    VALIDATION_FRACTION: float = 0.08
    CALIBRATION_FRACTION: float = 0.06
    TEST_FRACTION: float = 0.06

    # Mining data is kept separate from the above final partitions.
    MINING_POOL_FRACTION_OF_NEGATIVES: float = 0.10

    # -------------------------------------------------------------------------
    # Activation policy
    # -------------------------------------------------------------------------
    # Default policy:
    # Any natural utterance clearly containing the spoken word "Pingo" may
    # activate. Therefore, "Hey Pingo" and "Pingo play" are not automatically
    # treated as negative examples.
    ACTIVATION_POLICY: str = (
        "ACTIVATE_WHEN_CLEARLY_SPOKEN_PINGO_IS_PRESENT"
    )

    POSITIVE_PRONUNCIATION_VARIANTS: tuple[str, ...] = (
        "pingo",
        "pin-go",
        "ping-go",
        "peen-go",
    )

    CONFUSABLE_NEGATIVE_PHRASES: tuple[str, ...] = (
        "bingo",
        "dingo",
        "lingo",
        "gringo",
        "ping",
        "go",
        "pin go",
        "pinko",
        "pinku",
        "single",
        "window",
        "ringo",
        "pingu",
    )

    VALID_CONTAINING_PHRASES: tuple[str, ...] = (
        "pingo play",
        "hey pingo",
        "play pingo",
        "pingo song",
    )

    # -------------------------------------------------------------------------
    # Runtime switches
    # -------------------------------------------------------------------------
    QUICK_TEST_MODE: bool = QUICK_TEST_MODE
    FORCE_REBUILD: bool = FORCE_REBUILD
    RESUME_TRAINING: bool = RESUME_TRAINING
    USE_REAL_RECORDINGS: bool = USE_REAL_RECORDINGS
    USE_HARD_NEGATIVE_MINING: bool = USE_HARD_NEGATIVE_MINING
    USE_VAD_DURING_INFERENCE: bool = USE_VAD_DURING_INFERENCE
    ALLOW_FULL_CPU_RUN: bool = ALLOW_FULL_CPU_RUN
    ENABLE_GOOGLE_DRIVE_BACKUP: bool = ENABLE_GOOGLE_DRIVE_BACKUP

    # -------------------------------------------------------------------------
    # Target operating characteristics
    # -------------------------------------------------------------------------
    TARGET_FALSE_ACTIVATIONS_PER_HOUR: float = 0.5
    TARGET_FALSE_REJECT_RATE: float = 0.05

    # These are search ranges, not final selected values.
    THRESHOLD_SEARCH_MIN: float = 0.05
    THRESHOLD_SEARCH_MAX: float = 0.95
    THRESHOLD_SEARCH_STEP: float = 0.01

    REQUIRED_HITS_SEARCH_VALUES: tuple[int, ...] = (1, 2, 3, 4, 5)
    SMOOTHING_WINDOW_SEARCH_VALUES: tuple[int, ...] = (1, 2, 3, 4, 5)
    VAD_THRESHOLD_SEARCH_VALUES: tuple[float, ...] = (
        0.30,
        0.40,
        0.50,
        0.60,
    )
    COOLDOWN_SEARCH_SECONDS: tuple[float, ...] = (
        1.0,
        1.5,
        2.0,
        3.0,
    )

    # -------------------------------------------------------------------------
    # Training defaults
    # -------------------------------------------------------------------------
    TRAIN_BATCH_SIZE: int = 256
    VALIDATION_BATCH_SIZE: int = 512
    FEATURE_BATCH_SIZE: int = 256
    NUM_WORKERS: int = 2
    LEARNING_RATE: float = 1e-3
    WEIGHT_DECAY: float = 1e-5
    MAX_EPOCHS: int = 50
    EARLY_STOPPING_PATIENCE: int = 7
    GRADIENT_CLIP_NORM: float = 5.0
    USE_MIXED_PRECISION: bool = True

    # ONNX Runtime on Raspberry Pi should support this opset reliably.
    # The export cell will validate compatibility before accepting the model.
    ONNX_OPSET: int = 17
    ONNX_OUTPUT_TOLERANCE: float = 1e-4

    # -------------------------------------------------------------------------
    # Root paths
    # -------------------------------------------------------------------------
    CONTENT_ROOT: str = "/content"
    ENV_DIR: str = "/content/pingo-env"
    TRAINING_ROOT: str = "/content/pingo_training"

    CONFIG_DIR: str = "/content/pingo_training/config"
    CACHE_DIR: str = "/content/pingo_training/cache"
    CHECKPOINT_DIR: str = "/content/pingo_training/checkpoints"
    DATASETS_DIR: str = "/content/pingo_training/datasets"
    FEATURES_DIR: str = "/content/pingo_training/features"
    LOGS_DIR: str = "/content/pingo_training/logs"
    REPORTS_DIR: str = "/content/pingo_training/reports"
    MINING_DIR: str = "/content/pingo_training/mining"
    MODELS_DIR: str = "/content/pingo_training/models"
    OUTPUT_DIR: str = "/content/pingo_training/output"
    REPOSITORIES_DIR: str = "/content/pingo_training/repositories"

    POSITIVE_SYNTHETIC_DIR: str = (
        "/content/pingo_training/datasets/positive_synthetic"
    )
    POSITIVE_REAL_DIR: str = (
        "/content/pingo_training/datasets/positive_real"
    )
    HARD_NEGATIVE_SYNTHETIC_DIR: str = (
        "/content/pingo_training/datasets/hard_negative_synthetic"
    )
    NEGATIVE_SPEECH_DIR: str = (
        "/content/pingo_training/datasets/negative_speech"
    )
    BACKGROUND_NOISE_DIR: str = (
        "/content/pingo_training/datasets/background_noise"
    )
    MUSIC_DIR: str = "/content/pingo_training/datasets/music"
    ROOM_IMPULSE_RESPONSES_DIR: str = (
        "/content/pingo_training/datasets/room_impulse_responses"
    )

    TRAIN_SPLIT_DIR: str = "/content/pingo_training/datasets/train"
    VALIDATION_SPLIT_DIR: str = (
        "/content/pingo_training/datasets/validation"
    )
    CALIBRATION_SPLIT_DIR: str = (
        "/content/pingo_training/datasets/calibration"
    )
    TEST_SPLIT_DIR: str = "/content/pingo_training/datasets/test"
    MINING_POOL_DIR: str = (
        "/content/pingo_training/datasets/mining_pool"
    )

    OPENWAKEWORD_REPO_DIR: str = (
        "/content/pingo_training/repositories/openWakeWord"
    )
    PIPER_SAMPLE_GENERATOR_REPO_DIR: str = (
        "/content/pingo_training/repositories/piper-sample-generator"
    )

    # -------------------------------------------------------------------------
    # Expected output files
    # -------------------------------------------------------------------------
    OUTPUT_ONNX: str = "/content/pingo_training/output/pingo.onnx"
    OUTPUT_RPI_SCRIPT: str = (
        "/content/pingo_training/output/test_pingo_raspberry_pi.py"
    )
    OUTPUT_RPI_REQUIREMENTS: str = (
        "/content/pingo_training/output/requirements-rpi.txt"
    )
    OUTPUT_RUNTIME_CONFIG: str = (
        "/content/pingo_training/output/recommended_runtime_config.json"
    )
    RELEASE_ZIP: str = "/content/pingo_training/pingo_release.zip"

    TRAINING_CONFIG_JSON: str = (
        "/content/pingo_training/config/training_config.json"
    )
    RESOLVED_ENVIRONMENT_JSON: str = (
        "/content/pingo_training/config/resolved_environment.json"
    )
    DATASET_MANIFEST_JSONL: str = (
        "/content/pingo_training/reports/dataset_manifest.jsonl"
    )
    DATASET_LICENSE_MANIFEST_CSV: str = (
        "/content/pingo_training/reports/dataset_license_manifest.csv"
    )
    SPLIT_MANIFEST_JSONL: str = (
        "/content/pingo_training/reports/dataset_split_manifest.jsonl"
    )
    THRESHOLD_REPORT_CSV: str = (
        "/content/pingo_training/reports/threshold_report.csv"
    )
    EVALUATION_REPORT_JSON: str = (
        "/content/pingo_training/reports/evaluation_report.json"
    )

    GOOGLE_DRIVE_BACKUP_DIR: str = GOOGLE_DRIVE_BACKUP_DIR

    # -------------------------------------------------------------------------
    # Reproducibility metadata
    # -------------------------------------------------------------------------
    CREATED_AT_UTC: str = CURRENT_UTC_ISO
    CONFIG_SCHEMA_VERSION: str = "1.0.0"


CONFIG = PingoTrainingConfig()


# -----------------------------------------------------------------------------
# Validate configuration immediately
# -----------------------------------------------------------------------------

def validate_configuration(config: PingoTrainingConfig) -> None:
    """Raise a clear error when central configuration is inconsistent."""

    errors: list[str] = []

    if config.WAKE_WORD.lower().strip() != "pingo":
        errors.append(
            "WAKE_WORD must remain exactly 'pingo' for this notebook."
        )

    if config.MODEL_NAME.lower().strip() != "pingo":
        errors.append(
            "MODEL_NAME must remain exactly 'pingo' for the required output."
        )

    if config.SAMPLE_RATE != 16_000:
        errors.append("SAMPLE_RATE must be 16000 Hz.")

    if config.CHANNELS != 1:
        errors.append("CHANNELS must be 1 for mono audio.")

    if config.SAMPLE_WIDTH != 2:
        errors.append(
            "SAMPLE_WIDTH must be 2 bytes for signed 16-bit PCM."
        )

    if config.STREAM_CHUNK_SAMPLES <= 0:
        errors.append("STREAM_CHUNK_SAMPLES must be positive.")

    calculated_chunk_duration = (
        config.STREAM_CHUNK_SAMPLES / config.SAMPLE_RATE
    )

    if abs(
        calculated_chunk_duration
        - config.STREAM_CHUNK_DURATION_SECONDS
    ) > 1e-9:
        errors.append(
            "STREAM_CHUNK_DURATION_SECONDS does not match "
            "STREAM_CHUNK_SAMPLES / SAMPLE_RATE."
        )

    split_sum = (
        config.TRAIN_FRACTION
        + config.VALIDATION_FRACTION
        + config.CALIBRATION_FRACTION
        + config.TEST_FRACTION
    )

    if abs(split_sum - 1.0) > 1e-9:
        errors.append(
            f"Dataset split fractions must total 1.0, found {split_sum:.6f}."
        )

    if config.POSITIVE_SYNTHETIC_TARGET < 1:
        errors.append("POSITIVE_SYNTHETIC_TARGET must be at least 1.")

    if (
        config.POSITIVE_SYNTHETIC_TARGET
        > config.POSITIVE_SYNTHETIC_MAX
    ):
        errors.append(
            "POSITIVE_SYNTHETIC_TARGET cannot exceed "
            "POSITIVE_SYNTHETIC_MAX."
        )

    if not 0.0 < config.TARGET_FALSE_REJECT_RATE < 1.0:
        errors.append(
            "TARGET_FALSE_REJECT_RATE must be between 0 and 1."
        )

    if config.TARGET_FALSE_ACTIVATIONS_PER_HOUR < 0:
        errors.append(
            "TARGET_FALSE_ACTIVATIONS_PER_HOUR cannot be negative."
        )

    if (
        config.THRESHOLD_SEARCH_MIN <= 0
        or config.THRESHOLD_SEARCH_MAX >= 1
        or config.THRESHOLD_SEARCH_MIN
        >= config.THRESHOLD_SEARCH_MAX
    ):
        errors.append("Threshold search range is invalid.")

    if Path(config.OUTPUT_ONNX).name != "pingo.onnx":
        errors.append(
            "OUTPUT_ONNX must end with the exact filename pingo.onnx."
        )

    if errors:
        formatted = "\n".join(f"  - {error}" for error in errors)
        raise ValueError(
            "Central configuration validation failed:\n"
            f"{formatted}"
        )


validate_configuration(CONFIG)


# -----------------------------------------------------------------------------
# Set deterministic process-level seeds
# -----------------------------------------------------------------------------

os.environ["PYTHONHASHSEED"] = str(CONFIG.RANDOM_SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

random.seed(CONFIG.RANDOM_SEED)

# NumPy, PyTorch, TensorFlow, and Torchaudio are deliberately not imported here.
# Their seeds will be configured after compatible versions have been installed
# inside /content/pingo-env.


# -----------------------------------------------------------------------------
# Resolve effective sample targets
# -----------------------------------------------------------------------------

if CONFIG.QUICK_TEST_MODE:
    EFFECTIVE_POSITIVE_SYNTHETIC_TARGET = (
        CONFIG.QUICK_POSITIVE_SYNTHETIC_TARGET
    )
    EFFECTIVE_HARD_NEGATIVE_SYNTHETIC_TARGET = (
        CONFIG.QUICK_HARD_NEGATIVE_SYNTHETIC_TARGET
    )
    EFFECTIVE_PUBLIC_NEGATIVE_TARGET = (
        CONFIG.QUICK_PUBLIC_NEGATIVE_TARGET
    )
    EFFECTIVE_FINAL_TEST_CASE_TARGET = None
else:
    EFFECTIVE_POSITIVE_SYNTHETIC_TARGET = (
        CONFIG.POSITIVE_SYNTHETIC_TARGET
    )
    EFFECTIVE_HARD_NEGATIVE_SYNTHETIC_TARGET = (
        CONFIG.FULL_HARD_NEGATIVE_SYNTHETIC_TARGET
    )
    EFFECTIVE_PUBLIC_NEGATIVE_TARGET = (
        CONFIG.FULL_GENERAL_NEGATIVE_CLIP_TARGET
    )
    EFFECTIVE_FINAL_TEST_CASE_TARGET = (
        CONFIG.FULL_FINAL_TEST_CASE_TARGET
    )


# -----------------------------------------------------------------------------
# Full-run GPU safety gate
# -----------------------------------------------------------------------------

if (
    not CONFIG.QUICK_TEST_MODE
    and not GPU_INFO["available"]
    and not CONFIG.ALLOW_FULL_CPU_RUN
):
    raise RuntimeError(
        "\n"
        "FULL TRAINING BLOCKED: No NVIDIA GPU was detected.\n\n"
        "The full configuration requests at least "
        f"{CONFIG.POSITIVE_SYNTHETIC_TARGET:,} positive synthetic examples, "
        "large negative datasets, feature extraction, hard-negative mining, "
        "and extensive evaluation. Running this accidentally on a Colab CPU "
        "runtime would be impractical.\n\n"
        "Open:\n"
        "  Runtime -> Change runtime type -> Hardware accelerator -> GPU\n\n"
        "Then rerun this cell.\n\n"
        "For installation and pipeline smoke testing, set:\n"
        "  QUICK_TEST_MODE = True\n\n"
        "ALLOW_FULL_CPU_RUN=True can override this protection, but that is "
        "not recommended."
    )


# -----------------------------------------------------------------------------
# Resource estimates
# -----------------------------------------------------------------------------

def estimate_resources(
    positive_count: int,
    negative_count: int,
    clip_duration_seconds: float,
    sample_rate: int,
    sample_width: int,
    channels: int,
) -> Dict[str, Any]:
    """
    Estimate raw audio and feature storage.

    These are planning estimates only. Actual storage depends on compression,
    augmentation policy, feature dimensions, cache format, and dataset sources.
    """
    bytes_per_clip_pcm = (
        clip_duration_seconds
        * sample_rate
        * sample_width
        * channels
    )

    total_clip_count = positive_count + negative_count
    raw_pcm_bytes = total_clip_count * bytes_per_clip_pcm

    # WAV headers and metadata add little per clip, but a very large number of
    # small files can consume significantly more filesystem space.
    estimated_wav_storage = raw_pcm_bytes * 1.20

    # OpenWakeWord feature storage varies with feature context and dtype.
    # Use a conservative range rather than pretending an exact value is known
    # before inspecting the current repository architecture.
    estimated_feature_storage_low = raw_pcm_bytes * 0.50
    estimated_feature_storage_high = raw_pcm_bytes * 2.50

    return {
        "positive_count": positive_count,
        "negative_count": negative_count,
        "total_clip_count": total_clip_count,
        "clip_duration_seconds": clip_duration_seconds,
        "raw_pcm_gib": bytes_to_gib(raw_pcm_bytes),
        "estimated_wav_storage_gib": bytes_to_gib(
            estimated_wav_storage
        ),
        "estimated_feature_storage_low_gib": bytes_to_gib(
            estimated_feature_storage_low
        ),
        "estimated_feature_storage_high_gib": bytes_to_gib(
            estimated_feature_storage_high
        ),
        "expected_peak_host_ram_gib": (
            "Approximately 6–14 GiB with streaming and bounded batches; "
            "later cells must avoid loading all audio/features into RAM."
        ),
        "expected_gpu_memory_gib": (
            "Architecture-dependent; approximately 4–12 GiB is a practical "
            "planning range. Batch size must be reduced after real inspection "
            "if the selected GPU has less memory."
        ),
    }


RESOURCE_ESTIMATE = estimate_resources(
    positive_count=EFFECTIVE_POSITIVE_SYNTHETIC_TARGET,
    negative_count=(
        EFFECTIVE_HARD_NEGATIVE_SYNTHETIC_TARGET
        + EFFECTIVE_PUBLIC_NEGATIVE_TARGET
    ),
    clip_duration_seconds=CONFIG.CLIP_DURATION_SECONDS,
    sample_rate=CONFIG.SAMPLE_RATE,
    sample_width=CONFIG.SAMPLE_WIDTH,
    channels=CONFIG.CHANNELS,
)


# -----------------------------------------------------------------------------
# Serialize an in-memory configuration preview
# -----------------------------------------------------------------------------

CONFIG_DICT = asdict(CONFIG)
CONFIG_DICT["RUNTIME_DETECTION"] = {
    "is_colab": IS_COLAB,
    "system_python": sys.version,
    "system_python_executable": sys.executable,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "processor": platform.processor(),
    "cpu_count": os.cpu_count(),
    "gpu": GPU_INFO,
    "memory": MEMORY_INFO,
    "disk": DISK_INFO,
}
CONFIG_DICT["EFFECTIVE_TARGETS"] = {
    "positive_synthetic": EFFECTIVE_POSITIVE_SYNTHETIC_TARGET,
    "hard_negative_synthetic": (
        EFFECTIVE_HARD_NEGATIVE_SYNTHETIC_TARGET
    ),
    "public_negative": EFFECTIVE_PUBLIC_NEGATIVE_TARGET,
    "final_test_cases": EFFECTIVE_FINAL_TEST_CASE_TARGET,
}
CONFIG_DICT["RESOURCE_ESTIMATE"] = RESOURCE_ESTIMATE

CONFIG_SHA256 = stable_config_hash(CONFIG_DICT)


# -----------------------------------------------------------------------------
# Human-readable runtime report
# -----------------------------------------------------------------------------

print("=" * 88)
print("PINGO OPENWAKEWORD TRAINING — RUNTIME AND CONFIGURATION")
print("=" * 88)

print("\n[Runtime]")
print(f"Google Colab detected     : {IS_COLAB}")
print(f"System Python version     : {sys.version.splitlines()[0]}")
print(f"System Python executable  : {sys.executable}")
print(f"Operating system          : {platform.platform()}")
print(f"Machine architecture      : {platform.machine()}")
print(f"Logical CPU count         : {os.cpu_count()}")

print("\n[Memory]")
print(
    f"Total RAM                 : "
    f"{MEMORY_INFO['total_gib']:.2f} GiB"
)
print(
    f"Available RAM             : "
    f"{MEMORY_INFO['available_gib']:.2f} GiB"
)

print("\n[Disk]")
print(f"Checked filesystem        : {DISK_INFO['path']}")
print(
    f"Total disk                : "
    f"{DISK_INFO['total_gib']:.2f} GiB"
)
print(
    f"Used disk                 : "
    f"{DISK_INFO['used_gib']:.2f} GiB"
)
print(
    f"Free disk                 : "
    f"{DISK_INFO['free_gib']:.2f} GiB"
)

print("\n[GPU]")
if GPU_INFO["available"]:
    print("NVIDIA GPU detected       : YES")
    print(f"GPU model                  : {GPU_INFO['name']}")
    print(
        f"NVIDIA driver             : "
        f"{GPU_INFO['driver_version']}"
    )
    print(
        f"Compute capability        : "
        f"{GPU_INFO['compute_capability']}"
    )
    print(
        f"GPU memory total          : "
        f"{GPU_INFO['memory_total_mib']} MiB"
    )
    print(
        f"GPU memory currently free : "
        f"{GPU_INFO['memory_free_mib']} MiB"
    )
else:
    print("NVIDIA GPU detected       : NO")
    print(
        f"Diagnostic                : "
        f"{GPU_INFO['diagnostic']}"
    )
    print()
    print(
        "WARNING: CPU mode is suitable only for installation checks and "
        "small smoke tests."
    )
    print(
        "For full training, select a Colab GPU runtime before setting "
        "QUICK_TEST_MODE=False."
    )

print("\n[Training mode]")
print(f"QUICK_TEST_MODE           : {CONFIG.QUICK_TEST_MODE}")
print(f"FORCE_REBUILD             : {CONFIG.FORCE_REBUILD}")
print(f"RESUME_TRAINING           : {CONFIG.RESUME_TRAINING}")
print(f"USE_REAL_RECORDINGS       : {CONFIG.USE_REAL_RECORDINGS}")
print(
    f"USE_HARD_NEGATIVE_MINING : "
    f"{CONFIG.USE_HARD_NEGATIVE_MINING}"
)
print(
    f"USE_VAD_DURING_INFERENCE : "
    f"{CONFIG.USE_VAD_DURING_INFERENCE}"
)

print("\n[Effective dataset objectives]")
print(
    f"Positive synthetic        : "
    f"{EFFECTIVE_POSITIVE_SYNTHETIC_TARGET:,}"
)
print(
    f"Hard-negative synthetic   : "
    f"{EFFECTIVE_HARD_NEGATIVE_SYNTHETIC_TARGET:,}"
)
print(
    f"Public negative objective : "
    f"{EFFECTIVE_PUBLIC_NEGATIVE_TARGET:,}"
)
print(
    "Final test case objective : "
    + (
        f"{EFFECTIVE_FINAL_TEST_CASE_TARGET:,}"
        if EFFECTIVE_FINAL_TEST_CASE_TARGET is not None
        else "Smoke-test subset only"
    )
)

print("\n[Audio configuration]")
print(f"Wake word                 : {CONFIG.WAKE_WORD}")
print(f"Sample rate               : {CONFIG.SAMPLE_RATE} Hz")
print(f"Channels                  : {CONFIG.CHANNELS}")
print(
    f"Sample representation     : "
    f"Signed {CONFIG.SAMPLE_WIDTH * 8}-bit PCM"
)
print(
    f"Streaming chunk           : "
    f"{CONFIG.STREAM_CHUNK_SAMPLES} samples "
    f"({CONFIG.STREAM_CHUNK_DURATION_SECONDS * 1000:.1f} ms)"
)
print(
    f"Provisional clip duration : "
    f"{CONFIG.CLIP_DURATION_SECONDS:.2f} seconds"
)

print("\n[Operating targets]")
print(
    f"Target false activations  : "
    f"{CONFIG.TARGET_FALSE_ACTIVATIONS_PER_HOUR:.3f} per hour"
)
print(
    f"Target false reject rate  : "
    f"{CONFIG.TARGET_FALSE_REJECT_RATE:.3%}"
)
print(f"Activation policy         : {CONFIG.ACTIVATION_POLICY}")

print("\n[Estimated storage plan]")
print(
    f"Estimated raw PCM         : "
    f"{RESOURCE_ESTIMATE['raw_pcm_gib']:.2f} GiB"
)
print(
    f"Estimated WAV storage     : "
    f"{RESOURCE_ESTIMATE['estimated_wav_storage_gib']:.2f} GiB"
)
print(
    "Estimated feature storage: "
    f"{RESOURCE_ESTIMATE['estimated_feature_storage_low_gib']:.2f}–"
    f"{RESOURCE_ESTIMATE['estimated_feature_storage_high_gib']:.2f} GiB"
)
print(
    f"Peak RAM planning         : "
    f"{RESOURCE_ESTIMATE['expected_peak_host_ram_gib']}"
)
print(
    f"GPU memory planning       : "
    f"{RESOURCE_ESTIMATE['expected_gpu_memory_gib']}"
)

print("\n[Planned major stages]")
PLANNED_STAGES = (
    "Dependency and repository inspection",
    "Piper voice acquisition and API inspection",
    "Positive and hard-negative synthesis",
    "Public negative dataset preparation",
    "Real ReSpeaker recording validation",
    "Normalization and deterministic augmentation",
    "Leakage-safe dataset splitting",
    "OpenWakeWord feature generation",
    "First-pass classifier training",
    "Validation and checkpoint selection",
    "Threshold and temporal-logic calibration",
    "Hard-negative mining",
    "Optional second-pass retraining",
    "Final unseen clip and continuous-audio evaluation",
    "ONNX export and equivalence checking",
    "OpenWakeWord streaming compatibility verification",
    "Raspberry Pi runtime packaging",
)

for stage_number, stage_name in enumerate(PLANNED_STAGES, start=1):
    print(f"  {stage_number:02d}. {stage_name}")

print("\n[Paths]")
print(f"Isolated Python env       : {CONFIG.ENV_DIR}")
print(f"Training root             : {CONFIG.TRAINING_ROOT}")
print(f"Required ONNX output      : {CONFIG.OUTPUT_ONNX}")
print(f"Required release ZIP      : {CONFIG.RELEASE_ZIP}")

print("\n[Configuration identity]")
print(f"Created at UTC            : {CONFIG.CREATED_AT_UTC}")
print(f"Configuration SHA-256     : {CONFIG_SHA256}")

print("\n" + "=" * 88)

if CONFIG.QUICK_TEST_MODE:
    print(
        "SAFE MODE ACTIVE: Later cells will use small smoke-test datasets "
        "and limited training steps."
    )
else:
    print(
        "FULL MODE ACTIVE: Later cells will use the configured large-scale "
        "dataset objectives."
    )

if not GPU_INFO["available"]:
    print(
        "GPU WARNING: Do not disable QUICK_TEST_MODE until a CUDA GPU is "
        "available, unless you deliberately accept a very slow CPU run."
    )

print(
    "Cell 1 completed successfully. No packages were installed and no "
    "existing data was deleted."
)
print("=" * 88)

PINGO OPENWAKEWORD TRAINING — RUNTIME AND CONFIGURATION

[Runtime]
Google Colab detected     : True
System Python version     : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
System Python executable  : /usr/bin/python3
Operating system          : Linux-6.6.122+-x86_64-with-glibc2.35
Machine architecture      : x86_64
Logical CPU count         : 2

[Memory]
Total RAM                 : 12.67 GiB
Available RAM             : 10.56 GiB

[Disk]
Checked filesystem        : /content
Total disk                : 107.72 GiB
Used disk                 : 19.97 GiB
Free disk                 : 87.73 GiB

[GPU]
NVIDIA GPU detected       : NO
Diagnostic                : nvidia-smi is not installed or unavailable.

For full training, select a Colab GPU runtime before setting QUICK_TEST_MODE=False.

[Training mode]
QUICK_TEST_MODE           : True
FORCE_REBUILD             : False
RESUME_TRAINING           : True
USE_REAL_RECORDINGS       : True
USE_HARD_NEGATIVE_MINING : True
USE_VAD_DURING_INFERENC

In [ ]:
# =============================================================================
# Cell 2 — System dependency installation
# =============================================================================
#
# Purpose:
#   1. Install required Ubuntu system packages safely.
#   2. Avoid interactive package-manager prompts.
#   3. Log every command and its output.
#   4. Validate critical executables after installation.
#   5. Support restart/re-execution without deleting existing data.
#
# This cell does NOT:
#   - Replace or modify Colab's system Python.
#   - Install Python training libraries.
#   - Create the Python 3.11 virtual environment.
#   - Clone OpenWakeWord or Piper repositories.
#
# Python 3.11 isolation is handled in Cell 3.
#
# Expected environment:
#   Google Colab running Ubuntu/Debian with apt-get and root permissions.
#
# =============================================================================

from __future__ import annotations

import datetime as dt
import json
import os
import platform
import shlex
import shutil
import subprocess
import sys
import textwrap
from pathlib import Path
from typing import Dict, List, Optional, Sequence


# -----------------------------------------------------------------------------
# Recover minimal configuration when Cell 1 has not been executed
# -----------------------------------------------------------------------------

try:
    CONFIG
except NameError:
    print(
        "WARNING: CONFIG from Cell 1 was not found. "
        "Using the required default paths for Cell 2."
    )

    class _FallbackConfig:
        TRAINING_ROOT = "/content/pingo_training"
        LOGS_DIR = "/content/pingo_training/logs"
        CONFIG_DIR = "/content/pingo_training/config"
        CACHE_DIR = "/content/pingo_training/cache"
        FORCE_REBUILD = False

    CONFIG = _FallbackConfig()


TRAINING_ROOT = Path(CONFIG.TRAINING_ROOT)
LOGS_DIR = Path(CONFIG.LOGS_DIR)
CONFIG_DIR = Path(CONFIG.CONFIG_DIR)
CACHE_DIR = Path(CONFIG.CACHE_DIR)

for required_directory in (
    TRAINING_ROOT,
    LOGS_DIR,
    CONFIG_DIR,
    CACHE_DIR,
):
    required_directory.mkdir(parents=True, exist_ok=True)


# -----------------------------------------------------------------------------
# Installation paths and logs
# -----------------------------------------------------------------------------

INSTALL_LOG_PATH = LOGS_DIR / "cell_02_system_installation.log"
INSTALL_REPORT_PATH = CONFIG_DIR / "system_dependency_report.json"

INSTALL_STARTED_AT = dt.datetime.now(dt.timezone.utc)

# Truncate the log only at the beginning of this cell's current execution.
INSTALL_LOG_PATH.write_text(
    (
        "PINGO SYSTEM DEPENDENCY INSTALLATION LOG\n"
        f"Started UTC: {INSTALL_STARTED_AT.isoformat()}\n"
        f"System Python: {sys.version}\n"
        f"Platform: {platform.platform()}\n"
        f"Machine: {platform.machine()}\n"
        + "=" * 88
        + "\n"
    ),
    encoding="utf-8",
)


# -----------------------------------------------------------------------------
# Utility functions
# -----------------------------------------------------------------------------

class InstallationStageError(RuntimeError):
    """Raised when a system dependency stage fails validation."""


def append_log(message: str) -> None:
    """Append a message to the Cell 2 installation log."""
    with INSTALL_LOG_PATH.open("a", encoding="utf-8") as log_file:
        log_file.write(message.rstrip() + "\n")


def format_command(command: Sequence[str]) -> str:
    """Return a shell-readable representation for diagnostics only."""
    return " ".join(shlex.quote(str(part)) for part in command)


def tail_text(text: str, max_lines: int = 100) -> str:
    """Return the final lines of a potentially large command output."""
    lines = text.splitlines()

    if len(lines) <= max_lines:
        return text

    return "\n".join(
        [
            f"... output truncated; showing final {max_lines} lines ...",
            *lines[-max_lines:],
        ]
    )


def run_checked_command(
    command: Sequence[str],
    *,
    stage_name: str,
    timeout_seconds: int = 1800,
    extra_env: Optional[Dict[str, str]] = None,
    print_output: bool = True,
) -> subprocess.CompletedProcess[str]:
    """
    Execute a command safely without shell=True.

    Requirements satisfied:
      - Uses env=..., never environment=...
      - Prints the exact failing command
      - Checks return code
      - Logs stdout and stderr
      - Stops the current stage on failure
      - Avoids shell quoting errors
    """
    command = [str(part) for part in command]
    readable_command = format_command(command)

    environment = os.environ.copy()
    environment.update(
        {
            "DEBIAN_FRONTEND": "noninteractive",
            "APT_LISTCHANGES_FRONTEND": "none",
            "NEEDRESTART_MODE": "a",
            "PYTHONUNBUFFERED": "1",
        }
    )

    if extra_env:
        environment.update(extra_env)

    stage_header = (
        "\n"
        + "=" * 88
        + "\n"
        + f"STAGE: {stage_name}\n"
        + f"COMMAND: {readable_command}\n"
        + "=" * 88
    )

    print(stage_header)
    append_log(stage_header)

    try:
        result = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=False,
            timeout=timeout_seconds,
            env=environment,
            cwd="/content" if Path("/content").exists() else str(Path.cwd()),
        )
    except FileNotFoundError as exc:
        diagnostic = textwrap.dedent(
            f"""
            SYSTEM INSTALLATION ERROR

            Stage:
              {stage_name}

            Failing command:
              {readable_command}

            Error:
              Executable not found: {command[0]}

            Python:
              {sys.version}

            Expected installation log:
              {INSTALL_LOG_PATH}

            Original exception:
              {exc}
            """
        ).strip()

        append_log(diagnostic)
        raise InstallationStageError(diagnostic) from exc

    except subprocess.TimeoutExpired as exc:
        stdout = exc.stdout or ""
        stderr = exc.stderr or ""

        if isinstance(stdout, bytes):
            stdout = stdout.decode("utf-8", errors="replace")

        if isinstance(stderr, bytes):
            stderr = stderr.decode("utf-8", errors="replace")

        diagnostic = textwrap.dedent(
            f"""
            SYSTEM INSTALLATION TIMEOUT

            Stage:
              {stage_name}

            Failing command:
              {readable_command}

            Timeout:
              {timeout_seconds} seconds

            Python:
              {sys.version}

            Expected installation log:
              {INSTALL_LOG_PATH}

            Recent stdout:
            {tail_text(stdout)}

            Recent stderr:
            {tail_text(stderr)}
            """
        ).strip()

        append_log(diagnostic)
        raise InstallationStageError(diagnostic) from exc

    stdout = result.stdout or ""
    stderr = result.stderr or ""

    append_log("\n[STDOUT]\n" + stdout)
    append_log("\n[STDERR]\n" + stderr)
    append_log(f"\n[RETURN CODE]\n{result.returncode}")

    if print_output:
        if stdout.strip():
            print(tail_text(stdout, max_lines=80))

        if stderr.strip():
            print("\n[stderr]")
            print(tail_text(stderr, max_lines=80))

    if result.returncode != 0:
        diagnostic = textwrap.dedent(
            f"""
            SYSTEM INSTALLATION FAILED

            Stage:
              {stage_name}

            Failing command:
              {readable_command}

            Return code:
              {result.returncode}

            System Python:
              {sys.version}

            Platform:
              {platform.platform()}

            Expected installation log:
              {INSTALL_LOG_PATH}

            Expected training root:
              {TRAINING_ROOT}

            Recent stdout:
            {tail_text(stdout)}

            Recent stderr:
            {tail_text(stderr)}

            Action:
              Read the installation log above. Fix the apt/network/repository
              error, then rerun Cell 2. Do not continue to Cell 3 while this
              stage is failing.
            """
        ).strip()

        append_log(diagnostic)
        raise InstallationStageError(diagnostic)

    return result


def command_version(
    executable: str,
    version_arguments: Sequence[str],
    *,
    timeout_seconds: int = 30,
) -> Dict[str, object]:
    """Inspect an installed executable and capture its version output."""
    executable_path = shutil.which(executable)

    result_data: Dict[str, object] = {
        "executable": executable,
        "path": executable_path,
        "available": executable_path is not None,
        "return_code": None,
        "version_output": None,
    }

    if executable_path is None:
        return result_data

    try:
        result = subprocess.run(
            [executable_path, *version_arguments],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=False,
            timeout=timeout_seconds,
            env=os.environ.copy(),
        )

        output = "\n".join(
            part.strip()
            for part in (result.stdout, result.stderr)
            if part and part.strip()
        )

        result_data.update(
            {
                "return_code": result.returncode,
                "version_output": output[:4000],
            }
        )
    except Exception as exc:
        result_data["version_output"] = (
            f"Version inspection failed: {type(exc).__name__}: {exc}"
        )

    return result_data


def dpkg_package_status(package_name: str) -> Dict[str, object]:
    """Check whether an apt package is installed according to dpkg-query."""
    if shutil.which("dpkg-query") is None:
        return {
            "package": package_name,
            "installed": False,
            "version": None,
            "diagnostic": "dpkg-query is unavailable.",
        }

    result = subprocess.run(
        [
            "dpkg-query",
            "-W",
            "-f=${Status}\t${Version}",
            package_name,
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        check=False,
        timeout=30,
        env=os.environ.copy(),
    )

    output = result.stdout.strip()
    installed = (
        result.returncode == 0
        and output.startswith("install ok installed")
    )

    version = None

    if installed and "\t" in output:
        version = output.split("\t", maxsplit=1)[1].strip()

    return {
        "package": package_name,
        "installed": installed,
        "version": version,
        "diagnostic": result.stderr.strip() or None,
    }


# -----------------------------------------------------------------------------
# Validate the host package manager
# -----------------------------------------------------------------------------

if os.name != "posix":
    raise InstallationStageError(
        "Cell 2 requires a Linux/Ubuntu Colab runtime. "
        f"Detected os.name={os.name!r}."
    )

if shutil.which("apt-get") is None:
    raise InstallationStageError(
        "apt-get was not found. This cell expects an Ubuntu/Debian-based "
        "Google Colab runtime."
    )

if hasattr(os, "geteuid") and os.geteuid() != 0:
    raise InstallationStageError(
        "Root permission is required for apt-get installation. "
        "Standard Google Colab runtimes normally execute notebook cells "
        "with sufficient privileges."
    )


# -----------------------------------------------------------------------------
# System packages
# -----------------------------------------------------------------------------
#
# Package groups:
#
# Repository and download tools:
#   git, git-lfs, curl, wget, ca-certificates
#
# Build tools:
#   build-essential, cmake, ninja-build, pkg-config, autoconf, automake,
#   libtool
#
# Archive and file utilities:
#   unzip, zip, xz-utils, tar, rsync, file, jq
#
# Audio tools and native libraries:
#   ffmpeg, sox, libsox-fmt-all, libsndfile1, libsndfile1-dev,
#   libsamplerate0, libsamplerate0-dev, libspeexdsp-dev,
#   portaudio19-dev, libasound2-dev
#
# Piper/espeak support:
#   espeak-ng, espeak-ng-data
#
# Python 3.11 build/runtime prerequisites:
#   software-properties-common and common native development libraries.
#   Cell 3 will decide whether to use uv-managed Python 3.11 or another safe
#   isolated approach. These packages do not replace Colab's system Python.
#
# Additional media/codecs:
#   libopus-dev, libogg-dev, libvorbis-dev, flac
#
# General diagnostics:
#   lsof, procps, pciutils
#
# -----------------------------------------------------------------------------

SYSTEM_PACKAGES: List[str] = [
    # Version control and downloads
    "git",
    "git-lfs",
    "curl",
    "wget",
    "ca-certificates",
    "gnupg",

    # Build toolchain
    "build-essential",
    "cmake",
    "ninja-build",
    "pkg-config",
    "autoconf",
    "automake",
    "libtool",

    # Archive and data utilities
    "unzip",
    "zip",
    "xz-utils",
    "tar",
    "rsync",
    "file",
    "jq",

    # Audio conversion and inspection
    "ffmpeg",
    "sox",
    "libsox-fmt-all",
    "flac",

    # Native audio libraries
    "libsndfile1",
    "libsndfile1-dev",
    "libsamplerate0",
    "libsamplerate0-dev",
    "libspeexdsp-dev",
    "portaudio19-dev",
    "libasound2-dev",
    "libopus-dev",
    "libogg-dev",
    "libvorbis-dev",

    # Piper phonemization support
    "espeak-ng",
    "espeak-ng-data",

    # Native libraries commonly required by isolated Python environments
    "libssl-dev",
    "libffi-dev",
    "zlib1g-dev",
    "libbz2-dev",
    "libreadline-dev",
    "libsqlite3-dev",
    "liblzma-dev",
    "libncursesw5-dev",
    "libgdbm-dev",
    "libnss3-dev",
    "uuid-dev",
    "tk-dev",

    # System and hardware diagnostics
    "lsof",
    "procps",
    "pciutils",
]


# Remove duplicates while preserving declared order.
SYSTEM_PACKAGES = list(dict.fromkeys(SYSTEM_PACKAGES))


# -----------------------------------------------------------------------------
# Pre-installation report
# -----------------------------------------------------------------------------

print("=" * 88)
print("PINGO TRAINING — CELL 2 SYSTEM DEPENDENCY INSTALLATION")
print("=" * 88)
print(f"Training root          : {TRAINING_ROOT}")
print(f"Installation log       : {INSTALL_LOG_PATH}")
print(f"Installation report    : {INSTALL_REPORT_PATH}")
print(f"Number of apt packages : {len(SYSTEM_PACKAGES)}")
print(f"System Python          : {sys.version.splitlines()[0]}")
print(f"Platform               : {platform.platform()}")
print(f"Architecture           : {platform.machine()}")
print()
print("This cell may be rerun safely.")
print("apt-get will skip packages that are already installed.")
print("=" * 88)


# -----------------------------------------------------------------------------
# Stage 1: Repair interrupted dpkg state when necessary
# -----------------------------------------------------------------------------
#
# `dpkg --configure -a` is restart-safe and helps recover from a previous
# interrupted package installation. It does not remove completed datasets.
#
# -----------------------------------------------------------------------------

if shutil.which("dpkg") is not None:
    run_checked_command(
        ["dpkg", "--configure", "-a"],
        stage_name="Repair or complete interrupted dpkg configuration",
        timeout_seconds=900,
    )


# -----------------------------------------------------------------------------
# Stage 2: Refresh apt package metadata
# -----------------------------------------------------------------------------

run_checked_command(
    [
        "apt-get",
        "update",
        "-o",
        "Acquire::Retries=3",
        "-o",
        "Acquire::http::Timeout=60",
        "-o",
        "Acquire::https::Timeout=60",
    ],
    stage_name="Refresh Ubuntu package metadata",
    timeout_seconds=1800,
)


# -----------------------------------------------------------------------------
# Stage 3: Install required packages
# -----------------------------------------------------------------------------

run_checked_command(
    [
        "apt-get",
        "install",
        "-y",
        "--no-install-recommends",
        "-o",
        "Dpkg::Options::=--force-confdef",
        "-o",
        "Dpkg::Options::=--force-confold",
        "-o",
        "Acquire::Retries=3",
        *SYSTEM_PACKAGES,
    ],
    stage_name="Install required Ubuntu system packages",
    timeout_seconds=3600,
)


# -----------------------------------------------------------------------------
# Stage 4: Initialize Git LFS
# -----------------------------------------------------------------------------

run_checked_command(
    ["git", "lfs", "install"],
    stage_name="Initialize Git LFS",
    timeout_seconds=120,
)


# -----------------------------------------------------------------------------
# Stage 5: Refresh linker cache
# -----------------------------------------------------------------------------

if shutil.which("ldconfig") is not None:
    run_checked_command(
        ["ldconfig"],
        stage_name="Refresh shared-library linker cache",
        timeout_seconds=120,
    )


# -----------------------------------------------------------------------------
# Stage 6: Validate required commands
# -----------------------------------------------------------------------------

REQUIRED_COMMANDS: Dict[str, Sequence[str]] = {
    "git": ("--version",),
    "git-lfs": ("version",),
    "curl": ("--version",),
    "wget": ("--version",),
    "cmake": ("--version",),
    "ninja": ("--version",),
    "pkg-config": ("--version",),
    "ffmpeg": ("-version",),
    "ffprobe": ("-version",),
    "sox": ("--version",),
    "soxi": ("--version",),
    "flac": ("--version",),
    "espeak-ng": ("--version",),
    "unzip": ("-v",),
    "zip": ("-v",),
    "jq": ("--version",),
    "file": ("--version",),
    "rsync": ("--version",),
}

COMMAND_REPORT: Dict[str, Dict[str, object]] = {}
MISSING_COMMANDS: List[str] = []
FAILED_COMMAND_CHECKS: List[str] = []

print("\n" + "=" * 88)
print("VALIDATING REQUIRED EXECUTABLES")
print("=" * 88)

for command_name, version_arguments in REQUIRED_COMMANDS.items():
    information = command_version(
        command_name,
        version_arguments,
    )
    COMMAND_REPORT[command_name] = information

    if not information["available"]:
        MISSING_COMMANDS.append(command_name)
        print(f"[MISSING] {command_name}")
        continue

    return_code = information["return_code"]

    # Some tools such as zip may return unusual codes for version/help output.
    # Presence of the executable is the essential installation check, but
    # unexpected command failures are still reported.
    if return_code not in (0, 1):
        FAILED_COMMAND_CHECKS.append(command_name)
        print(
            f"[WARNING] {command_name}: executable found, "
            f"version check returned {return_code}"
        )
    else:
        version_output = str(
            information["version_output"] or ""
        ).splitlines()

        first_line = (
            version_output[0]
            if version_output
            else "version output unavailable"
        )

        print(
            f"[OK] {command_name:<12} "
            f"{information['path']} | {first_line[:120]}"
        )


# -----------------------------------------------------------------------------
# Stage 7: Validate apt package status
# -----------------------------------------------------------------------------

print("\n" + "=" * 88)
print("VALIDATING APT PACKAGE STATUS")
print("=" * 88)

PACKAGE_REPORT: Dict[str, Dict[str, object]] = {}
MISSING_PACKAGES: List[str] = []

for package_name in SYSTEM_PACKAGES:
    package_information = dpkg_package_status(package_name)
    PACKAGE_REPORT[package_name] = package_information

    if package_information["installed"]:
        print(
            f"[OK] {package_name:<28} "
            f"{package_information['version']}"
        )
    else:
        MISSING_PACKAGES.append(package_name)
        print(f"[MISSING] {package_name}")


# -----------------------------------------------------------------------------
# Stage 8: Native-library checks
# -----------------------------------------------------------------------------

NATIVE_LIBRARY_CHECKS: Dict[str, Sequence[str]] = {
    "sndfile": (
        "pkg-config",
        "--modversion",
        "sndfile",
    ),
    "samplerate": (
        "pkg-config",
        "--modversion",
        "samplerate",
    ),
    "speexdsp": (
        "pkg-config",
        "--modversion",
        "speexdsp",
    ),
    "portaudio-2.0": (
        "pkg-config",
        "--modversion",
        "portaudio-2.0",
    ),
    "alsa": (
        "pkg-config",
        "--modversion",
        "alsa",
    ),
}

NATIVE_LIBRARY_REPORT: Dict[str, Dict[str, object]] = {}
FAILED_NATIVE_LIBRARIES: List[str] = []

print("\n" + "=" * 88)
print("VALIDATING NATIVE AUDIO DEVELOPMENT LIBRARIES")
print("=" * 88)

for library_name, check_command in NATIVE_LIBRARY_CHECKS.items():
    try:
        result = subprocess.run(
            list(check_command),
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=False,
            timeout=30,
            env=os.environ.copy(),
        )

        version_output = result.stdout.strip()

        NATIVE_LIBRARY_REPORT[library_name] = {
            "available": result.returncode == 0,
            "return_code": result.returncode,
            "version": version_output or None,
            "stderr": result.stderr.strip() or None,
        }

        if result.returncode == 0:
            print(f"[OK] {library_name:<18} {version_output}")
        else:
            FAILED_NATIVE_LIBRARIES.append(library_name)
            print(
                f"[MISSING] {library_name:<18} "
                f"{result.stderr.strip()}"
            )

    except Exception as exc:
        FAILED_NATIVE_LIBRARIES.append(library_name)
        NATIVE_LIBRARY_REPORT[library_name] = {
            "available": False,
            "return_code": None,
            "version": None,
            "stderr": f"{type(exc).__name__}: {exc}",
        }
        print(
            f"[ERROR] {library_name:<18} "
            f"{type(exc).__name__}: {exc}"
        )


# -----------------------------------------------------------------------------
# Stage 9: Functional audio-tool smoke tests
# -----------------------------------------------------------------------------

SMOKE_TEST_DIR = CACHE_DIR / "cell_02_audio_smoke_test"
SMOKE_TEST_DIR.mkdir(parents=True, exist_ok=True)

TEST_WAV = SMOKE_TEST_DIR / "system_dependency_test.wav"
TEST_RESAMPLED_WAV = (
    SMOKE_TEST_DIR / "system_dependency_test_16khz_mono.wav"
)

# Generate a very short synthetic sine wave using SoX. This confirms that SoX,
# libsndfile, and the installed audio format handlers can create valid WAV.
run_checked_command(
    [
        "sox",
        "-n",
        "-r",
        "48000",
        "-c",
        "2",
        "-b",
        "16",
        str(TEST_WAV),
        "synth",
        "0.25",
        "sine",
        "440",
        "vol",
        "0.05",
    ],
    stage_name="Generate temporary WAV for audio-tool smoke test",
    timeout_seconds=120,
)

# Convert with FFmpeg to the deployment format:
# WAV, 16 kHz, mono, signed 16-bit PCM.
run_checked_command(
    [
        "ffmpeg",
        "-hide_banner",
        "-loglevel",
        "error",
        "-y",
        "-i",
        str(TEST_WAV),
        "-ar",
        "16000",
        "-ac",
        "1",
        "-c:a",
        "pcm_s16le",
        str(TEST_RESAMPLED_WAV),
    ],
    stage_name="Validate FFmpeg WAV conversion to 16 kHz mono PCM16",
    timeout_seconds=120,
)

if not TEST_RESAMPLED_WAV.exists():
    raise InstallationStageError(
        "FFmpeg returned success but the expected smoke-test output "
        f"does not exist: {TEST_RESAMPLED_WAV}"
    )

if TEST_RESAMPLED_WAV.stat().st_size <= 44:
    raise InstallationStageError(
        "The converted WAV is empty or invalid: "
        f"{TEST_RESAMPLED_WAV} "
        f"({TEST_RESAMPLED_WAV.stat().st_size} bytes)"
    )

ffprobe_result = run_checked_command(
    [
        "ffprobe",
        "-v",
        "error",
        "-select_streams",
        "a:0",
        "-show_entries",
        "stream=codec_name,sample_rate,channels,sample_fmt",
        "-of",
        "json",
        str(TEST_RESAMPLED_WAV),
    ],
    stage_name="Inspect converted smoke-test WAV",
    timeout_seconds=120,
)

try:
    ffprobe_data = json.loads(ffprobe_result.stdout)
except json.JSONDecodeError as exc:
    raise InstallationStageError(
        "ffprobe did not return valid JSON for the smoke-test WAV.\n"
        f"File: {TEST_RESAMPLED_WAV}\n"
        f"Output: {ffprobe_result.stdout}\n"
        f"Error: {exc}"
    ) from exc

streams = ffprobe_data.get("streams", [])

if len(streams) != 1:
    raise InstallationStageError(
        "Expected exactly one audio stream in the smoke-test WAV, "
        f"found {len(streams)}."
    )

stream = streams[0]

expected_audio_properties = {
    "codec_name": "pcm_s16le",
    "sample_rate": "16000",
    "channels": 1,
}

audio_property_errors: List[str] = []

for property_name, expected_value in expected_audio_properties.items():
    actual_value = stream.get(property_name)

    if actual_value != expected_value:
        audio_property_errors.append(
            f"{property_name}: expected {expected_value!r}, "
            f"found {actual_value!r}"
        )

if audio_property_errors:
    raise InstallationStageError(
        "Audio conversion smoke test produced an unexpected format:\n  - "
        + "\n  - ".join(audio_property_errors)
    )

print(
    "\n[OK] Audio smoke test produced a valid 16 kHz mono PCM16 WAV:"
)
print(f"     {TEST_RESAMPLED_WAV}")
print(f"     Size: {TEST_RESAMPLED_WAV.stat().st_size:,} bytes")


# -----------------------------------------------------------------------------
# Stage 10: Check remaining disk space
# -----------------------------------------------------------------------------

disk_usage = shutil.disk_usage(TRAINING_ROOT)
free_disk_gib = disk_usage.free / (1024**3)
total_disk_gib = disk_usage.total / (1024**3)

print("\n" + "=" * 88)
print("DISK STATUS AFTER SYSTEM INSTALLATION")
print("=" * 88)
print(f"Filesystem total : {total_disk_gib:.2f} GiB")
print(f"Filesystem free  : {free_disk_gib:.2f} GiB")

DISK_WARNING = None

if free_disk_gib < 5:
    DISK_WARNING = (
        "Less than 5 GiB of free disk remains. Do not begin dataset "
        "generation. Restart with a runtime that has more storage or reduce "
        "the configured dataset targets."
    )
elif free_disk_gib < 20:
    DISK_WARNING = (
        "Less than 20 GiB of free disk remains. Quick-test mode should work, "
        "but a full 50,000-positive training pipeline will likely require "
        "careful streaming, cache cleanup, or external Drive storage."
    )

if DISK_WARNING:
    print(f"WARNING: {DISK_WARNING}")
else:
    print("[OK] Free disk is sufficient for the next setup stages.")


# -----------------------------------------------------------------------------
# Write machine-readable installation report
# -----------------------------------------------------------------------------

INSTALL_COMPLETED_AT = dt.datetime.now(dt.timezone.utc)

INSTALL_REPORT = {
    "stage": "Cell 2 — System dependency installation",
    "started_at_utc": INSTALL_STARTED_AT.isoformat(),
    "completed_at_utc": INSTALL_COMPLETED_AT.isoformat(),
    "platform": {
        "python": sys.version,
        "python_executable": sys.executable,
        "platform": platform.platform(),
        "machine": platform.machine(),
    },
    "paths": {
        "training_root": str(TRAINING_ROOT),
        "installation_log": str(INSTALL_LOG_PATH),
        "installation_report": str(INSTALL_REPORT_PATH),
        "audio_smoke_test_input": str(TEST_WAV),
        "audio_smoke_test_output": str(TEST_RESAMPLED_WAV),
    },
    "apt_packages": PACKAGE_REPORT,
    "commands": COMMAND_REPORT,
    "native_libraries": NATIVE_LIBRARY_REPORT,
    "audio_smoke_test": {
        "passed": True,
        "ffprobe_stream": stream,
        "output_size_bytes": TEST_RESAMPLED_WAV.stat().st_size,
    },
    "disk": {
        "total_gib": round(total_disk_gib, 3),
        "free_gib": round(free_disk_gib, 3),
        "warning": DISK_WARNING,
    },
    "validation": {
        "missing_commands": MISSING_COMMANDS,
        "failed_command_checks": FAILED_COMMAND_CHECKS,
        "missing_packages": MISSING_PACKAGES,
        "failed_native_libraries": FAILED_NATIVE_LIBRARIES,
    },
}

INSTALL_REPORT_PATH.write_text(
    json.dumps(
        INSTALL_REPORT,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

append_log(
    "\nINSTALLATION REPORT\n"
    + json.dumps(INSTALL_REPORT, indent=2, ensure_ascii=False)
)


# -----------------------------------------------------------------------------
# Final hard validation
# -----------------------------------------------------------------------------

fatal_validation_errors: List[str] = []

if MISSING_COMMANDS:
    fatal_validation_errors.append(
        "Required executables are missing: "
        + ", ".join(MISSING_COMMANDS)
    )

if MISSING_PACKAGES:
    fatal_validation_errors.append(
        "Required apt packages are not installed: "
        + ", ".join(MISSING_PACKAGES)
    )

if FAILED_NATIVE_LIBRARIES:
    fatal_validation_errors.append(
        "Required native development libraries failed pkg-config checks: "
        + ", ".join(FAILED_NATIVE_LIBRARIES)
    )

if fatal_validation_errors:
    diagnostic = (
        "CELL 2 VALIDATION FAILED\n\n"
        + "\n".join(
            f"  - {error}" for error in fatal_validation_errors
        )
        + "\n\n"
        + f"Full log: {INSTALL_LOG_PATH}\n"
        + f"Report: {INSTALL_REPORT_PATH}\n"
        + "Do not continue to Cell 3 until these errors are resolved."
    )

    append_log(diagnostic)
    raise InstallationStageError(diagnostic)


# -----------------------------------------------------------------------------
# Completion summary
# -----------------------------------------------------------------------------

print("\n" + "=" * 88)
print("CELL 2 COMPLETED SUCCESSFULLY")
print("=" * 88)
print(f"Installed apt packages : {len(SYSTEM_PACKAGES)}")
print(f"Validated commands     : {len(REQUIRED_COMMANDS)}")
print(f"Validated libraries    : {len(NATIVE_LIBRARY_CHECKS)}")
print(f"Audio conversion test  : PASSED")
print(f"Installation log       : {INSTALL_LOG_PATH}")
print(f"Installation report    : {INSTALL_REPORT_PATH}")
print(f"Free disk              : {free_disk_gib:.2f} GiB")
print()
print("No system Python version was replaced or modified.")
print("Existing training data was not deleted.")
print()
print(
    "Next: Cell 3 will install uv, acquire an isolated Python 3.11 runtime, "
    "create /content/pingo-env with seeded pip support, and verify that the "
    "environment is independent from Colab's system Python."
)
print("=" * 88)

PINGO TRAINING — CELL 2 SYSTEM DEPENDENCY INSTALLATION
Training root          : /content/pingo_training
Installation log       : /content/pingo_training/logs/cell_02_system_installation.log
Installation report    : /content/pingo_training/config/system_dependency_report.json
Number of apt packages : 51
System Python          : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform               : Linux-6.6.122+-x86_64-with-glibc2.35
Architecture           : x86_64

This cell may be rerun safely.
apt-get will skip packages that are already installed.

STAGE: Repair or complete interrupted dpkg configuration
COMMAND: dpkg --configure -a

STAGE: Refresh Ubuntu package metadata
COMMAND: apt-get update -o Acquire::Retries=3 -o Acquire::http::Timeout=60 -o Acquire::https::Timeout=60
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InR

In [ ]:
# =============================================================================
# Cell 3 — Isolated Python environment
# =============================================================================
#
# Purpose:
#   1. Install or locate the official Astral uv environment manager.
#   2. Install an isolated CPython 3.11 runtime.
#   3. Create /content/pingo-env using `uv venv --seed`.
#   4. Confirm pip is available through:
#
#          /content/pingo-env/bin/python -m pip
#
#   5. Verify the environment does not reuse Colab's system Python.
#   6. Upgrade only basic packaging tools inside the isolated environment.
#   7. Save a machine-readable environment report.
#
# Important:
#   - This cell does not replace or uninstall Colab's system Python.
#   - This cell does not install OpenWakeWord, PyTorch, TensorFlow, or
#     training packages. Dependency resolution happens after repository
#     inspection in Cells 4 and 5.
#   - The cell is restart-safe. A valid existing Python 3.11 environment is
#     reused unless FORCE_REBUILD=True.
#
# =============================================================================

from __future__ import annotations

import datetime as dt
import hashlib
import json
import os
import platform
import re
import shlex
import shutil
import subprocess
import sys
import textwrap
import urllib.request
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence


# -----------------------------------------------------------------------------
# Recover configuration when Cell 1 was not executed
# -----------------------------------------------------------------------------

try:
    CONFIG
except NameError:
    print(
        "WARNING: CONFIG from Cell 1 was not found. "
        "Using the mandatory default paths."
    )

    class _FallbackConfig:
        TRAINING_ROOT = "/content/pingo_training"
        LOGS_DIR = "/content/pingo_training/logs"
        CONFIG_DIR = "/content/pingo_training/config"
        CACHE_DIR = "/content/pingo_training/cache"
        ENV_DIR = "/content/pingo-env"
        FORCE_REBUILD = False
        RANDOM_SEED = 42

    CONFIG = _FallbackConfig()


TRAINING_ROOT = Path(CONFIG.TRAINING_ROOT)
LOGS_DIR = Path(CONFIG.LOGS_DIR)
CONFIG_DIR = Path(CONFIG.CONFIG_DIR)
CACHE_DIR = Path(CONFIG.CACHE_DIR)
ENV_DIR = Path(CONFIG.ENV_DIR)

for directory in (
    TRAINING_ROOT,
    LOGS_DIR,
    CONFIG_DIR,
    CACHE_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)


# -----------------------------------------------------------------------------
# Constants and output files
# -----------------------------------------------------------------------------

REQUIRED_PYTHON_MAJOR = 3
REQUIRED_PYTHON_MINOR = 11

CELL_STARTED_AT = dt.datetime.now(dt.timezone.utc)

INSTALL_LOG_PATH = LOGS_DIR / "cell_03_isolated_environment.log"
ENVIRONMENT_REPORT_PATH = (
    CONFIG_DIR / "isolated_python_environment.json"
)
UV_INSTALL_SCRIPT_PATH = CACHE_DIR / "uv_install.sh"
UV_DOWNLOAD_METADATA_PATH = (
    CACHE_DIR / "uv_install_download_metadata.json"
)

ENV_PYTHON = ENV_DIR / "bin" / "python"
ENV_PYTHON3 = ENV_DIR / "bin" / "python3"
ENV_PIP = ENV_DIR / "bin" / "pip"
ENV_ACTIVATE = ENV_DIR / "bin" / "activate"
ENV_PYVENV_CONFIG = ENV_DIR / "pyvenv.cfg"

UV_INSTALL_URL = "https://astral.sh/uv/install.sh"

# uv is commonly installed here by the official installer.
EXPECTED_UV_LOCATIONS = (
    Path.home() / ".local" / "bin" / "uv",
    Path.home() / ".cargo" / "bin" / "uv",
    Path("/usr/local/bin/uv"),
    Path("/usr/bin/uv"),
)


# -----------------------------------------------------------------------------
# Logging and command helpers
# -----------------------------------------------------------------------------

class EnvironmentStageError(RuntimeError):
    """Raised when creation or validation of the isolated environment fails."""


INSTALL_LOG_PATH.write_text(
    (
        "PINGO ISOLATED PYTHON ENVIRONMENT LOG\n"
        f"Started UTC: {CELL_STARTED_AT.isoformat()}\n"
        f"System Python: {sys.version}\n"
        f"System executable: {sys.executable}\n"
        f"Platform: {platform.platform()}\n"
        f"Target environment: {ENV_DIR}\n"
        + "=" * 88
        + "\n"
    ),
    encoding="utf-8",
)


def append_log(message: str) -> None:
    """Append text to the Cell 3 log."""
    with INSTALL_LOG_PATH.open("a", encoding="utf-8") as log_file:
        log_file.write(message.rstrip() + "\n")


def format_command(command: Sequence[str]) -> str:
    """Format a command for human-readable diagnostics."""
    return " ".join(shlex.quote(str(part)) for part in command)


def tail_text(text: str, max_lines: int = 120) -> str:
    """Limit long subprocess output while preserving recent diagnostics."""
    lines = text.splitlines()

    if len(lines) <= max_lines:
        return text

    return "\n".join(
        [
            f"... output truncated; showing final {max_lines} lines ...",
            *lines[-max_lines:],
        ]
    )


def run_command(
    command: Sequence[str],
    *,
    stage_name: str,
    timeout_seconds: int = 1800,
    cwd: Optional[Path] = None,
    extra_env: Optional[Dict[str, str]] = None,
    print_output: bool = True,
) -> subprocess.CompletedProcess[str]:
    """
    Run a command safely and stop the stage when it fails.

    The function:
      - never uses shell=True;
      - passes subprocess variables with env=...;
      - captures stdout and stderr;
      - logs the exact command;
      - prints actionable diagnostics;
      - checks return codes.
    """
    command = [str(part) for part in command]
    readable_command = format_command(command)

    environment = os.environ.copy()
    environment.update(
        {
            "PYTHONUNBUFFERED": "1",
            "UV_NO_PROGRESS": "1",
            "UV_PYTHON_DOWNLOADS": "automatic",
            "UV_LINK_MODE": "copy",
        }
    )

    if extra_env:
        environment.update(extra_env)

    actual_cwd = (
        str(cwd.resolve())
        if cwd is not None
        else (
            "/content"
            if Path("/content").exists()
            else str(Path.cwd())
        )
    )

    header = (
        "\n"
        + "=" * 88
        + "\n"
        + f"STAGE: {stage_name}\n"
        + f"COMMAND: {readable_command}\n"
        + f"WORKING DIRECTORY: {actual_cwd}\n"
        + "=" * 88
    )

    print(header)
    append_log(header)

    try:
        result = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=False,
            timeout=timeout_seconds,
            cwd=actual_cwd,
            env=environment,
        )
    except FileNotFoundError as exc:
        diagnostic = textwrap.dedent(
            f"""
            ISOLATED ENVIRONMENT ERROR

            Stage:
              {stage_name}

            Failing command:
              {readable_command}

            Error:
              Executable not found: {command[0]}

            System Python:
              {sys.version}

            Target environment:
              {ENV_DIR}

            Log:
              {INSTALL_LOG_PATH}

            Original exception:
              {exc}
            """
        ).strip()

        append_log(diagnostic)
        raise EnvironmentStageError(diagnostic) from exc

    except subprocess.TimeoutExpired as exc:
        stdout = exc.stdout or ""
        stderr = exc.stderr or ""

        if isinstance(stdout, bytes):
            stdout = stdout.decode("utf-8", errors="replace")

        if isinstance(stderr, bytes):
            stderr = stderr.decode("utf-8", errors="replace")

        diagnostic = textwrap.dedent(
            f"""
            ISOLATED ENVIRONMENT TIMEOUT

            Stage:
              {stage_name}

            Failing command:
              {readable_command}

            Timeout:
              {timeout_seconds} seconds

            Target environment:
              {ENV_DIR}

            Log:
              {INSTALL_LOG_PATH}

            Recent stdout:
            {tail_text(stdout)}

            Recent stderr:
            {tail_text(stderr)}
            """
        ).strip()

        append_log(diagnostic)
        raise EnvironmentStageError(diagnostic) from exc

    stdout = result.stdout or ""
    stderr = result.stderr or ""

    append_log("\n[STDOUT]\n" + stdout)
    append_log("\n[STDERR]\n" + stderr)
    append_log(f"\n[RETURN CODE]\n{result.returncode}")

    if print_output:
        if stdout.strip():
            print(tail_text(stdout))

        if stderr.strip():
            print("\n[stderr]")
            print(tail_text(stderr))

    if result.returncode != 0:
        diagnostic = textwrap.dedent(
            f"""
            ISOLATED ENVIRONMENT STAGE FAILED

            Stage:
              {stage_name}

            Failing command:
              {readable_command}

            Return code:
              {result.returncode}

            System Python:
              {sys.version}

            System executable:
              {sys.executable}

            Target environment:
              {ENV_DIR}

            Expected environment Python:
              {ENV_PYTHON}

            Log:
              {INSTALL_LOG_PATH}

            Recent stdout:
            {tail_text(stdout)}

            Recent stderr:
            {tail_text(stderr)}

            Action:
              Resolve the reported network, filesystem, uv, or Python download
              error and rerun Cell 3. Do not continue to Cell 4 while this
              stage is failing.
            """
        ).strip()

        append_log(diagnostic)
        raise EnvironmentStageError(diagnostic)

    return result


def calculate_sha256(path: Path) -> str:
    """Calculate SHA-256 for a file without loading it fully into memory."""
    digest = hashlib.sha256()

    with path.open("rb") as file_handle:
        while True:
            block = file_handle.read(1024 * 1024)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


def parse_python_version(version_output: str) -> tuple[int, int, int]:
    """Extract a semantic Python version from command output."""
    match = re.search(
        r"Python\s+(\d+)\.(\d+)\.(\d+)",
        version_output,
    )

    if not match:
        raise EnvironmentStageError(
            "Could not parse Python version from output: "
            f"{version_output!r}"
        )

    return tuple(int(value) for value in match.groups())


def find_uv() -> Optional[Path]:
    """Locate uv on PATH or in the standard installer directories."""
    path_result = shutil.which("uv")

    if path_result:
        return Path(path_result).resolve()

    for candidate in EXPECTED_UV_LOCATIONS:
        if candidate.exists() and os.access(candidate, os.X_OK):
            return candidate.resolve()

    return None


def inspect_environment_python() -> Dict[str, Any]:
    """Return detailed information from the isolated Python interpreter."""
    if not ENV_PYTHON.exists():
        return {
            "valid": False,
            "diagnostic": f"Missing interpreter: {ENV_PYTHON}",
        }

    inspection_code = r"""
import json
import os
import platform
import site
import sys
import sysconfig

data = {
    "version": sys.version,
    "version_info": list(sys.version_info[:5]),
    "executable": sys.executable,
    "prefix": sys.prefix,
    "base_prefix": sys.base_prefix,
    "exec_prefix": sys.exec_prefix,
    "base_exec_prefix": sys.base_exec_prefix,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "implementation": platform.python_implementation(),
    "virtual_environment": os.environ.get("VIRTUAL_ENV"),
    "site_packages": site.getsitepackages(),
    "user_site": site.getusersitepackages(),
    "purelib": sysconfig.get_path("purelib"),
    "platlib": sysconfig.get_path("platlib"),
}
print(json.dumps(data, sort_keys=True))
"""

    result = run_command(
        [str(ENV_PYTHON), "-c", inspection_code],
        stage_name="Inspect isolated Python environment",
        timeout_seconds=120,
        print_output=False,
    )

    try:
        information = json.loads(result.stdout.strip())
    except json.JSONDecodeError as exc:
        raise EnvironmentStageError(
            "The isolated interpreter returned invalid inspection JSON.\n"
            f"Output: {result.stdout!r}\n"
            f"Error: {exc}"
        ) from exc

    version_info = information.get("version_info", [])
    valid_version = (
        len(version_info) >= 2
        and version_info[0] == REQUIRED_PYTHON_MAJOR
        and version_info[1] == REQUIRED_PYTHON_MINOR
    )

    environment_prefix = Path(
        str(information.get("prefix", ""))
    ).resolve()

    valid_prefix = environment_prefix == ENV_DIR.resolve()

    isolated_from_system = (
        Path(str(information["executable"])).resolve()
        != Path(sys.executable).resolve()
    )

    information.update(
        {
            "valid": (
                valid_version
                and valid_prefix
                and isolated_from_system
            ),
            "valid_python_version": valid_version,
            "valid_environment_prefix": valid_prefix,
            "isolated_from_system_python": isolated_from_system,
        }
    )

    return information


# -----------------------------------------------------------------------------
# Initial diagnostics
# -----------------------------------------------------------------------------

print("=" * 88)
print("PINGO TRAINING — CELL 3 ISOLATED PYTHON 3.11 ENVIRONMENT")
print("=" * 88)
print(f"System Python version : {sys.version.splitlines()[0]}")
print(f"System executable     : {sys.executable}")
print(f"Target environment    : {ENV_DIR}")
print(f"FORCE_REBUILD         : {bool(CONFIG.FORCE_REBUILD)}")
print(f"Installation log      : {INSTALL_LOG_PATH}")
print(f"Environment report    : {ENVIRONMENT_REPORT_PATH}")
print("=" * 88)


# -----------------------------------------------------------------------------
# Stage 1: Validate basic system commands
# -----------------------------------------------------------------------------

required_host_commands = (
    "curl",
    "git",
)

missing_host_commands = [
    command
    for command in required_host_commands
    if shutil.which(command) is None
]

if missing_host_commands:
    raise EnvironmentStageError(
        "Required host commands are unavailable: "
        + ", ".join(missing_host_commands)
        + ". Run Cell 2 successfully before Cell 3."
    )


# -----------------------------------------------------------------------------
# Stage 2: Reuse or rebuild an existing environment
# -----------------------------------------------------------------------------

existing_environment_information: Optional[Dict[str, Any]] = None

if ENV_DIR.exists() and not bool(CONFIG.FORCE_REBUILD):
    print(
        "\nExisting environment detected. Validating it before deciding "
        "whether it can be reused."
    )

    try:
        existing_environment_information = inspect_environment_python()
    except Exception as exc:
        existing_environment_information = {
            "valid": False,
            "diagnostic": f"{type(exc).__name__}: {exc}",
        }

    if existing_environment_information.get("valid"):
        print(
            "[OK] Existing /content/pingo-env is a valid isolated "
            "Python 3.11 environment. It will be reused."
        )
    else:
        diagnostic = json.dumps(
            existing_environment_information,
            indent=2,
            ensure_ascii=False,
        )

        raise EnvironmentStageError(
            "An existing environment was found, but it is not a valid "
            "isolated Python 3.11 environment.\n\n"
            f"Path: {ENV_DIR}\n"
            f"Inspection:\n{diagnostic}\n\n"
            "Set FORCE_REBUILD=True in Cell 1, rerun Cell 1, and then rerun "
            "Cell 3. This safety rule prevents silently deleting an existing "
            "environment."
        )

elif ENV_DIR.exists() and bool(CONFIG.FORCE_REBUILD):
    resolved_env = ENV_DIR.resolve()
    protected_paths = {
        Path("/").resolve(),
        Path("/content").resolve(),
        TRAINING_ROOT.resolve(),
        Path.home().resolve(),
    }

    if resolved_env in protected_paths:
        raise EnvironmentStageError(
            f"Refusing to delete protected path: {resolved_env}"
        )

    if resolved_env.name != "pingo-env":
        raise EnvironmentStageError(
            "Refusing to delete an unexpected environment directory: "
            f"{resolved_env}"
        )

    print(
        f"\nFORCE_REBUILD=True: removing only the isolated environment "
        f"directory {resolved_env}"
    )
    shutil.rmtree(resolved_env)


# -----------------------------------------------------------------------------
# Stage 3: Install or locate uv
# -----------------------------------------------------------------------------

UV_PATH = find_uv()
UV_INSTALL_SCRIPT_SHA256: Optional[str] = None
UV_INSTALL_HTTP_STATUS: Optional[int] = None

if UV_PATH is None:
    print("\nuv was not found. Downloading the official installer.")

    request = urllib.request.Request(
        UV_INSTALL_URL,
        headers={
            "User-Agent": "Pingo-OpenWakeWord-Colab/1.0",
            "Accept": "text/plain,*/*",
        },
        method="GET",
    )

    try:
        with urllib.request.urlopen(
            request,
            timeout=120,
        ) as response:
            UV_INSTALL_HTTP_STATUS = int(response.status)
            installer_bytes = response.read()

    except Exception as exc:
        raise EnvironmentStageError(
            "Failed to download the official uv installation script.\n"
            f"URL: {UV_INSTALL_URL}\n"
            f"Expected path: {UV_INSTALL_SCRIPT_PATH}\n"
            f"Error: {type(exc).__name__}: {exc}"
        ) from exc

    if UV_INSTALL_HTTP_STATUS != 200:
        raise EnvironmentStageError(
            "Unexpected HTTP status while downloading uv installer.\n"
            f"URL: {UV_INSTALL_URL}\n"
            f"HTTP status: {UV_INSTALL_HTTP_STATUS}\n"
            f"Expected: 200"
        )

    if len(installer_bytes) < 500:
        raise EnvironmentStageError(
            "Downloaded uv installer is unexpectedly small.\n"
            f"URL: {UV_INSTALL_URL}\n"
            f"Downloaded bytes: {len(installer_bytes)}"
        )

    UV_INSTALL_SCRIPT_PATH.write_bytes(installer_bytes)
    UV_INSTALL_SCRIPT_PATH.chmod(0o700)

    UV_INSTALL_SCRIPT_SHA256 = calculate_sha256(
        UV_INSTALL_SCRIPT_PATH
    )

    UV_DOWNLOAD_METADATA_PATH.write_text(
        json.dumps(
            {
                "url": UV_INSTALL_URL,
                "http_status": UV_INSTALL_HTTP_STATUS,
                "size_bytes": len(installer_bytes),
                "sha256": UV_INSTALL_SCRIPT_SHA256,
                "downloaded_at_utc": dt.datetime.now(
                    dt.timezone.utc
                ).isoformat(),
            },
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )

    print(f"uv installer path      : {UV_INSTALL_SCRIPT_PATH}")
    print(f"uv installer size      : {len(installer_bytes):,} bytes")
    print(f"uv installer SHA-256   : {UV_INSTALL_SCRIPT_SHA256}")

    # Execute the downloaded installer directly with bash.
    # `env=...` is used by run_command; `environment=...` is never used.
    run_command(
        ["bash", str(UV_INSTALL_SCRIPT_PATH)],
        stage_name="Install uv using the official Astral installer",
        timeout_seconds=900,
        extra_env={
            "UV_UNMANAGED_INSTALL": str(
                Path.home() / ".local" / "bin"
            ),
        },
    )

    UV_PATH = find_uv()

if UV_PATH is None:
    raise EnvironmentStageError(
        "The uv installation command completed, but the uv executable "
        "could not be located.\n"
        f"Checked PATH and: {', '.join(str(p) for p in EXPECTED_UV_LOCATIONS)}"
    )

print(f"\n[OK] uv executable: {UV_PATH}")

uv_version_result = run_command(
    [str(UV_PATH), "--version"],
    stage_name="Verify uv installation",
    timeout_seconds=120,
)

UV_VERSION_OUTPUT = uv_version_result.stdout.strip()

if not UV_VERSION_OUTPUT:
    raise EnvironmentStageError(
        "uv returned no version information."
    )


# -----------------------------------------------------------------------------
# Stage 4: Install isolated CPython 3.11 through uv
# -----------------------------------------------------------------------------

run_command(
    [
        str(UV_PATH),
        "python",
        "install",
        "3.11",
    ],
    stage_name="Install an isolated CPython 3.11 runtime with uv",
    timeout_seconds=1800,
)


# -----------------------------------------------------------------------------
# Stage 5: Inspect uv-managed Python installations
# -----------------------------------------------------------------------------

uv_python_list_result = run_command(
    [
        str(UV_PATH),
        "python",
        "list",
        "--only-installed",
    ],
    stage_name="List installed uv Python runtimes",
    timeout_seconds=120,
)

UV_PYTHON_LIST_OUTPUT = uv_python_list_result.stdout.strip()

if "3.11" not in UV_PYTHON_LIST_OUTPUT:
    raise EnvironmentStageError(
        "uv completed Python installation but its installed-runtime list "
        "does not show Python 3.11.\n\n"
        f"uv output:\n{UV_PYTHON_LIST_OUTPUT}"
    )


# -----------------------------------------------------------------------------
# Stage 6: Create the virtual environment with seeded pip
# -----------------------------------------------------------------------------

if not ENV_DIR.exists():
    run_command(
        [
            str(UV_PATH),
            "venv",
            "--python",
            "3.11",
            "--seed",
            str(ENV_DIR),
        ],
        stage_name=(
            "Create /content/pingo-env with Python 3.11 and seeded pip"
        ),
        timeout_seconds=1200,
    )


# -----------------------------------------------------------------------------
# Stage 7: Validate required environment files
# -----------------------------------------------------------------------------

required_environment_files = (
    ENV_PYTHON,
    ENV_PYTHON3,
    ENV_ACTIVATE,
    ENV_PYVENV_CONFIG,
)

missing_environment_files = [
    str(path)
    for path in required_environment_files
    if not path.exists()
]

if missing_environment_files:
    raise EnvironmentStageError(
        "The environment creation command completed but required files "
        "are missing:\n  - "
        + "\n  - ".join(missing_environment_files)
    )

if not os.access(ENV_PYTHON, os.X_OK):
    raise EnvironmentStageError(
        f"Environment Python is not executable: {ENV_PYTHON}"
    )


# -----------------------------------------------------------------------------
# Stage 8: Verify the environment Python version
# -----------------------------------------------------------------------------

python_version_result = run_command(
    [str(ENV_PYTHON), "--version"],
    stage_name="Verify isolated Python version",
    timeout_seconds=120,
)

combined_python_version_output = "\n".join(
    part.strip()
    for part in (
        python_version_result.stdout,
        python_version_result.stderr,
    )
    if part and part.strip()
)

environment_python_version = parse_python_version(
    combined_python_version_output
)

if environment_python_version[:2] != (
    REQUIRED_PYTHON_MAJOR,
    REQUIRED_PYTHON_MINOR,
):
    raise EnvironmentStageError(
        "Incorrect Python version in the isolated environment.\n"
        f"Expected: {REQUIRED_PYTHON_MAJOR}.{REQUIRED_PYTHON_MINOR}.x\n"
        f"Found: {environment_python_version}\n"
        f"Interpreter: {ENV_PYTHON}"
    )


# -----------------------------------------------------------------------------
# Stage 9: Verify pip using `python -m pip`
# -----------------------------------------------------------------------------

pip_version_result = run_command(
    [
        str(ENV_PYTHON),
        "-m",
        "pip",
        "--version",
    ],
    stage_name="Verify pip inside the isolated environment",
    timeout_seconds=120,
)

PIP_VERSION_BEFORE_UPGRADE = pip_version_result.stdout.strip()

if not PIP_VERSION_BEFORE_UPGRADE:
    # `uv venv --seed` should have installed pip. This is a defensive fallback,
    # not the primary path.
    print(
        "Seeded pip did not return version information. "
        "Attempting python -m ensurepip."
    )

    run_command(
        [
            str(ENV_PYTHON),
            "-m",
            "ensurepip",
            "--upgrade",
        ],
        stage_name="Install pip using ensurepip fallback",
        timeout_seconds=300,
    )

    pip_version_result = run_command(
        [
            str(ENV_PYTHON),
            "-m",
            "pip",
            "--version",
        ],
        stage_name="Reverify pip after ensurepip",
        timeout_seconds=120,
    )

    PIP_VERSION_BEFORE_UPGRADE = pip_version_result.stdout.strip()

if not PIP_VERSION_BEFORE_UPGRADE:
    raise EnvironmentStageError(
        "pip is unavailable in the isolated environment even after "
        "`uv venv --seed` and the ensurepip fallback.\n"
        f"Expected interpreter: {ENV_PYTHON}"
    )


# -----------------------------------------------------------------------------
# Stage 10: Upgrade only environment bootstrap tooling
# -----------------------------------------------------------------------------
#
# Do not install NumPy, Torch, TensorFlow, ONNX Runtime, OpenWakeWord, or
# other ML dependencies here. Their compatible versions must be resolved from
# repository metadata in Cells 4 and 5.
#
# -----------------------------------------------------------------------------

run_command(
    [
        str(ENV_PYTHON),
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--disable-pip-version-check",
        "pip",
        "setuptools",
        "wheel",
        "packaging",
    ],
    stage_name=(
        "Upgrade packaging bootstrap tools inside /content/pingo-env"
    ),
    timeout_seconds=900,
)


# -----------------------------------------------------------------------------
# Stage 11: Inspect environment and confirm isolation
# -----------------------------------------------------------------------------

ENVIRONMENT_INFORMATION = inspect_environment_python()

if not ENVIRONMENT_INFORMATION.get("valid"):
    raise EnvironmentStageError(
        "The newly created environment failed isolation validation.\n\n"
        + json.dumps(
            ENVIRONMENT_INFORMATION,
            indent=2,
            ensure_ascii=False,
        )
    )

if Path(ENVIRONMENT_INFORMATION["executable"]).resolve() != (
    ENV_PYTHON.resolve()
):
    raise EnvironmentStageError(
        "The isolated interpreter reported an unexpected executable.\n"
        f"Expected: {ENV_PYTHON.resolve()}\n"
        f"Reported: {ENVIRONMENT_INFORMATION['executable']}"
    )

if Path(sys.executable).resolve() == ENV_PYTHON.resolve():
    raise EnvironmentStageError(
        "Isolation check failed: the environment Python unexpectedly "
        "matches Colab's system Python."
    )


# -----------------------------------------------------------------------------
# Stage 12: Verify pip and packaging versions after upgrade
# -----------------------------------------------------------------------------

packaging_version_code = r"""
import importlib.metadata
import json
import sys

packages = ("pip", "setuptools", "wheel", "packaging")
versions = {}

for package in packages:
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package] = None

print(json.dumps({
    "python": sys.version,
    "executable": sys.executable,
    "packages": versions,
}, sort_keys=True))
"""

packaging_result = run_command(
    [
        str(ENV_PYTHON),
        "-c",
        packaging_version_code,
    ],
    stage_name="Inspect isolated packaging tool versions",
    timeout_seconds=120,
    print_output=False,
)

try:
    PACKAGING_INFORMATION = json.loads(
        packaging_result.stdout.strip()
    )
except json.JSONDecodeError as exc:
    raise EnvironmentStageError(
        "Failed to parse packaging-version inspection output.\n"
        f"Output: {packaging_result.stdout!r}\n"
        f"Error: {exc}"
    ) from exc

missing_packaging_tools = [
    package
    for package, version
    in PACKAGING_INFORMATION["packages"].items()
    if version is None
]

if missing_packaging_tools:
    raise EnvironmentStageError(
        "Required bootstrap packages are missing from the isolated "
        "environment: "
        + ", ".join(missing_packaging_tools)
    )


# -----------------------------------------------------------------------------
# Stage 13: Verify that system site-packages are not inherited
# -----------------------------------------------------------------------------

site_isolation_code = r"""
import json
import pathlib
import site
import sys

prefix = pathlib.Path(sys.prefix).resolve()
paths = [pathlib.Path(p).resolve() for p in sys.path if p]

outside_site_packages = []

for path in paths:
    text = str(path)
    if "site-packages" in text and prefix not in path.parents and path != prefix:
        outside_site_packages.append(text)

print(json.dumps({
    "prefix": str(prefix),
    "sys_path": [str(path) for path in paths],
    "outside_site_packages": outside_site_packages,
    "enable_user_site": site.ENABLE_USER_SITE,
}, sort_keys=True))
"""

site_isolation_result = run_command(
    [
        str(ENV_PYTHON),
        "-c",
        site_isolation_code,
    ],
    stage_name="Verify site-package isolation",
    timeout_seconds=120,
    print_output=False,
)

try:
    SITE_ISOLATION_INFORMATION = json.loads(
        site_isolation_result.stdout.strip()
    )
except json.JSONDecodeError as exc:
    raise EnvironmentStageError(
        "Failed to parse site-isolation inspection output.\n"
        f"Output: {site_isolation_result.stdout!r}\n"
        f"Error: {exc}"
    ) from exc

outside_site_packages = SITE_ISOLATION_INFORMATION.get(
    "outside_site_packages",
    [],
)

if outside_site_packages:
    raise EnvironmentStageError(
        "The isolated environment appears to inherit external "
        "site-packages:\n  - "
        + "\n  - ".join(outside_site_packages)
    )


# -----------------------------------------------------------------------------
# Stage 14: Create a reusable environment command helper
# -----------------------------------------------------------------------------
#
# Later notebook cells should invoke:
#
#     /content/pingo-env/bin/python -m pip ...
#     /content/pingo-env/bin/python script.py
#
# Activating a venv in one Colab subprocess does not reliably affect later
# notebook cells, so absolute interpreter paths are preferred.
#
# -----------------------------------------------------------------------------

ENV_HELPER_PATH = CONFIG_DIR / "environment_paths.json"

ENV_HELPER_DATA = {
    "environment_root": str(ENV_DIR),
    "python": str(ENV_PYTHON),
    "python3": str(ENV_PYTHON3),
    "pip_command": [
        str(ENV_PYTHON),
        "-m",
        "pip",
    ],
    "activate_script": str(ENV_ACTIVATE),
    "uv": str(UV_PATH),
}

ENV_HELPER_PATH.write_text(
    json.dumps(
        ENV_HELPER_DATA,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


# -----------------------------------------------------------------------------
# Stage 15: Save exact Cell 3 report
# -----------------------------------------------------------------------------

CELL_COMPLETED_AT = dt.datetime.now(dt.timezone.utc)

ENVIRONMENT_REPORT = {
    "stage": "Cell 3 — Isolated Python environment",
    "passed": True,
    "started_at_utc": CELL_STARTED_AT.isoformat(),
    "completed_at_utc": CELL_COMPLETED_AT.isoformat(),
    "system": {
        "python_version": sys.version,
        "python_executable": sys.executable,
        "platform": platform.platform(),
        "machine": platform.machine(),
    },
    "uv": {
        "path": str(UV_PATH),
        "version_output": UV_VERSION_OUTPUT,
        "installed_python_list": UV_PYTHON_LIST_OUTPUT,
        "installer_url": UV_INSTALL_URL,
        "installer_http_status": UV_INSTALL_HTTP_STATUS,
        "installer_path": (
            str(UV_INSTALL_SCRIPT_PATH)
            if UV_INSTALL_SCRIPT_PATH.exists()
            else None
        ),
        "installer_sha256": UV_INSTALL_SCRIPT_SHA256,
    },
    "isolated_environment": {
        "path": str(ENV_DIR),
        "python_path": str(ENV_PYTHON),
        "python_version": list(environment_python_version),
        "pip_before_upgrade": PIP_VERSION_BEFORE_UPGRADE,
        "inspection": ENVIRONMENT_INFORMATION,
        "packaging": PACKAGING_INFORMATION,
        "site_isolation": SITE_ISOLATION_INFORMATION,
        "pyvenv_config": (
            ENV_PYVENV_CONFIG.read_text(
                encoding="utf-8",
                errors="replace",
            )
            if ENV_PYVENV_CONFIG.exists()
            else None
        ),
    },
    "paths": {
        "log": str(INSTALL_LOG_PATH),
        "report": str(ENVIRONMENT_REPORT_PATH),
        "helper": str(ENV_HELPER_PATH),
    },
}

ENVIRONMENT_REPORT_PATH.write_text(
    json.dumps(
        ENVIRONMENT_REPORT,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)

append_log(
    "\nFINAL ENVIRONMENT REPORT\n"
    + json.dumps(
        ENVIRONMENT_REPORT,
        indent=2,
        ensure_ascii=False,
    )
)


# -----------------------------------------------------------------------------
# Completion report
# -----------------------------------------------------------------------------

print("\n" + "=" * 88)
print("CELL 3 COMPLETED SUCCESSFULLY")
print("=" * 88)
print(f"uv executable            : {UV_PATH}")
print(f"uv version               : {UV_VERSION_OUTPUT}")
print(f"Environment directory    : {ENV_DIR}")
print(
    "Environment Python       : "
    f"{ENVIRONMENT_INFORMATION['version'].splitlines()[0]}"
)
print(f"Environment executable   : {ENV_PYTHON}")
print(f"System executable        : {sys.executable}")
print(
    "Isolated from system     : "
    f"{ENVIRONMENT_INFORMATION['isolated_from_system_python']}"
)
print(f"Environment prefix valid : {ENVIRONMENT_INFORMATION['valid_environment_prefix']}")
print(f"pip version              : {PACKAGING_INFORMATION['packages']['pip']}")
print(
    f"setuptools version       : "
    f"{PACKAGING_INFORMATION['packages']['setuptools']}"
)
print(f"wheel version            : {PACKAGING_INFORMATION['packages']['wheel']}")
print(
    f"packaging version        : "
    f"{PACKAGING_INFORMATION['packages']['packaging']}"
)
print(f"External site-packages   : {outside_site_packages}")
print(f"Environment report       : {ENVIRONMENT_REPORT_PATH}")
print(f"Environment helper       : {ENV_HELPER_PATH}")
print(f"Full log                 : {INSTALL_LOG_PATH}")
print()
print("Use this command form in all later cells:")
print(f"  {ENV_PYTHON} -m pip <arguments>")
print()
print("Do not use Colab's system pip for the Pingo training environment.")
print()
print(
    "Next: Cell 4 will clone the current official OpenWakeWord and "
    "Piper sample-generator repositories, record their exact Git commit "
    "hashes, and inspect their current dependency files and training APIs "
    "before any ML package versions are selected."
)
print("=" * 88)

PINGO TRAINING — CELL 3 ISOLATED PYTHON 3.11 ENVIRONMENT
System Python version : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
System executable     : /usr/bin/python3
Target environment    : /content/pingo-env
FORCE_REBUILD         : False
Installation log      : /content/pingo_training/logs/cell_03_isolated_environment.log
Environment report    : /content/pingo_training/config/isolated_python_environment.json

[OK] uv executable: /usr/local/bin/uv

STAGE: Verify uv installation
COMMAND: /usr/local/bin/uv --version
WORKING DIRECTORY: /content
uv 0.11.19 (x86_64-unknown-linux-gnu)


STAGE: Install an isolated CPython 3.11 runtime with uv
COMMAND: /usr/local/bin/uv python install 3.11
WORKING DIRECTORY: /content

[stderr]
Installed Python 3.11.15 in 1.89s
 + cpython-3.11.15-linux-x86_64-gnu (python3.11)


STAGE: List installed uv Python runtimes
COMMAND: /usr/local/bin/uv python list --only-installed
WORKING DIRECTORY: /content
cpython-3.12.13-linux-x86_64-gnu    /usr/local/bin/pyt

In [ ]:
# =============================================================================
# Cell 4 — Clone OpenWakeWord and Piper Sample Generator
# =============================================================================

from pathlib import Path
import json
import shutil
import subprocess
import datetime as dt

TRAINING_ROOT = Path("/content/pingo_training")
REPOSITORIES_DIR = TRAINING_ROOT / "repositories"
CONFIG_DIR = TRAINING_ROOT / "config"
LOGS_DIR = TRAINING_ROOT / "logs"

OPENWAKEWORD_REPO = REPOSITORIES_DIR / "openWakeWord"
PIPER_REPO = REPOSITORIES_DIR / "piper-sample-generator"

OPENWAKEWORD_URL = "https://github.com/dscripka/openWakeWord.git"
PIPER_URL = "https://github.com/rhasspy/piper-sample-generator.git"

for directory in (REPOSITORIES_DIR, CONFIG_DIR, LOGS_DIR):
    directory.mkdir(parents=True, exist_ok=True)


def run_command(command, *, cwd=None, timeout=1800):
    print("\n" + "=" * 88)
    print("COMMAND:", " ".join(str(part) for part in command))
    print("WORKING DIRECTORY:", cwd or "/content")
    print("=" * 88)

    result = subprocess.run(
        [str(part) for part in command],
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        timeout=timeout,
        check=False,
    )

    if result.stdout.strip():
        print(result.stdout)

    if result.stderr.strip():
        print("[stderr]")
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed with return code {result.returncode}:\n"
            + " ".join(str(part) for part in command)
        )

    return result


def validate_git_repository(path: Path) -> bool:
    return (
        path.is_dir()
        and (path / ".git").is_dir()
        and shutil.which("git") is not None
    )


def clone_or_reuse(name: str, url: str, destination: Path):
    print(f"\nProcessing {name}")
    print(f"Destination: {destination}")

    if validate_git_repository(destination):
        print("[OK] Existing Git repository found. Reusing it.")

        run_command(
            ["git", "remote", "-v"],
            cwd=destination,
            timeout=120,
        )

        run_command(
            ["git", "fetch", "--all", "--tags", "--prune"],
            cwd=destination,
            timeout=1800,
        )

        return

    if destination.exists():
        print(
            f"Removing incomplete non-Git directory: {destination}"
        )
        shutil.rmtree(destination)

    run_command(
        [
            "git",
            "clone",
            "--recurse-submodules",
            url,
            str(destination),
        ],
        cwd=REPOSITORIES_DIR,
        timeout=1800,
    )

    if not validate_git_repository(destination):
        raise RuntimeError(
            f"{name} clone finished but repository validation failed: "
            f"{destination}"
        )

    print(f"[OK] {name} cloned successfully.")


print("=" * 88)
print("PINGO TRAINING — CELL 4 REPOSITORY SETUP")
print("=" * 88)
print(f"Repository root : {REPOSITORIES_DIR}")
print(f"OpenWakeWord    : {OPENWAKEWORD_REPO}")
print(f"Piper generator : {PIPER_REPO}")

clone_or_reuse(
    "OpenWakeWord",
    OPENWAKEWORD_URL,
    OPENWAKEWORD_REPO,
)

clone_or_reuse(
    "Piper Sample Generator",
    PIPER_URL,
    PIPER_REPO,
)


def repository_information(path: Path):
    commit = run_command(
        ["git", "rev-parse", "HEAD"],
        cwd=path,
        timeout=120,
    ).stdout.strip()

    branch_result = run_command(
        ["git", "branch", "--show-current"],
        cwd=path,
        timeout=120,
    )

    branch = branch_result.stdout.strip() or "detached-head"

    remote = run_command(
        ["git", "remote", "get-url", "origin"],
        cwd=path,
        timeout=120,
    ).stdout.strip()

    status = run_command(
        ["git", "status", "--short"],
        cwd=path,
        timeout=120,
    ).stdout.strip()

    return {
        "path": str(path),
        "remote": remote,
        "branch": branch,
        "commit": commit,
        "clean": not bool(status),
        "status": status,
    }


report = {
    "created_at_utc": dt.datetime.now(
        dt.timezone.utc
    ).isoformat(),
    "openwakeword": repository_information(
        OPENWAKEWORD_REPO
    ),
    "piper_sample_generator": repository_information(
        PIPER_REPO
    ),
}

report_path = CONFIG_DIR / "repository_versions.json"

report_path.write_text(
    json.dumps(
        report,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

print("\n" + "=" * 88)
print("CELL 4 COMPLETED SUCCESSFULLY")
print("=" * 88)
print(f"OpenWakeWord repository:")
print(f"  {OPENWAKEWORD_REPO}")
print(f"Piper Sample Generator repository:")
print(f"  {PIPER_REPO}")
print(f"Repository report:")
print(f"  {report_path}")
print("=" * 88)

PINGO TRAINING — CELL 4 REPOSITORY SETUP
Repository root : /content/pingo_training/repositories
OpenWakeWord    : /content/pingo_training/repositories/openWakeWord
Piper generator : /content/pingo_training/repositories/piper-sample-generator

Processing OpenWakeWord
Destination: /content/pingo_training/repositories/openWakeWord

COMMAND: git clone --recurse-submodules https://github.com/dscripka/openWakeWord.git /content/pingo_training/repositories/openWakeWord
WORKING DIRECTORY: /content/pingo_training/repositories
[stderr]
Cloning into '/content/pingo_training/repositories/openWakeWord'...

[OK] OpenWakeWord cloned successfully.

Processing Piper Sample Generator
Destination: /content/pingo_training/repositories/piper-sample-generator

COMMAND: git clone --recurse-submodules https://github.com/rhasspy/piper-sample-generator.git /content/pingo_training/repositories/piper-sample-generator
WORKING DIRECTORY: /content/pingo_training/repositories
[stderr]
Cloning into '/content/pingo_trai

In [ ]:
from pathlib import Path

repositories = {
    "OpenWakeWord": Path(
        "/content/pingo_training/repositories/openWakeWord"
    ),
    "Piper Sample Generator": Path(
        "/content/pingo_training/repositories/piper-sample-generator"
    ),
}

for name, path in repositories.items():
    print(f"\n{name}")
    print("-" * 70)
    print("Path exists :", path.exists())
    print("Git exists  :", (path / ".git").exists())
    print("Path        :", path)

    if not path.exists() or not (path / ".git").exists():
        raise FileNotFoundError(
            f"{name} repository is missing or incomplete: {path}"
        )

print("\nBoth repositories are ready.")


OpenWakeWord
----------------------------------------------------------------------
Path exists : True
Git exists  : True
Path        : /content/pingo_training/repositories/openWakeWord

Piper Sample Generator
----------------------------------------------------------------------
Path exists : True
Git exists  : True
Path        : /content/pingo_training/repositories/piper-sample-generator

Both repositories are ready.


In [ ]:
from pathlib import Path
import shutil
import subprocess

# Search for every Piper Sample Generator Git repository under /content.
candidates = []

for path in Path("/content").rglob("piper-sample-generator"):
    if path.is_dir() and (path / ".git").exists():
        candidates.append(path)

if not candidates:
    raise FileNotFoundError(
        "Could not find the Piper Sample Generator Git repository.\n"
        "Rerun Cell 4 from the beginning so it clones the repository."
    )

print("Repositories found:")

for index, path in enumerate(candidates, start=1):
    print(f"{index}. {path}")

# Normally there should be only one relevant repository.
repo = candidates[0]

print(f"\nUsing repository: {repo}")

# Remove all generated egg-info directories inside the repository.
egg_info_directories = list(repo.rglob("*.egg-info"))

if egg_info_directories:
    for egg_info in egg_info_directories:
        if egg_info.is_dir():
            shutil.rmtree(egg_info)
            print(f"Removed generated metadata: {egg_info}")
else:
    print("No egg-info directories were found.")

# Show the remaining Git changes.
result = subprocess.run(
    ["git", "status", "--short"],
    cwd=str(repo),
    text=True,
    capture_output=True,
    check=True,
)

print("\nCurrent Git status:")

if result.stdout.strip():
    print(result.stdout)
else:
    print("Clean repository")

Repositories found:
1. /content/pingo_training/repositories/piper-sample-generator

Using repository: /content/pingo_training/repositories/piper-sample-generator
No egg-info directories were found.

Current Git status:
Clean repository


In [ ]:
from pathlib import Path

repo_root = Path(
    "/content/pingo_training/repositories"
)

openwakeword_repo = repo_root / "openWakeWord"
piper_repo = repo_root / "piper-sample-generator"

print("Repository root:")
print(repo_root)

print("\nOpenWakeWord")
print("Path exists :", openwakeword_repo.exists())
print("Git exists  :", (openwakeword_repo / ".git").exists())
print("Path        :", openwakeword_repo)

print("\nPiper Sample Generator")
print("Path exists :", piper_repo.exists())
print("Git exists  :", (piper_repo / ".git").exists())
print("Path        :", piper_repo)

if not (openwakeword_repo / ".git").exists():
    raise FileNotFoundError(
        f"OpenWakeWord is missing: {openwakeword_repo}\n"
        "Rerun Cell 4."
    )

if not (piper_repo / ".git").exists():
    raise FileNotFoundError(
        f"Piper repository is missing: {piper_repo}\n"
        "Rerun Cell 4."
    )

print("\nBoth repositories are ready for Cell 5.")

Repository root:
/content/pingo_training/repositories

OpenWakeWord
Path exists : True
Git exists  : True
Path        : /content/pingo_training/repositories/openWakeWord

Piper Sample Generator
Path exists : True
Git exists  : True
Path        : /content/pingo_training/repositories/piper-sample-generator

Both repositories are ready for Cell 5.


In [ ]:
# =============================================================================
# Cell 5 — Install repository and training dependencies
# =============================================================================
#
# This cell:
#   1. Uses only /content/pingo-env/bin/python.
#   2. Installs OpenWakeWord from the exact Cell 4 checkout.
#   3. Installs Piper Sample Generator from the exact Cell 4 checkout.
#   4. Installs the audio, data, evaluation, ONNX, and training toolchain.
#   5. Selects the correct PyTorch installation path for GPU or CPU.
#   6. Runs `pip check`.
#   7. Saves a complete `pip freeze` and dependency report.
#
# It does not install packages into Colab's system Python.
#
# =============================================================================

from __future__ import annotations

import datetime as dt
import importlib.util
import json
import os
import platform
import re
import shlex
import shutil
import subprocess
import sys
import textwrap
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence

# Prevent notebook-specific Matplotlib backend settings from breaking
# TensorFlow/Keras imports inside the isolated environment.
os.environ["MPLBACKEND"] = "Agg"

# -----------------------------------------------------------------------------
# Configuration and paths
# -----------------------------------------------------------------------------

try:
    CONFIG
except NameError:
    class _FallbackConfig:
        TRAINING_ROOT = "/content/pingo_training"
        LOGS_DIR = "/content/pingo_training/logs"
        CONFIG_DIR = "/content/pingo_training/config"
        CACHE_DIR = "/content/pingo_training/cache"
        REPOSITORIES_DIR = (
            "/content/pingo_training/repositories"
        )
        ENV_DIR = "/content/pingo-env"

    CONFIG = _FallbackConfig()


TRAINING_ROOT = Path(CONFIG.TRAINING_ROOT)
LOGS_DIR = Path(CONFIG.LOGS_DIR)
CONFIG_DIR = Path(CONFIG.CONFIG_DIR)
CACHE_DIR = Path(CONFIG.CACHE_DIR)

REPOS_DIR = Path(
    getattr(
        CONFIG,
        "REPOSITORIES_DIR",
        TRAINING_ROOT / "repositories",
    )
)

ENV_DIR = Path(CONFIG.ENV_DIR)
ENV_PYTHON = ENV_DIR / "bin" / "python"

OPENWAKEWORD_REPO = REPOS_DIR / "openWakeWord"
PIPER_GENERATOR_REPO = (
    REPOS_DIR / "piper-sample-generator"
)

CELL_LOG_PATH = (
    LOGS_DIR / "cell_05_dependency_installation.log"
)
PIP_FREEZE_PATH = (
    CONFIG_DIR / "requirements_cell_05_freeze.txt"
)
DEPENDENCY_REPORT_PATH = (
    CONFIG_DIR / "dependency_report.json"
)

for directory in (
    LOGS_DIR,
    CONFIG_DIR,
    CACHE_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)



# -----------------------------------------------------------------------------
# Execution helper
# -----------------------------------------------------------------------------

class DependencyStageError(RuntimeError):
    """Raised when package installation or validation fails."""


CELL_STARTED_AT = dt.datetime.now(dt.timezone.utc)

CELL_LOG_PATH.write_text(
    (
        "PINGO PYTHON DEPENDENCY INSTALLATION LOG\n"
        f"Started UTC: {CELL_STARTED_AT.isoformat()}\n"
        f"Environment Python: {ENV_PYTHON}\n"
        + "=" * 88
        + "\n"
    ),
    encoding="utf-8",
)


def append_log(message: str) -> None:
    with CELL_LOG_PATH.open("a", encoding="utf-8") as file_handle:
        file_handle.write(message.rstrip() + "\n")


def format_command(command: Sequence[str]) -> str:
    return " ".join(shlex.quote(str(part)) for part in command)


def tail_text(text: str, max_lines: int = 120) -> str:
    lines = text.splitlines()

    if len(lines) <= max_lines:
        return text

    return "\n".join(
        [
            f"... showing final {max_lines} lines ...",
            *lines[-max_lines:],
        ]
    )


def run_command(
    command: Sequence[str],
    *,
    stage_name: str,
    cwd: Optional[Path] = None,
    timeout_seconds: int = 1800,
    print_output: bool = True,
) -> subprocess.CompletedProcess[str]:
    command = [str(item) for item in command]
    readable = format_command(command)

    environment = os.environ.copy()
    environment.update(
      {
          "PYTHONUNBUFFERED": "1",
          "PIP_DISABLE_PIP_VERSION_CHECK": "1",
          "PIP_NO_INPUT": "1",
          "UV_LINK_MODE": "copy",
          "TOKENIZERS_PARALLELISM": "false",

          # Prevent Colab's notebook-only Matplotlib backend from being
          # inherited by subprocesses running inside /content/pingo-env.
          "MPLBACKEND": "Agg",
      }
    )

    working_directory = (
        str(cwd.resolve())
        if cwd is not None
        else (
            "/content"
            if Path("/content").exists()
            else str(Path.cwd())
        )
    )

    header = (
        "\n"
        + "=" * 88
        + "\n"
        + f"STAGE: {stage_name}\n"
        + f"COMMAND: {readable}\n"
        + f"CWD: {working_directory}\n"
        + "=" * 88
    )

    print(header)
    append_log(header)

    try:
        result = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=False,
            cwd=working_directory,
            timeout=timeout_seconds,
            env=environment,
        )
    except Exception as exc:
        raise DependencyStageError(
            f"{stage_name} could not execute.\n"
            f"Command: {readable}\n"
            f"Error: {type(exc).__name__}: {exc}"
        ) from exc

    stdout = result.stdout or ""
    stderr = result.stderr or ""

    append_log("\n[STDOUT]\n" + stdout)
    append_log("\n[STDERR]\n" + stderr)
    append_log(f"\n[RETURN CODE]\n{result.returncode}")

    if print_output and stdout.strip():
        print(tail_text(stdout))

    if print_output and stderr.strip():
        print("[stderr]")
        print(tail_text(stderr))

    if result.returncode != 0:
        raise DependencyStageError(
            textwrap.dedent(
                f"""
                DEPENDENCY INSTALLATION FAILED

                Stage:
                  {stage_name}

                Command:
                  {readable}

                Return code:
                  {result.returncode}

                Recent stdout:
                {tail_text(stdout)}

                Recent stderr:
                {tail_text(stderr)}

                Full log:
                  {CELL_LOG_PATH}
                """
            ).strip()
        )

    return result


def pip_install(
    packages: Sequence[str],
    *,
    stage_name: str,
    extra_arguments: Optional[Sequence[str]] = None,
    timeout_seconds: int = 1800,
) -> None:
    command = [
        str(ENV_PYTHON),
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--prefer-binary",
    ]

    if extra_arguments:
        command.extend(extra_arguments)

    command.extend(packages)

    run_command(
        command,
        stage_name=stage_name,
        timeout_seconds=timeout_seconds,
    )


# -----------------------------------------------------------------------------
# Preconditions
# -----------------------------------------------------------------------------

if not ENV_PYTHON.exists():
    raise DependencyStageError(
        f"Missing environment interpreter: {ENV_PYTHON}\n"
        "Run Cell 3 first."
    )

if not (OPENWAKEWORD_REPO / ".git").exists():
    raise DependencyStageError(
        f"Missing OpenWakeWord repository: {OPENWAKEWORD_REPO}\n"
        "Run Cell 4 first."
    )

if not (PIPER_GENERATOR_REPO / ".git").exists():
    raise DependencyStageError(
        f"Missing Piper repository: {PIPER_GENERATOR_REPO}\n"
        "Run Cell 4 first."
    )


# -----------------------------------------------------------------------------
# Validate Python
# -----------------------------------------------------------------------------

python_result = run_command(
    [
        str(ENV_PYTHON),
        "-c",
        (
            "import json, sys; "
            "print(json.dumps({"
            "'version': sys.version, "
            "'executable': sys.executable, "
            "'version_info': list(sys.version_info[:3])"
            "}))"
        ),
    ],
    stage_name="Validate isolated Python before installation",
    timeout_seconds=120,
    print_output=False,
)

python_information = json.loads(python_result.stdout)

if python_information["version_info"][:2] != [3, 11]:
    raise DependencyStageError(
        "Cell 5 requires Python 3.11.\n"
        f"Detected: {python_information}"
    )

# -----------------------------------------------------------------------------
# Detect NVIDIA GPU availability without importing PyTorch
# -----------------------------------------------------------------------------

NVIDIA_SMI = shutil.which("nvidia-smi")

GPU_AVAILABLE = False
GPU_INFORMATION: Dict[str, Any] = {
    "nvidia_smi_available": NVIDIA_SMI is not None,
    "gpu_available": False,
    "details": None,
    "detection_error": None,
}

if NVIDIA_SMI is not None:
    try:
        gpu_result = run_command(
            [
                NVIDIA_SMI,
                "--query-gpu=name,driver_version,memory.total",
                "--format=csv,noheader",
            ],
            stage_name="Inspect NVIDIA GPU",
            timeout_seconds=120,
            print_output=False,
        )

        gpu_details = gpu_result.stdout.strip()

        if gpu_details:
            GPU_AVAILABLE = True
            GPU_INFORMATION["gpu_available"] = True
            GPU_INFORMATION["details"] = gpu_details

    except Exception as exc:
        GPU_INFORMATION["detection_error"] = (
            f"{type(exc).__name__}: {exc}"
        )
        GPU_AVAILABLE = False

if GPU_AVAILABLE:
    gpu_result = run_command(
        [
            NVIDIA_SMI,
            "--query-gpu=name,driver_version,memory.total",
            "--format=csv,noheader",
        ],
        stage_name="Inspect NVIDIA GPU",
        timeout_seconds=120,
        print_output=False,
    )
    GPU_INFORMATION["details"] = gpu_result.stdout.strip()

print("=" * 88)
print("CELL 5 — DEPENDENCY INSTALLATION")
print("=" * 88)
print(f"Python          : {python_information['version']}")
print(f"Environment     : {ENV_DIR}")
print(f"NVIDIA detected : {GPU_AVAILABLE}")

if GPU_INFORMATION["details"]:
    print(f"GPU             : {GPU_INFORMATION['details']}")

print("=" * 88)


# -----------------------------------------------------------------------------
# Stage 1: Packaging and build tools
# -----------------------------------------------------------------------------

pip_install(
    [
        "pip",
        "setuptools",
        "wheel",
        "packaging",
        "build",
        "cython",
    ],
    stage_name="Install Python build and packaging tools",
    timeout_seconds=900,
)


# -----------------------------------------------------------------------------
# Stage 2: Install hardware-appropriate PyTorch
# -----------------------------------------------------------------------------

if GPU_AVAILABLE:
    print("\nInstalling GPU-capable PyTorch and TorchAudio...")

    pip_install(
        [
            "torch",
            "torchaudio",
        ],
        stage_name="Install GPU-capable PyTorch and TorchAudio",
        timeout_seconds=3600,
    )

else:
    print(
        "\nNo NVIDIA GPU detected. Installing CPU-only PyTorch "
        "to avoid unnecessary CUDA packages."
    )

    pip_install(
        [
            "torch",
            "torchaudio",
        ],
        stage_name="Install CPU-only PyTorch and TorchAudio",
        extra_arguments=[
            "--index-url",
            "https://download.pytorch.org/whl/cpu",
        ],
        timeout_seconds=2400,
    )


# -----------------------------------------------------------------------------
# Stage 3: Core scientific, audio, and data stack
# -----------------------------------------------------------------------------

CORE_PACKAGES = [
    "numpy",
    "scipy",
    "pandas",
    "scikit-learn",
    "matplotlib",
    "tqdm",
    "pyyaml",
    "requests",
    "soundfile",
    "librosa",
    "audioread",
    "resampy",
    "audiomentations",
    "torch-audiomentations",
    "webrtcvad-wheels",
    "h5py",
    "joblib",
    "psutil",
    "rich",
    "click",
    "typer",
    "huggingface-hub",
    "datasets",
]

pip_install(
    CORE_PACKAGES,
    stage_name="Install scientific, audio, augmentation, and data packages",
    timeout_seconds=2400,
)

# -----------------------------------------------------------------------------
# Validate PyTorch installation immediately
# -----------------------------------------------------------------------------

torch_validation_result = run_command(
    [
        str(ENV_PYTHON),
        "-c",
        (
            "import json, torch; "
            "print(json.dumps({"
            "'version': torch.__version__, "
            "'cuda_available': torch.cuda.is_available(), "
            "'cuda_version': torch.version.cuda, "
            "'gpu_count': torch.cuda.device_count(), "
            "'gpu_names': ["
            "torch.cuda.get_device_name(i) "
            "for i in range(torch.cuda.device_count())"
            "]"
            "}))"
        ),
    ],
    stage_name="Validate PyTorch hardware support",
    timeout_seconds=300,
    print_output=False,
)

try:
    INITIAL_TORCH_INFORMATION = json.loads(
        torch_validation_result.stdout.strip().splitlines()[-1]
    )
except Exception as exc:
    raise DependencyStageError(
        "Could not parse PyTorch validation output.\n"
        f"Output:\n{torch_validation_result.stdout}\n"
        f"Error: {exc}"
    ) from exc

print("\nPyTorch hardware validation:")
print(
    json.dumps(
        INITIAL_TORCH_INFORMATION,
        indent=2,
        sort_keys=True,
    )
)

if GPU_AVAILABLE and not INITIAL_TORCH_INFORMATION["cuda_available"]:
  raise DependencyStageError(
        "NVIDIA GPU detected, but PyTorch CUDA is unavailable.\n"
        "Training approximately 50,000 synthetic samples on CPU is not "
        "recommended.\n"
        "Fix the PyTorch installation before continuing."
    )

if not GPU_AVAILABLE:
    print(
        "\nWARNING: No NVIDIA GPU is available. Installation validation "
        "can continue, but full wake-word generation and training will be "
        "very slow."
    )

# -----------------------------------------------------------------------------
# Stage 4: ONNX and evaluation stack
# -----------------------------------------------------------------------------

ONNX_PACKAGES = [
    "onnx",
    "onnxruntime",
    "onnxsim",
    "skl2onnx",
]

pip_install(
    ONNX_PACKAGES,
    stage_name="Install ONNX export and inference packages",
    timeout_seconds=1800,
)


# -----------------------------------------------------------------------------
# Stage 5: TensorFlow training support
# -----------------------------------------------------------------------------
#
# OpenWakeWord's repository history includes TensorFlow/Keras-based training
# utilities. Install current compatible Python 3.11 packages, then validate the
# actual imported API in Cell 6.
#
# -----------------------------------------------------------------------------

TENSORFLOW_PACKAGES = [
    "tensorflow",
    "tensorflow-probability",
    "tf-keras",
]

pip_install(
    TENSORFLOW_PACKAGES,
    stage_name="Install TensorFlow training packages",
    timeout_seconds=3000,
)


# -----------------------------------------------------------------------------
# Stage 6: Install exact repository checkouts
# -----------------------------------------------------------------------------
#
# --no-deps is deliberately NOT used. Repository-declared requirements remain
# authoritative. `pip check` below will detect unresolved conflicts.
#
# -----------------------------------------------------------------------------

run_command(
    [
        str(ENV_PYTHON),
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--prefer-binary",
        "-e",
        str(OPENWAKEWORD_REPO),
    ],
    stage_name="Install OpenWakeWord from the Cell 4 checkout",
    timeout_seconds=1800,
)

run_command(
    [
        str(ENV_PYTHON),
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--prefer-binary",
        "-e",
        str(PIPER_GENERATOR_REPO),
    ],
    stage_name="Install Piper Sample Generator from the Cell 4 checkout",
    timeout_seconds=2400,
)

# -----------------------------------------------------------------------------
# Validate available disk space
# -----------------------------------------------------------------------------

disk_usage = shutil.disk_usage("/content")

DISK_TOTAL_GB = disk_usage.total / (1024 ** 3)
DISK_USED_GB = disk_usage.used / (1024 ** 3)
DISK_FREE_GB = disk_usage.free / (1024 ** 3)

print("\nDisk-space validation")
print(f"  Total : {DISK_TOTAL_GB:.2f} GB")
print(f"  Used  : {DISK_USED_GB:.2f} GB")
print(f"  Free  : {DISK_FREE_GB:.2f} GB")

MINIMUM_FREE_GB = 12.0

if DISK_FREE_GB < MINIMUM_FREE_GB:
    raise DependencyStageError(
        f"Insufficient free disk space: {DISK_FREE_GB:.2f} GB.\n"
        f"At least {MINIMUM_FREE_GB:.2f} GB is required before "
        "installing the complete training environment."
    )

# -----------------------------------------------------------------------------
# Stage 7: pip dependency consistency check
# -----------------------------------------------------------------------------

pip_check_result = run_command(
    [
        str(ENV_PYTHON),
        "-m",
        "pip",
        "check",
    ],
    stage_name="Validate installed dependency consistency",
    timeout_seconds=300,
)

PIP_CHECK_OUTPUT = pip_check_result.stdout.strip()

if (
    PIP_CHECK_OUTPUT
    and "No broken requirements found" not in PIP_CHECK_OUTPUT
):
    raise DependencyStageError(
        "pip check did not confirm a consistent environment:\n"
        + PIP_CHECK_OUTPUT
    )


# -----------------------------------------------------------------------------
# Stage 8: Capture exact installed versions
# -----------------------------------------------------------------------------

freeze_result = run_command(
    [
        str(ENV_PYTHON),
        "-m",
        "pip",
        "freeze",
        "--all",
    ],
    stage_name="Capture exact installed package versions",
    timeout_seconds=300,
    print_output=False,
)

PIP_FREEZE_PATH.write_text(
    freeze_result.stdout,
    encoding="utf-8",
)


# -----------------------------------------------------------------------------
# Stage 9: Inspect key package versions and GPU support
# -----------------------------------------------------------------------------

inspection_code = r"""
import importlib
import importlib.metadata
import json
import platform
import sys

distribution_names = [
    "openwakeword",
    "piper-sample-generator",
    "torch",
    "torchaudio",
    "tensorflow",
    "tensorflow-probability",
    "numpy",
    "scipy",
    "librosa",
    "soundfile",
    "audiomentations",
    "onnx",
    "onnxruntime",
    "scikit-learn",
    "pandas",
]

versions = {}

for name in distribution_names:
    try:
        versions[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        versions[name] = None

torch_information = None
tensorflow_information = None

try:
    import torch

    torch_information = {
        "version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
        "cudnn_available": torch.backends.cudnn.is_available(),
        "gpu_count": torch.cuda.device_count(),
        "gpu_names": [
            torch.cuda.get_device_name(index)
            for index in range(torch.cuda.device_count())
        ],
    }
except Exception as exc:
    torch_information = {
        "error": f"{type(exc).__name__}: {exc}"
    }

try:
    import tensorflow as tf

    tensorflow_information = {
        "version": tf.__version__,
        "physical_gpus": [
            device.name
            for device in tf.config.list_physical_devices("GPU")
        ],
    }
except Exception as exc:
    tensorflow_information = {
        "error": f"{type(exc).__name__}: {exc}"
    }

print(json.dumps({
    "python": sys.version,
    "executable": sys.executable,
    "platform": platform.platform(),
    "versions": versions,
    "torch": torch_information,
    "tensorflow": tensorflow_information,
}, sort_keys=True))
"""

inspection_result = run_command(
    [
        str(ENV_PYTHON),
        "-c",
        inspection_code,
    ],
    stage_name="Inspect installed ML environment",
    timeout_seconds=600,
    print_output=False,
)

try:
    INSTALLATION_INFORMATION = json.loads(
        inspection_result.stdout.strip().splitlines()[-1]
    )
except Exception as exc:
    raise DependencyStageError(
        "Could not parse package inspection output.\n"
        f"Output:\n{inspection_result.stdout}\n"
        f"Error: {exc}"
    ) from exc


# -----------------------------------------------------------------------------
# Hard validation
# -----------------------------------------------------------------------------

required_distributions = (
    "openwakeword",
    "piper-sample-generator",
    "torch",
    "tensorflow",
    "numpy",
    "onnx",
    "onnxruntime",
)

missing_distributions = [
    name
    for name in required_distributions
    if INSTALLATION_INFORMATION["versions"].get(name) is None
]

if missing_distributions:
    raise DependencyStageError(
        "Required distributions are missing after installation: "
        + ", ".join(missing_distributions)
    )

torch_error = INSTALLATION_INFORMATION["torch"].get("error")
tensorflow_error = INSTALLATION_INFORMATION["tensorflow"].get(
    "error"
)

if torch_error:
    raise DependencyStageError(
        f"PyTorch import failed: {torch_error}"
    )

if tensorflow_error:
    raise DependencyStageError(
        f"TensorFlow import failed: {tensorflow_error}"
    )

if GPU_AVAILABLE and not INSTALLATION_INFORMATION["torch"][
    "cuda_available"
]:
    print(
        "\nWARNING: nvidia-smi detected a GPU, but PyTorch reports "
        "CUDA unavailable. Training can continue on CPU, but will be slow."
    )


# -----------------------------------------------------------------------------
# Save report
# -----------------------------------------------------------------------------

CELL_COMPLETED_AT = dt.datetime.now(dt.timezone.utc)

DEPENDENCY_REPORT = {
    "stage": "Cell 5 — Dependency installation",
    "passed": True,
    "started_at_utc": CELL_STARTED_AT.isoformat(),
    "completed_at_utc": CELL_COMPLETED_AT.isoformat(),
    "environment_python": str(ENV_PYTHON),
    "gpu_detection": GPU_INFORMATION,
    "installation_information": INSTALLATION_INFORMATION,
    "pip_check": PIP_CHECK_OUTPUT,
    "paths": {
        "openwakeword_repo": str(OPENWAKEWORD_REPO),
        "piper_generator_repo": str(PIPER_GENERATOR_REPO),
        "pip_freeze": str(PIP_FREEZE_PATH),
        "report": str(DEPENDENCY_REPORT_PATH),
        "log": str(CELL_LOG_PATH),
    },
}

DEPENDENCY_REPORT_PATH.write_text(
    json.dumps(
        DEPENDENCY_REPORT,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

append_log(
    "\nFINAL DEPENDENCY REPORT\n"
    + json.dumps(
        DEPENDENCY_REPORT,
        indent=2,
        ensure_ascii=False,
    )
)


# -----------------------------------------------------------------------------
# Completion report
# -----------------------------------------------------------------------------

print("\n" + "=" * 88)
print("CELL 5 COMPLETED SUCCESSFULLY")
print("=" * 88)

for package_name, package_version in sorted(
    INSTALLATION_INFORMATION["versions"].items()
):
    print(f"{package_name:<28}: {package_version}")

print("\nPyTorch")
for key, value in INSTALLATION_INFORMATION["torch"].items():
    print(f"  {key:<18}: {value}")

print("\nTensorFlow")
for key, value in INSTALLATION_INFORMATION[
    "tensorflow"
].items():
    print(f"  {key:<18}: {value}")

print(f"\npip check     : {PIP_CHECK_OUTPUT}")
print(f"pip freeze    : {PIP_FREEZE_PATH}")
print(f"Report        : {DEPENDENCY_REPORT_PATH}")
print(f"Full log      : {CELL_LOG_PATH}")

print(
    "\nNext: Cell 6 validates imports, discovers the exact upstream "
    "training and generation APIs, and tests basic audio processing."
)
print("=" * 88)


STAGE: Validate isolated Python before installation
COMMAND: /content/pingo-env/bin/python -c 'import json, sys; print(json.dumps({'"'"'version'"'"': sys.version, '"'"'executable'"'"': sys.executable, '"'"'version_info'"'"': list(sys.version_info[:3])}))'
CWD: /content
CELL 5 — DEPENDENCY INSTALLATION
Python          : 3.11.15 (main, Jun  2 2026, 22:26:03) [Clang 22.1.3 ]
Environment     : /content/pingo-env
NVIDIA detected : False

STAGE: Install Python build and packaging tools
COMMAND: /content/pingo-env/bin/python -m pip install --upgrade --prefer-binary pip setuptools wheel packaging build cython
CWD: /content
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 52.5 MB/s  0:00:00



No NVIDIA GPU detected. Installing CPU-only PyTorch to avoid unnecessary CUDA packages.

STAGE: Install CPU-only PyTorch and TorchAudio
COMMAND: /content/pingo-env/bin/python -m pip install --upgrade --prefer-binary --index-url https://download.pytorch.org/whl/cpu torch torchaudio
CWD: /content
Loo

In [ ]:
# =============================================================================
# Cell 6 — Import validation and upstream API discovery
# =============================================================================
#
# This cell runs all validation inside /content/pingo-env.
#
# It:
#   1. Imports OpenWakeWord and Piper Sample Generator.
#   2. Detects available public functions/classes without assuming obsolete APIs.
#   3. Verifies that the Piper module CLI works.
#   4. Tests NumPy, SoundFile, Librosa, Torch, TensorFlow, and ONNX Runtime.
#   5. Generates and processes a temporary WAV.
#   6. Saves a current API manifest for later cells.
#
# =============================================================================

from __future__ import annotations

import datetime as dt
import json
import os
import shlex
import subprocess
import textwrap
from pathlib import Path
from typing import Dict, Optional, Sequence


# -----------------------------------------------------------------------------
# Paths — canonical repository resolution
# -----------------------------------------------------------------------------

try:
    CONFIG
except NameError:
    class _FallbackConfig:
        TRAINING_ROOT = "/content/pingo_training"
        LOGS_DIR = "/content/pingo_training/logs"
        CONFIG_DIR = "/content/pingo_training/config"
        CACHE_DIR = "/content/pingo_training/cache"
        REPOSITORIES_DIR = "/content/pingo_training/repositories"
        ENV_DIR = "/content/pingo-env"

    CONFIG = _FallbackConfig()


TRAINING_ROOT = Path(CONFIG.TRAINING_ROOT)
LOGS_DIR = Path(CONFIG.LOGS_DIR)
CONFIG_DIR = Path(CONFIG.CONFIG_DIR)
CACHE_DIR = Path(CONFIG.CACHE_DIR)
ENV_DIR = Path(CONFIG.ENV_DIR)

ENV_PYTHON = ENV_DIR / "bin" / "python"


# Prefer the canonical repository directory used by Cell 4.
repository_root_candidates = [
    Path(
        getattr(
            CONFIG,
            "REPOSITORIES_DIR",
            TRAINING_ROOT / "repositories",
        )
    ),
    Path(
        getattr(
            CONFIG,
            "REPOS_DIR",
            TRAINING_ROOT / "repos",
        )
    ),
    TRAINING_ROOT / "repositories",
    TRAINING_ROOT / "repos",
]


def is_valid_repository_root(root: Path) -> bool:
    return (
        (root / "openWakeWord" / ".git").exists()
        and (
            root
            / "piper-sample-generator"
            / ".git"
        ).exists()
    )


REPOS_DIR = next(
    (
        root
        for root in repository_root_candidates
        if is_valid_repository_root(root)
    ),
    TRAINING_ROOT / "repositories",
)

OPENWAKEWORD_REPO = REPOS_DIR / "openWakeWord"
PIPER_GENERATOR_REPO = (
    REPOS_DIR / "piper-sample-generator"
)


CELL_LOG_PATH = LOGS_DIR / "cell_06_api_validation.log"
API_MANIFEST_PATH = CONFIG_DIR / "upstream_api_manifest.json"
SMOKE_TEST_DIR = CACHE_DIR / "cell_06_smoke_test"

VALIDATION_SCRIPT_PATH = (
    SMOKE_TEST_DIR / "validate_upstream_apis.py"
)

VALIDATION_OUTPUT_PATH = (
    SMOKE_TEST_DIR / "validation_output.json"
)


print("Resolved Cell 6 paths:")
print(f"  Repository root : {REPOS_DIR}")
print(f"  OpenWakeWord    : {OPENWAKEWORD_REPO}")
print(f"  Piper generator : {PIPER_GENERATOR_REPO}")
print(f"  Python          : {ENV_PYTHON}")

# Prevent Colab's notebook-only Matplotlib backend from being inherited.
os.environ["MPLBACKEND"] = "Agg"


# -----------------------------------------------------------------------------
# Exceptions and logging
# -----------------------------------------------------------------------------

class APIValidationError(RuntimeError):
    """Raised when an import or runtime smoke test fails."""


CELL_STARTED_AT = dt.datetime.now(dt.timezone.utc)

CELL_LOG_PATH.write_text(
    (
        "PINGO API VALIDATION LOG\n"
        f"Started UTC: {CELL_STARTED_AT.isoformat()}\n"
        f"Python: {ENV_PYTHON}\n"
        + "=" * 88
        + "\n"
    ),
    encoding="utf-8",
)


def append_log(message: str) -> None:
    """Append a message to the Cell 6 log file."""

    with CELL_LOG_PATH.open(
        "a",
        encoding="utf-8",
    ) as file_handle:
        file_handle.write(
            message.rstrip() + "\n"
        )


def format_command(
    command: Sequence[str],
) -> str:
    """Return a shell-readable command string."""

    return " ".join(
        shlex.quote(str(item))
        for item in command
    )


def tail_text(
    text: str,
    max_lines: int = 100,
) -> str:
    """Return only the final lines of long output."""

    lines = text.splitlines()

    if len(lines) <= max_lines:
        return text

    return "\n".join(
        lines[-max_lines:]
    )


# -----------------------------------------------------------------------------
# Subprocess helper
# -----------------------------------------------------------------------------

def run_command(
    command: Sequence[str],
    *,
    stage_name: str,
    timeout_seconds: int = 900,
    cwd: Optional[Path] = None,
    print_output: bool = True,
    environment_overrides: Optional[Dict[str, str]] = None,
) -> subprocess.CompletedProcess[str]:
    """
    Run a command using a controlled subprocess environment.

    environment_overrides can be used to pass command-specific environment
    variables such as PINGO_VALIDATION_OUTPUT.
    """

    command = [
        str(item)
        for item in command
    ]

    readable = format_command(command)

    environment = os.environ.copy()

    environment.update(
        {
            "PYTHONUNBUFFERED": "1",
            "TF_CPP_MIN_LOG_LEVEL": "2",
            "TOKENIZERS_PARALLELISM": "false",
            "MPLBACKEND": "Agg",
        }
    )

    # -------------------------------------------------------------------------
    # Make repository-local packages importable.
    #
    # Piper Sample Generator includes piper_train as a source directory inside
    # the repository. Adding the repository root to PYTHONPATH allows:
    #
    #     from piper_train.vits import commons
    #
    # to work inside isolated subprocesses.
    # -------------------------------------------------------------------------

    repository_paths = []

    if PIPER_GENERATOR_REPO.exists():
        repository_paths.append(
            str(PIPER_GENERATOR_REPO.resolve())
        )

    if OPENWAKEWORD_REPO.exists():
        repository_paths.append(
            str(OPENWAKEWORD_REPO.resolve())
        )

    existing_pythonpath = environment.get(
        "PYTHONPATH",
        "",
    ).strip()

    if existing_pythonpath:
        repository_paths.append(existing_pythonpath)

    environment["PYTHONPATH"] = os.pathsep.join(
        repository_paths
    )

    # Critical fix:
    # Apply command-specific environment variables before subprocess.run().
    if environment_overrides:
        environment.update(
            {
                str(key): str(value)
                for key, value
                in environment_overrides.items()
            }
        )

    actual_cwd = (
        str(cwd.resolve())
        if cwd is not None
        else str(SMOKE_TEST_DIR.resolve())
    )

    header = (
        "\n"
        + "=" * 88
        + "\n"
        + f"STAGE: {stage_name}\n"
        + f"COMMAND: {readable}\n"
        + f"CWD: {actual_cwd}\n"
        + "=" * 88
    )

    print(header)
    append_log(header)

    try:
        result = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=False,
            timeout=timeout_seconds,
            cwd=actual_cwd,
            env=environment,
        )
    except subprocess.TimeoutExpired as exc:
        raise APIValidationError(
            textwrap.dedent(
                f"""
                API VALIDATION TIMED OUT

                Stage:
                  {stage_name}

                Command:
                  {readable}

                Timeout:
                  {timeout_seconds} seconds

                Full log:
                  {CELL_LOG_PATH}
                """
            ).strip()
        ) from exc
    except Exception as exc:
        raise APIValidationError(
            textwrap.dedent(
                f"""
                API VALIDATION COMMAND FAILED TO START

                Stage:
                  {stage_name}

                Command:
                  {readable}

                Error:
                  {type(exc).__name__}: {exc}

                Full log:
                  {CELL_LOG_PATH}
                """
            ).strip()
        ) from exc

    stdout = result.stdout or ""
    stderr = result.stderr or ""

    append_log(
        "\n[STDOUT]\n"
        + stdout
    )

    append_log(
        "\n[STDERR]\n"
        + stderr
    )

    append_log(
        f"\n[RETURN CODE]\n{result.returncode}"
    )

    if print_output and stdout.strip():
        print(
            tail_text(stdout)
        )

    if print_output and stderr.strip():
        print("[stderr]")
        print(
            tail_text(stderr)
        )

    if result.returncode != 0:
        raise APIValidationError(
            textwrap.dedent(
                f"""
                API VALIDATION FAILED

                Stage:
                  {stage_name}

                Command:
                  {readable}

                Return code:
                  {result.returncode}

                Stdout:
                {tail_text(stdout)}

                Stderr:
                {tail_text(stderr)}

                Full log:
                  {CELL_LOG_PATH}
                """
            ).strip()
        )

    return result


# -----------------------------------------------------------------------------
# Basic environment validation
# -----------------------------------------------------------------------------

if not ENV_PYTHON.exists():
    raise APIValidationError(
        "Missing isolated Python executable:\n"
        f"{ENV_PYTHON}\n\n"
        "Run the dependency installation cell before Cell 6."
    )

if not ENV_PYTHON.is_file():
    raise APIValidationError(
        "The isolated Python path is not a file:\n"
        f"{ENV_PYTHON}"
    )


# Remove stale output so this run cannot accidentally use an old manifest.
if VALIDATION_OUTPUT_PATH.exists():
    VALIDATION_OUTPUT_PATH.unlink()


# -----------------------------------------------------------------------------
# Create the isolated-environment validation program
# -----------------------------------------------------------------------------

validation_script = r'''
from __future__ import annotations

import importlib
import importlib.metadata
import inspect
import json
import math
import os
import pkgutil
import sys
from pathlib import Path


# -------------------------------------------------------------------------
# Safe environment configuration
# -------------------------------------------------------------------------

os.environ["MPLBACKEND"] = "Agg"
os.environ.setdefault(
    "TF_CPP_MIN_LOG_LEVEL",
    "2",
)


output_value = os.environ.get(
    "PINGO_VALIDATION_OUTPUT"
)

if not output_value:
    raise RuntimeError(
        "PINGO_VALIDATION_OUTPUT was not passed "
        "to the validation subprocess."
    )


OUTPUT_PATH = Path(output_value)
WORK_DIR = OUTPUT_PATH.parent

WORK_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# -------------------------------------------------------------------------
# Base imports
# -------------------------------------------------------------------------

import numpy as np
import soundfile as sf


# -------------------------------------------------------------------------
# Helper functions
# -------------------------------------------------------------------------

def public_members(module):
    result = {}

    for name in sorted(dir(module)):
        if name.startswith("_"):
            continue

        try:
            value = getattr(
                module,
                name,
            )
        except Exception as exc:
            result[name] = {
                "type": "unavailable",
                "error": (
                    f"{type(exc).__name__}: {exc}"
                ),
            }
            continue

        item = {
            "type": type(value).__name__,
            "module": getattr(
                value,
                "__module__",
                None,
            ),
        }

        if (
            inspect.isfunction(value)
            or inspect.isclass(value)
        ):
            try:
                item["signature"] = str(
                    inspect.signature(value)
                )
            except Exception:
                item["signature"] = None

        result[name] = item

    return result


def package_submodules(package):
    package_path = getattr(
        package,
        "__path__",
        None,
    )

    if package_path is None:
        return []

    return sorted(
        module.name
        for module in pkgutil.walk_packages(
            package_path,
            prefix=package.__name__ + ".",
        )
    )


# -------------------------------------------------------------------------
# Report structure
# -------------------------------------------------------------------------

report = {
    "python": {
        "version": sys.version,
        "executable": sys.executable,
        "sys_path": sys.path,
    },
    "packages": {},
    "audio_smoke_test": {},
}


# -------------------------------------------------------------------------
# OpenWakeWord validation
# -------------------------------------------------------------------------

import openwakeword


report["packages"]["openwakeword"] = {
    "distribution_version": (
        importlib.metadata.version(
            "openwakeword"
        )
    ),
    "module_file": getattr(
        openwakeword,
        "__file__",
        None,
    ),
    "public_members": public_members(
        openwakeword
    ),
    "submodules": package_submodules(
        openwakeword
    ),
}


for module_name in (
    "openwakeword.model",
    "openwakeword.utils",
    "openwakeword.train",
    "openwakeword.data",
):
    try:
        module = importlib.import_module(
            module_name
        )

        report["packages"][
            "openwakeword"
        ].setdefault(
            "inspected_modules",
            {},
        )[module_name] = {
            "module_file": getattr(
                module,
                "__file__",
                None,
            ),
            "public_members": public_members(
                module
            ),
        }

    except Exception as exc:
        report["packages"][
            "openwakeword"
        ].setdefault(
            "unavailable_modules",
            {},
        )[module_name] = (
            f"{type(exc).__name__}: {exc}"
        )


# -------------------------------------------------------------------------
# Piper Sample Generator validation
# -------------------------------------------------------------------------

import piper_sample_generator


report["packages"]["piper_sample_generator"] = {
    "distribution_version": (
        importlib.metadata.version(
            "piper-sample-generator"
        )
    ),
    "module_file": getattr(
        piper_sample_generator,
        "__file__",
        None,
    ),
    "public_members": public_members(
        piper_sample_generator
    ),
    "submodules": package_submodules(
        piper_sample_generator
    ),
}


for module_name in (
    "piper_sample_generator.__main__",
    "piper_sample_generator.augment",
):
    try:
        module = importlib.import_module(
            module_name
        )

        report["packages"][
            "piper_sample_generator"
        ].setdefault(
            "inspected_modules",
            {},
        )[module_name] = {
            "module_file": getattr(
                module,
                "__file__",
                None,
            ),
            "public_members": public_members(
                module
            ),
        }

    except SystemExit as exc:
        report["packages"][
            "piper_sample_generator"
        ].setdefault(
            "special_import_results",
            {},
        )[module_name] = (
            f"SystemExit: {exc}"
        )

    except Exception as exc:
        report["packages"][
            "piper_sample_generator"
        ].setdefault(
            "unavailable_modules",
            {},
        )[module_name] = (
            f"{type(exc).__name__}: {exc}"
        )


# -------------------------------------------------------------------------
# Machine-learning package validation
# -------------------------------------------------------------------------

import torch
import torchaudio
import tensorflow as tf
import onnx
import onnxruntime
import librosa
import scipy
import sklearn


report["packages"]["torch"] = {
    "version": torch.__version__,
    "cuda_available": (
        torch.cuda.is_available()
    ),
    "cuda_version": torch.version.cuda,
    "gpu_count": (
        torch.cuda.device_count()
    ),
    "gpu_names": [
        torch.cuda.get_device_name(index)
        for index in range(
            torch.cuda.device_count()
        )
    ],
}


report["packages"]["torchaudio"] = {
    "version": torchaudio.__version__,
}


report["packages"]["tensorflow"] = {
    "version": tf.__version__,
    "gpus": [
        device.name
        for device in (
            tf.config.list_physical_devices(
                "GPU"
            )
        )
    ],
}


report["packages"]["onnx"] = {
    "version": onnx.__version__,
}


report["packages"]["onnxruntime"] = {
    "version": onnxruntime.__version__,
    "providers": (
        onnxruntime.get_available_providers()
    ),
}


report["packages"]["librosa"] = {
    "version": librosa.__version__,
}


report["packages"]["numpy"] = {
    "version": np.__version__,
}


report["packages"]["scipy"] = {
    "version": scipy.__version__,
}


report["packages"]["sklearn"] = {
    "version": sklearn.__version__,
}


# -------------------------------------------------------------------------
# Audio smoke test
# -------------------------------------------------------------------------

sample_rate = 16000
duration_seconds = 1.0

sample_count = int(
    sample_rate
    * duration_seconds
)


time_axis = (
    np.arange(
        sample_count,
        dtype=np.float32,
    )
    / sample_rate
)


signal = (
    0.1
    * np.sin(
        2.0
        * math.pi
        * 440.0
        * time_axis
    )
).astype(
    np.float32
)


wav_path = (
    WORK_DIR
    / "cell_06_test_16khz.wav"
)


sf.write(
    wav_path,
    signal,
    sample_rate,
    subtype="PCM_16",
)


loaded_audio, loaded_rate = sf.read(
    wav_path,
    dtype="float32",
    always_2d=False,
)


if loaded_rate != sample_rate:
    raise RuntimeError(
        f"Expected {sample_rate} Hz, "
        f"received {loaded_rate}"
    )


if loaded_audio.ndim != 1:
    raise RuntimeError(
        "Expected mono audio, "
        f"shape={loaded_audio.shape}"
    )


if len(loaded_audio) != sample_count:
    raise RuntimeError(
        f"Expected {sample_count} samples, "
        f"received {len(loaded_audio)}"
    )


mel = librosa.feature.melspectrogram(
    y=loaded_audio,
    sr=loaded_rate,
    n_fft=512,
    hop_length=160,
    win_length=400,
    n_mels=40,
    power=2.0,
)


if (
    mel.ndim != 2
    or mel.shape[0] != 40
):
    raise RuntimeError(
        f"Unexpected mel shape: {mel.shape}"
    )


torch_tensor = torch.from_numpy(
    loaded_audio.copy()
)


if torch_tensor.numel() != sample_count:
    raise RuntimeError(
        "Torch tensor conversion produced "
        "an incorrect number of samples."
    )


tensorflow_tensor = tf.convert_to_tensor(
    loaded_audio
)


tensorflow_size = int(
    tf.size(
        tensorflow_tensor
    ).numpy()
)


if tensorflow_size != sample_count:
    raise RuntimeError(
        "TensorFlow tensor conversion produced "
        "an incorrect number of samples."
    )


report["audio_smoke_test"] = {
    "passed": True,
    "wav_path": str(wav_path),
    "sample_rate": int(loaded_rate),
    "sample_count": int(
        len(loaded_audio)
    ),
    "duration_seconds": float(
        len(loaded_audio)
        / loaded_rate
    ),
    "channels": 1,
    "dtype": str(
        loaded_audio.dtype
    ),
    "minimum": float(
        np.min(loaded_audio)
    ),
    "maximum": float(
        np.max(loaded_audio)
    ),
    "rms": float(
        np.sqrt(
            np.mean(
                np.square(
                    loaded_audio
                )
            )
        )
    ),
    "mel_shape": [
        int(value)
        for value in mel.shape
    ],
    "torch_tensor_shape": [
        int(value)
        for value in torch_tensor.shape
    ],
    "tensorflow_tensor_shape": [
        int(value)
        for value in tensorflow_tensor.shape
    ],
}


# -------------------------------------------------------------------------
# Write validation result
# -------------------------------------------------------------------------

OUTPUT_PATH.write_text(
    json.dumps(
        report,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


print(
    json.dumps(
        {
            "passed": True,
            "output": str(
                OUTPUT_PATH
            ),
            "openwakeword": (
                report["packages"][
                    "openwakeword"
                ][
                    "distribution_version"
                ]
            ),
            "piper_sample_generator": (
                report["packages"][
                    "piper_sample_generator"
                ][
                    "distribution_version"
                ]
            ),
            "audio_smoke_test": (
                report[
                    "audio_smoke_test"
                ]
            ),
        },
        sort_keys=True,
    )
)
'''


VALIDATION_SCRIPT_PATH.write_text(
    validation_script,
    encoding="utf-8",
)


# -----------------------------------------------------------------------------
# Run isolated API and audio validation
# -----------------------------------------------------------------------------

validation_result = run_command(
    [
        str(ENV_PYTHON),
        str(VALIDATION_SCRIPT_PATH),
    ],
    stage_name=(
        "Run isolated Python API and audio validation"
    ),
    timeout_seconds=1200,
    print_output=True,
    cwd=SMOKE_TEST_DIR,
    environment_overrides={
        "PINGO_VALIDATION_OUTPUT": str(
            VALIDATION_OUTPUT_PATH
        ),
    },
)


# -----------------------------------------------------------------------------
# Read validation output
# -----------------------------------------------------------------------------

if not VALIDATION_OUTPUT_PATH.exists():
    raise APIValidationError(
        "Validation returned success but did not create:\n"
        f"{VALIDATION_OUTPUT_PATH}"
    )


try:
    API_MANIFEST = json.loads(
        VALIDATION_OUTPUT_PATH.read_text(
            encoding="utf-8"
        )
    )
except json.JSONDecodeError as exc:
    raise APIValidationError(
        "Validation output is not valid JSON:\n"
        f"{VALIDATION_OUTPUT_PATH}\n\n"
        f"Error: {exc}"
    ) from exc


if not API_MANIFEST.get(
    "audio_smoke_test",
    {},
).get(
    "passed",
    False,
):
    raise APIValidationError(
        "The validation script completed, but the "
        "audio smoke test did not pass."
    )


# -----------------------------------------------------------------------------
# Piper CLI validation
# -----------------------------------------------------------------------------

piper_help_result = run_command(
    [
        str(ENV_PYTHON),
        "-m",
        "piper_sample_generator",
        "--help",
    ],
    stage_name=(
        "Validate Piper Sample Generator module CLI"
    ),
    timeout_seconds=300,
    print_output=False,
    cwd=PIPER_GENERATOR_REPO,
    environment_overrides={
        "PYTHONPATH": os.pathsep.join(
            [
                str(PIPER_GENERATOR_REPO.resolve()),
                os.environ.get("PYTHONPATH", ""),
            ]
        ).strip(os.pathsep),
    },
)


PIPER_HELP_TEXT = (
    (piper_help_result.stdout or "")
    + "\n"
    + (piper_help_result.stderr or "")
).strip()


if not PIPER_HELP_TEXT:
    raise APIValidationError(
        "Piper Sample Generator CLI returned "
        "no help text."
    )


API_MANIFEST[
    "piper_cli_help"
] = PIPER_HELP_TEXT


# -----------------------------------------------------------------------------
# OpenWakeWord Model API validation
# -----------------------------------------------------------------------------

openwakeword_model_result = run_command(
    [
        str(ENV_PYTHON),
        "-c",
        (
            "from openwakeword.model import Model; "
            "import inspect; "
            "print(inspect.signature(Model))"
        ),
    ],
    stage_name=(
        "Validate openwakeword.model.Model import"
    ),
    timeout_seconds=300,
    print_output=False,
)


OPENWAKEWORD_MODEL_SIGNATURE = (
    openwakeword_model_result.stdout.strip()
)


if not OPENWAKEWORD_MODEL_SIGNATURE:
    raise APIValidationError(
        "openwakeword.model.Model imported, "
        "but no constructor signature was returned."
    )


API_MANIFEST[
    "openwakeword_model_signature"
] = OPENWAKEWORD_MODEL_SIGNATURE


# -----------------------------------------------------------------------------
# Save final API manifest
# -----------------------------------------------------------------------------

CELL_COMPLETED_AT = dt.datetime.now(
    dt.timezone.utc
)


API_MANIFEST["cell"] = {
    "stage": "Cell 6 — API validation",
    "started_at_utc": (
        CELL_STARTED_AT.isoformat()
    ),
    "completed_at_utc": (
        CELL_COMPLETED_AT.isoformat()
    ),
    "passed": True,
}


API_MANIFEST["paths"] = {
    "validation_script": str(
        VALIDATION_SCRIPT_PATH
    ),
    "validation_output": str(
        VALIDATION_OUTPUT_PATH
    ),
    "final_manifest": str(
        API_MANIFEST_PATH
    ),
    "log": str(
        CELL_LOG_PATH
    ),
}


API_MANIFEST_PATH.write_text(
    json.dumps(
        API_MANIFEST,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


append_log(
    "\nFINAL API MANIFEST\n"
    + json.dumps(
        API_MANIFEST,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )
)


# -----------------------------------------------------------------------------
# Completion summary
# -----------------------------------------------------------------------------

print("\n" + "=" * 88)
print("CELL 6 COMPLETED SUCCESSFULLY")
print("=" * 88)


print(
    "OpenWakeWord version        : "
    + str(
        API_MANIFEST[
            "packages"
        ][
            "openwakeword"
        ][
            "distribution_version"
        ]
    )
)


print(
    "Piper generator version     : "
    + str(
        API_MANIFEST[
            "packages"
        ][
            "piper_sample_generator"
        ][
            "distribution_version"
        ]
    )
)


print(
    "PyTorch CUDA available      : "
    + str(
        API_MANIFEST[
            "packages"
        ][
            "torch"
        ][
            "cuda_available"
        ]
    )
)


print(
    "TensorFlow GPU devices      : "
    + str(
        API_MANIFEST[
            "packages"
        ][
            "tensorflow"
        ][
            "gpus"
        ]
    )
)


print(
    "ONNX Runtime providers      : "
    + str(
        API_MANIFEST[
            "packages"
        ][
            "onnxruntime"
        ][
            "providers"
        ]
    )
)


print(
    "Audio smoke test            : "
    + str(
        API_MANIFEST[
            "audio_smoke_test"
        ][
            "passed"
        ]
    )
)


print(
    "OpenWakeWord Model signature: "
    + API_MANIFEST[
        "openwakeword_model_signature"
    ]
)


print(
    f"API manifest                : "
    f"{API_MANIFEST_PATH}"
)


print(
    f"Full log                    : "
    f"{CELL_LOG_PATH}"
)


print(
    "\nNext: Cell 7 downloads and validates the required "
    "OpenWakeWord assets and Piper generator model."
)

print("=" * 88)

Resolved Cell 6 paths:
  Repository root : /content/pingo_training/repositories
  OpenWakeWord    : /content/pingo_training/repositories/openWakeWord
  Piper generator : /content/pingo_training/repositories/piper-sample-generator
  Python          : /content/pingo-env/bin/python


FileNotFoundError: [Errno 2] No such file or directory: '/content/pingo_training/cache/cell_06_smoke_test/validate_upstream_apis.py'

In [ ]:
# =============================================================================
# Repair Cell — Locate or restore Piper Sample Generator repository
# Run this immediately before Cell 7
# =============================================================================

from pathlib import Path
import os
import shutil
import subprocess


TRAINING_ROOT = Path("/content/pingo_training")
EXPECTED_REPOS_DIR = TRAINING_ROOT / "repos"
EXPECTED_PIPER_REPO = EXPECTED_REPOS_DIR / "piper-sample-generator"

ENV_DIR = Path("/content/pingo-env")
ENV_PYTHON = ENV_DIR / "bin" / "python"

EXPECTED_REPOS_DIR.mkdir(parents=True, exist_ok=True)


def run(command, cwd=None):
    print("\n$", " ".join(str(item) for item in command))

    result = subprocess.run(
        [str(item) for item in command],
        cwd=str(cwd) if cwd else None,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    print(result.stdout)

    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed with return code {result.returncode}:\n"
            + " ".join(str(item) for item in command)
        )

    return result


def valid_piper_repository(path: Path) -> bool:
    required_paths = [
        path,
        path / "piper_sample_generator",
        path / "piper_train",
        path / "piper_train" / "vits",
    ]

    return all(required.exists() for required in required_paths)


# -----------------------------------------------------------------------------
# 1. Search common existing locations
# -----------------------------------------------------------------------------

candidate_paths = [
    TRAINING_ROOT / "repositories" / "piper-sample-generator",
    TRAINING_ROOT / "repos" / "piper-sample-generator",
    TRAINING_ROOT / "piper-sample-generator",
    Path("/content/piper-sample-generator"),
]

# Also search all matching directories under /content.
for candidate in Path("/content").rglob("piper-sample-generator"):
    if candidate.is_dir() and candidate not in candidate_paths:
        candidate_paths.append(candidate)


print("=" * 88)
print("SEARCHING FOR EXISTING PIPER REPOSITORIES")
print("=" * 88)

valid_source = None

for candidate in candidate_paths:
    if not candidate.exists():
        continue

    print(f"\nFound candidate: {candidate}")

    required = {
        "piper_sample_generator": candidate / "piper_sample_generator",
        "piper_train": candidate / "piper_train",
        "piper_train/vits": candidate / "piper_train" / "vits",
    }

    for name, path in required.items():
        print(
            f"  {'[OK]' if path.exists() else '[MISSING]'} "
            f"{name}: {path}"
        )

    if valid_piper_repository(candidate):
        valid_source = candidate
        print(f"[VALID] Complete repository found: {candidate}")
        break


# -----------------------------------------------------------------------------
# 2. Copy an existing valid repository into the expected location
# -----------------------------------------------------------------------------

if valid_source is not None:
    if valid_source.resolve() != EXPECTED_PIPER_REPO.resolve():
        if EXPECTED_PIPER_REPO.exists():
            print(
                "\nRemoving incomplete expected repository:\n"
                f"  {EXPECTED_PIPER_REPO}"
            )
            shutil.rmtree(EXPECTED_PIPER_REPO)

        print(
            "\nCopying complete repository:\n"
            f"  From: {valid_source}\n"
            f"  To:   {EXPECTED_PIPER_REPO}"
        )

        shutil.copytree(
            valid_source,
            EXPECTED_PIPER_REPO,
            symlinks=True,
        )


# -----------------------------------------------------------------------------
# 3. Clone afresh when no complete local repository exists
# -----------------------------------------------------------------------------

else:
    print("\nNo complete existing repository was found.")

    if EXPECTED_PIPER_REPO.exists():
        print(f"Removing incomplete repository: {EXPECTED_PIPER_REPO}")
        shutil.rmtree(EXPECTED_PIPER_REPO)

    run(
        [
            "git",
            "clone",
            "--recursive",
            "https://github.com/rhasspy/piper-sample-generator.git",
            str(EXPECTED_PIPER_REPO),
        ]
    )

    # Ensure all submodules are present, if the selected revision uses them.
    run(
        [
            "git",
            "submodule",
            "update",
            "--init",
            "--recursive",
        ],
        cwd=EXPECTED_PIPER_REPO,
    )


# -----------------------------------------------------------------------------
# 4. Validate repository structure
# -----------------------------------------------------------------------------

required_paths = [
    EXPECTED_PIPER_REPO,
    EXPECTED_PIPER_REPO / "piper_sample_generator",
    EXPECTED_PIPER_REPO / "piper_train",
    EXPECTED_PIPER_REPO / "piper_train" / "vits",
]

missing_paths = [
    path for path in required_paths
    if not path.exists()
]

print("\n" + "=" * 88)
print("FINAL REPOSITORY VALIDATION")
print("=" * 88)

for path in required_paths:
    print(f"{'[OK]' if path.exists() else '[MISSING]'} {path}")


if missing_paths:
    print("\nRepository top-level contents:")

    if EXPECTED_PIPER_REPO.exists():
        for item in sorted(EXPECTED_PIPER_REPO.iterdir()):
            item_type = "DIR " if item.is_dir() else "FILE"
            print(f"  {item_type}: {item.name}")

    raise RuntimeError(
        "\nThe cloned Piper repository does not match the old repository "
        "layout expected by Cell 7.\n\n"
        "Missing:\n"
        + "\n".join(f"  - {path}" for path in missing_paths)
        + "\n\nDo not rerun Cell 7 yet. The notebook revision and Piper "
        "repository revision are incompatible."
    )


# -----------------------------------------------------------------------------
# 5. Install repository into the isolated environment
# -----------------------------------------------------------------------------

if not ENV_PYTHON.exists():
    raise FileNotFoundError(
        f"Environment Python not found: {ENV_PYTHON}\n"
        "Run the environment setup cells first."
    )

run(
    [
        str(ENV_PYTHON),
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "-e",
        str(EXPECTED_PIPER_REPO),
    ]
)


# -----------------------------------------------------------------------------
# 6. Import validation
# -----------------------------------------------------------------------------

validation_code = f"""
import sys
from pathlib import Path

repo = Path({str(EXPECTED_PIPER_REPO)!r})
sys.path.insert(0, str(repo))

import piper_sample_generator
import piper_train
from piper_train.vits import commons

print("piper_sample_generator:", piper_sample_generator.__file__)
print("piper_train:", piper_train.__file__)
print("piper_train.vits.commons:", commons.__file__)
print("PIPER_REPOSITORY_OK")
"""

run(
    [
        str(ENV_PYTHON),
        "-c",
        validation_code,
    ],
    cwd=EXPECTED_PIPER_REPO,
)


print("\n" + "=" * 88)
print("PIPER REPOSITORY REPAIR COMPLETED")
print("=" * 88)
print(f"Repository: {EXPECTED_PIPER_REPO}")
print("You may now rerun Cell 7.")
print("=" * 88)

In [ ]:
# =============================================================================
# Cell 7 — Download and validate OpenWakeWord and Piper assets
# =============================================================================
#
# This cell:
#   1. Downloads OpenWakeWord's required bundled/pretrained model assets.
#   2. Downloads the Piper LibriTTS-R generator model.
#   3. Records URLs, sizes, and SHA-256 hashes.
#   4. Validates ONNX assets where applicable.
#   5. Validates the Piper checkpoint with piper_train import support.
#   6. Generates a small PINGO synthetic-audio smoke-test dataset.
#   7. Converts and validates every generated WAV as 16 kHz mono PCM16.
#
# This is only a smoke test. It does not generate the full 50,000 positives.
# =============================================================================

from __future__ import annotations

import datetime as dt
import hashlib
import json
import os
import shutil
import shlex
import subprocess
import textwrap
import urllib.request
import wave
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

try:
    CONFIG
except NameError:
    class _FallbackConfig:
        TRAINING_ROOT = "/content/pingo_training"
        LOGS_DIR = "/content/pingo_training/logs"
        CONFIG_DIR = "/content/pingo_training/config"
        CACHE_DIR = "/content/pingo_training/cache"
        REPOSITORIES_DIR = "/content/pingo_training/repositories"
        ENV_DIR = "/content/pingo-env"
        WAKE_WORD = "PINGO"
        RANDOM_SEED = 42

    CONFIG = _FallbackConfig()


TRAINING_ROOT = Path(CONFIG.TRAINING_ROOT)
LOGS_DIR = Path(CONFIG.LOGS_DIR)
CONFIG_DIR = Path(CONFIG.CONFIG_DIR)
CACHE_DIR = Path(CONFIG.CACHE_DIR)
MODELS_DIR = Path(
    getattr(CONFIG, "MODELS_DIR", TRAINING_ROOT / "models")
)
REPOS_DIR = Path(
    getattr(CONFIG, "REPOS_DIR", TRAINING_ROOT / "repos")
)
ENV_DIR = Path(CONFIG.ENV_DIR)
ENV_PYTHON = ENV_DIR / "bin" / "python"

WAKE_WORD = str(
    getattr(CONFIG, "WAKE_WORD", "PINGO")
).strip()

OPENWAKEWORD_REPO = REPOS_DIR / "openWakeWord"
PIPER_GENERATOR_REPO = REPOS_DIR / "piper-sample-generator"

PIPER_MODELS_DIR = MODELS_DIR / "piper"
OPENWAKEWORD_MODELS_DIR = MODELS_DIR / "openwakeword"
SMOKE_OUTPUT_DIR = CACHE_DIR / "cell_07_pingo_generation_smoke_test"
CONVERTED_DIR = SMOKE_OUTPUT_DIR / "converted_16khz_mono"

PIPER_GENERATOR_MODEL_PATH = (
    PIPER_MODELS_DIR / "en_US-libritts_r-medium.pt"
)

PIPER_GENERATOR_MODEL_URL = (
    "https://github.com/rhasspy/piper-sample-generator/"
    "releases/download/v2.0.0/"
    "en_US-libritts_r-medium.pt"
)

CELL_LOG_PATH = LOGS_DIR / "cell_07_model_assets.log"
ASSET_MANIFEST_PATH = CONFIG_DIR / "model_asset_manifest.json"

for directory in (
    LOGS_DIR,
    CONFIG_DIR,
    CACHE_DIR,
    MODELS_DIR,
    PIPER_MODELS_DIR,
    OPENWAKEWORD_MODELS_DIR,
    SMOKE_OUTPUT_DIR,
    CONVERTED_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

os.environ["MPLBACKEND"] = "Agg"


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------

class AssetStageError(RuntimeError):
    """Raised when asset download or validation fails."""


CELL_STARTED_AT = dt.datetime.now(dt.timezone.utc)

CELL_LOG_PATH.write_text(
    (
        "PINGO MODEL ASSET SETUP LOG\n"
        f"Started UTC: {CELL_STARTED_AT.isoformat()}\n"
        f"Wake word: {WAKE_WORD}\n"
        f"Environment Python: {ENV_PYTHON}\n"
        f"Piper repository: {PIPER_GENERATOR_REPO}\n"
        + "=" * 88
        + "\n"
    ),
    encoding="utf-8",
)


def append_log(message: str) -> None:
    with CELL_LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(message.rstrip() + "\n")


def format_command(command: Sequence[str]) -> str:
    return " ".join(shlex.quote(str(item)) for item in command)


def tail_text(text: str, max_lines: int = 120) -> str:
    lines = text.splitlines()

    if len(lines) <= max_lines:
        return text

    return "\n".join(
        [
            f"... showing final {max_lines} lines ...",
            *lines[-max_lines:],
        ]
    )


def build_pythonpath(*paths: Path) -> str:
    """Build a deterministic PYTHONPATH without losing the existing value."""

    entries: List[str] = []

    for path in paths:
        if path.exists():
            resolved = str(path.resolve())
            if resolved not in entries:
                entries.append(resolved)

    existing = os.environ.get("PYTHONPATH", "").strip()

    if existing:
        for entry in existing.split(os.pathsep):
            entry = entry.strip()
            if entry and entry not in entries:
                entries.append(entry)

    return os.pathsep.join(entries)


def run_command(
    command: Sequence[str],
    *,
    stage_name: str,
    timeout_seconds: int = 1800,
    cwd: Optional[Path] = None,
    extra_env: Optional[Dict[str, str]] = None,
    print_output: bool = True,
) -> subprocess.CompletedProcess[str]:
    command = [str(item) for item in command]
    readable = format_command(command)

    environment = os.environ.copy()
    environment.update(
        {
            "PYTHONUNBUFFERED": "1",
            "TF_CPP_MIN_LOG_LEVEL": "2",
            "TOKENIZERS_PARALLELISM": "false",
            "MPLBACKEND": "Agg",
        }
    )

    if extra_env:
        environment.update(
            {
                str(key): str(value)
                for key, value in extra_env.items()
            }
        )

    actual_cwd = (
        str(cwd.resolve())
        if cwd is not None
        else str(TRAINING_ROOT.resolve())
    )

    header = (
        "\n"
        + "=" * 88
        + "\n"
        + f"STAGE: {stage_name}\n"
        + f"COMMAND: {readable}\n"
        + f"CWD: {actual_cwd}\n"
        + "=" * 88
    )

    print(header)
    append_log(header)

    try:
        result = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=False,
            timeout=timeout_seconds,
            cwd=actual_cwd,
            env=environment,
        )
    except subprocess.TimeoutExpired as exc:
        raise AssetStageError(
            textwrap.dedent(
                f"""
                ASSET STAGE TIMED OUT

                Stage:
                  {stage_name}

                Command:
                  {readable}

                Timeout:
                  {timeout_seconds} seconds

                Full log:
                  {CELL_LOG_PATH}
                """
            ).strip()
        ) from exc
    except Exception as exc:
        raise AssetStageError(
            textwrap.dedent(
                f"""
                ASSET STAGE COULD NOT START

                Stage:
                  {stage_name}

                Command:
                  {readable}

                Error:
                  {type(exc).__name__}: {exc}

                Full log:
                  {CELL_LOG_PATH}
                """
            ).strip()
        ) from exc

    stdout = result.stdout or ""
    stderr = result.stderr or ""

    append_log("\n[STDOUT]\n" + stdout)
    append_log("\n[STDERR]\n" + stderr)
    append_log(f"\n[RETURN CODE]\n{result.returncode}")

    if print_output and stdout.strip():
        print(tail_text(stdout))

    if print_output and stderr.strip():
        print("[stderr]")
        print(tail_text(stderr))

    if result.returncode != 0:
        raise AssetStageError(
            textwrap.dedent(
                f"""
                ASSET STAGE FAILED

                Stage:
                  {stage_name}

                Command:
                  {readable}

                Return code:
                  {result.returncode}

                Stdout:
                {tail_text(stdout)}

                Stderr:
                {tail_text(stderr)}

                Full log:
                  {CELL_LOG_PATH}
                """
            ).strip()
        )

    return result


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            block = handle.read(1024 * 1024)
            if not block:
                break
            digest.update(block)

    return digest.hexdigest()


def download_with_resume_safety(
    *,
    url: str,
    destination: Path,
    minimum_bytes: int,
    timeout_seconds: int = 300,
) -> Dict[str, Any]:
    """Download an asset atomically and reuse an existing valid-size file."""

    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = destination.with_suffix(destination.suffix + ".part")

    if destination.exists():
        existing_size = destination.stat().st_size

        if existing_size >= minimum_bytes:
            print(f"[REUSE] {destination} ({existing_size:,} bytes)")
            return {
                "url": url,
                "path": str(destination),
                "size_bytes": existing_size,
                "sha256": sha256_file(destination),
                "reused": True,
            }

        destination.unlink()

    if temporary_path.exists():
        temporary_path.unlink()

    print(f"[DOWNLOAD] {url}")
    print(f"[TARGET]   {destination}")

    request = urllib.request.Request(
        url,
        headers={
            "User-Agent": "Pingo-Training-Notebook/1.0",
            "Accept": "*/*",
        },
    )

    try:
        with urllib.request.urlopen(
            request,
            timeout=timeout_seconds,
        ) as response:
            status = int(getattr(response, "status", 200))

            if status != 200:
                raise AssetStageError(
                    f"Unexpected HTTP status {status} for {url}"
                )

            with temporary_path.open("wb") as output:
                while True:
                    block = response.read(1024 * 1024)
                    if not block:
                        break
                    output.write(block)
    except Exception:
        if temporary_path.exists():
            temporary_path.unlink()
        raise

    downloaded_size = temporary_path.stat().st_size

    if downloaded_size < minimum_bytes:
        temporary_path.unlink(missing_ok=True)
        raise AssetStageError(
            f"Downloaded asset is unexpectedly small.\n"
            f"URL: {url}\n"
            f"Size: {downloaded_size:,} bytes\n"
            f"Minimum expected: {minimum_bytes:,} bytes"
        )

    temporary_path.replace(destination)

    return {
        "url": url,
        "path": str(destination),
        "size_bytes": destination.stat().st_size,
        "sha256": sha256_file(destination),
        "reused": False,
    }


def inspect_wav(path: Path) -> Dict[str, Any]:
    try:
        with wave.open(str(path), "rb") as wav_file:
            channels = wav_file.getnchannels()
            sample_width = wav_file.getsampwidth()
            sample_rate = wav_file.getframerate()
            frame_count = wav_file.getnframes()
            duration = frame_count / sample_rate if sample_rate > 0 else 0.0
    except Exception as exc:
        raise AssetStageError(
            f"Invalid WAV file: {path}\n"
            f"Error: {type(exc).__name__}: {exc}"
        ) from exc

    if channels < 1:
        raise AssetStageError(f"WAV has no audio channels: {path}")

    if sample_rate < 8000:
        raise AssetStageError(
            f"Unexpectedly low WAV sample rate: {sample_rate} Hz in {path}"
        )

    if frame_count <= 0 or duration <= 0:
        raise AssetStageError(f"WAV contains no audio frames: {path}")

    return {
        "path": str(path),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
        "channels": channels,
        "sample_width_bytes": sample_width,
        "sample_rate": sample_rate,
        "frame_count": frame_count,
        "duration_seconds": duration,
    }


# -----------------------------------------------------------------------------
# Preconditions
# -----------------------------------------------------------------------------

if not ENV_PYTHON.exists():
    raise AssetStageError(
        f"Missing environment Python: {ENV_PYTHON}\n"
        "Run Cells 3–6 first."
    )

if not WAKE_WORD:
    raise AssetStageError("WAKE_WORD cannot be empty.")

required_piper_paths = [
    PIPER_GENERATOR_REPO,
    PIPER_GENERATOR_REPO / "piper_sample_generator",
    PIPER_GENERATOR_REPO / "piper_train",
    PIPER_GENERATOR_REPO / "piper_train" / "vits",
]

missing_piper_paths = [
    path for path in required_piper_paths if not path.exists()
]

if missing_piper_paths:
    raise AssetStageError(
        "The Piper repository is incomplete.\n"
        "Missing paths:\n"
        + "\n".join(f"  - {path}" for path in missing_piper_paths)
    )

PIPER_PYTHONPATH = build_pythonpath(PIPER_GENERATOR_REPO)


# -----------------------------------------------------------------------------
# Stage 1: Download OpenWakeWord model assets
# -----------------------------------------------------------------------------

openwakeword_download_script = r'''
from __future__ import annotations

import inspect
import json
import os
from pathlib import Path

import openwakeword
import openwakeword.utils as utils

target_dir = Path(os.environ["PINGO_OWW_MODEL_DIR"])
target_dir.mkdir(parents=True, exist_ok=True)

download_function = getattr(utils, "download_models", None)

if download_function is None:
    download_function = getattr(openwakeword, "download_models", None)

if download_function is None:
    raise RuntimeError(
        "No OpenWakeWord download_models function is available."
    )

signature = inspect.signature(download_function)
kwargs = {}

for parameter_name in (
    "target_directory",
    "download_directory",
    "model_dir",
    "output_dir",
):
    if parameter_name in signature.parameters:
        kwargs[parameter_name] = str(target_dir)
        break

result = download_function(**kwargs)
files = []

for root in {
    target_dir,
    Path(openwakeword.__file__).resolve().parent,
}:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in {".onnx", ".tflite"}:
            files.append(str(path.resolve()))

print(
    json.dumps(
        {
            "function_signature": str(signature),
            "kwargs": kwargs,
            "result": repr(result),
            "files": sorted(set(files)),
        },
        sort_keys=True,
    )
)
'''

oww_result = run_command(
    [
        str(ENV_PYTHON),
        "-c",
        openwakeword_download_script,
    ],
    stage_name="Download OpenWakeWord model assets",
    timeout_seconds=1800,
    extra_env={
        "PINGO_OWW_MODEL_DIR": str(OPENWAKEWORD_MODELS_DIR),
    },
    print_output=False,
)

try:
    OPENWAKEWORD_DOWNLOAD_REPORT = json.loads(
        oww_result.stdout.strip().splitlines()[-1]
    )
except Exception as exc:
    raise AssetStageError(
        "Could not parse OpenWakeWord download output.\n"
        f"Output:\n{oww_result.stdout}\n"
        f"Error: {exc}"
    ) from exc

OPENWAKEWORD_ASSET_FILES = [
    Path(path)
    for path in OPENWAKEWORD_DOWNLOAD_REPORT.get("files", [])
    if Path(path).exists()
]

if not OPENWAKEWORD_ASSET_FILES:
    raise AssetStageError(
        "OpenWakeWord download_models completed, but no ONNX or TFLite "
        "assets were discovered."
    )

OPENWAKEWORD_ASSET_REPORTS = [
    {
        "path": str(path),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
        "suffix": path.suffix.lower(),
    }
    for path in OPENWAKEWORD_ASSET_FILES
]


# -----------------------------------------------------------------------------
# Stage 2: Download Piper generator model
# -----------------------------------------------------------------------------

PIPER_MODEL_REPORT = download_with_resume_safety(
    url=PIPER_GENERATOR_MODEL_URL,
    destination=PIPER_GENERATOR_MODEL_PATH,
    minimum_bytes=1_000_000,
    timeout_seconds=600,
)
# -----------------------------------------------------------------------------
# Stage 2B: Install the Piper generator companion JSON configuration
# -----------------------------------------------------------------------------
#
# piper_sample_generator expects:
#
#     <model-path>.json
#
# Therefore:
#
#     en_US-libritts_r-medium.pt
#
# must have:
#
#     en_US-libritts_r-medium.pt.json
#
# beside it.
# -----------------------------------------------------------------------------

PIPER_GENERATOR_CONFIG_PATH = Path(
    str(PIPER_GENERATOR_MODEL_PATH) + ".json"
)

PIPER_REPOSITORY_CONFIG_CANDIDATES = [
    PIPER_GENERATOR_REPO
    / "models"
    / "en_US-libritts_r-medium.pt.json",

    PIPER_GENERATOR_REPO
    / "models"
    / "en-us-libritts-high.pt.json",
]


def validate_piper_generator_config(
    path: Path,
) -> Dict[str, Any]:
    """Validate the Piper generator JSON configuration."""

    if not path.exists():
        raise AssetStageError(
            f"Piper generator configuration does not exist: {path}"
        )

    try:
        config_data = json.loads(
            path.read_text(encoding="utf-8")
        )
    except json.JSONDecodeError as exc:
        raise AssetStageError(
            "Piper generator configuration is invalid JSON.\n"
            f"Path: {path}\n"
            f"Error: {exc}"
        ) from exc

    required_top_level_keys = {
        "audio",
        "espeak",
        "num_speakers",
        "phoneme_id_map",
    }

    missing_keys = sorted(
        required_top_level_keys
        - set(config_data.keys())
    )

    if missing_keys:
        raise AssetStageError(
            "Piper generator configuration is missing required keys.\n"
            f"Path: {path}\n"
            f"Missing: {missing_keys}"
        )

    audio_config = config_data.get("audio", {})
    espeak_config = config_data.get("espeak", {})

    sample_rate = audio_config.get("sample_rate")
    espeak_voice = espeak_config.get("voice")
    num_speakers = config_data.get("num_speakers")
    phoneme_id_map = config_data.get("phoneme_id_map")

    if not isinstance(sample_rate, int) or sample_rate <= 0:
        raise AssetStageError(
            "Piper configuration contains an invalid audio sample rate.\n"
            f"Path: {path}\n"
            f"Value: {sample_rate!r}"
        )

    if not isinstance(espeak_voice, str) or not espeak_voice.strip():
        raise AssetStageError(
            "Piper configuration contains an invalid espeak voice.\n"
            f"Path: {path}\n"
            f"Value: {espeak_voice!r}"
        )

    if not isinstance(num_speakers, int) or num_speakers <= 0:
        raise AssetStageError(
            "Piper configuration contains an invalid speaker count.\n"
            f"Path: {path}\n"
            f"Value: {num_speakers!r}"
        )

    if not isinstance(phoneme_id_map, dict) or not phoneme_id_map:
        raise AssetStageError(
            "Piper configuration contains an invalid phoneme_id_map.\n"
            f"Path: {path}"
        )

    required_phonemes = {
        "^",
        "_",
        "$",
    }

    missing_phonemes = sorted(
        required_phonemes
        - set(phoneme_id_map.keys())
    )

    if missing_phonemes:
        raise AssetStageError(
            "Piper configuration is missing required control phonemes.\n"
            f"Path: {path}\n"
            f"Missing: {missing_phonemes}"
        )

    return {
        "path": str(path),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
        "sample_rate": sample_rate,
        "espeak_voice": espeak_voice,
        "num_speakers": num_speakers,
        "phoneme_count": len(phoneme_id_map),
    }


if PIPER_GENERATOR_CONFIG_PATH.exists():
    print(
        "[REUSE] Piper generator configuration: "
        f"{PIPER_GENERATOR_CONFIG_PATH}"
    )

else:
    repository_config_path = next(
        (
            candidate
            for candidate in PIPER_REPOSITORY_CONFIG_CANDIDATES
            if candidate.exists()
        ),
        None,
    )

    if repository_config_path is None:
        searched_paths = "\n".join(
            f"  - {candidate}"
            for candidate
            in PIPER_REPOSITORY_CONFIG_CANDIDATES
        )

        raise AssetStageError(
            "The Piper generator companion configuration was not found.\n\n"
            "Searched:\n"
            f"{searched_paths}\n\n"
            "Expected destination:\n"
            f"  {PIPER_GENERATOR_CONFIG_PATH}"
        )

    PIPER_GENERATOR_CONFIG_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        repository_config_path,
        PIPER_GENERATOR_CONFIG_PATH,
    )

    print(
        "[COPY] Piper generator configuration\n"
        f"  From: {repository_config_path}\n"
        f"  To:   {PIPER_GENERATOR_CONFIG_PATH}"
    )


PIPER_CONFIG_REPORT = validate_piper_generator_config(
    PIPER_GENERATOR_CONFIG_PATH
)

print(
    "[OK] Piper generator configuration validated\n"
    f"  Path         : {PIPER_CONFIG_REPORT['path']}\n"
    f"  Sample rate  : {PIPER_CONFIG_REPORT['sample_rate']}\n"
    f"  Speakers     : {PIPER_CONFIG_REPORT['num_speakers']}\n"
    f"  Espeak voice : {PIPER_CONFIG_REPORT['espeak_voice']}"
)

# -----------------------------------------------------------------------------
# Stage 3: Validate Piper generator model with PyTorch
# -----------------------------------------------------------------------------

torch_validation_code = r'''
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

path = Path(os.environ["PINGO_PIPER_GENERATOR_MODEL"]).resolve()
piper_repo = Path(os.environ["PINGO_PIPER_REPOSITORY"]).resolve()

if not path.exists():
    raise FileNotFoundError(f"Piper generator model not found: {path}")

if not piper_repo.exists():
    raise FileNotFoundError(
        f"Piper Sample Generator repository not found: {piper_repo}"
    )

piper_train_dir = piper_repo / "piper_train"

if not piper_train_dir.exists():
    raise FileNotFoundError(
        "The Piper repository does not contain the required "
        f"piper_train directory: {piper_train_dir}"
    )

piper_repo_string = str(piper_repo)
if piper_repo_string not in sys.path:
    sys.path.insert(0, piper_repo_string)

try:
    import piper_train
    from piper_train.vits import commons
except Exception as exc:
    raise RuntimeError(
        "The piper_train package could not be imported before loading "
        "the Piper checkpoint.\n"
        f"Repository: {piper_repo}\n"
        f"sys.path: {sys.path}\n"
        f"Error: {type(exc).__name__}: {exc}"
    ) from exc

import torch

load_attempts = []
loaded_type = None
loaded_keys = None
checkpoint_summary = {}

try:
    value = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )
    loaded_type = type(value).__name__

    if isinstance(value, dict):
        loaded_keys = sorted(str(key) for key in value.keys())[:100]
        checkpoint_summary = {
            "dictionary_size": len(value),
            "contains_model": "model" in value,
            "contains_config": "config" in value,
            "contains_state_dict": "state_dict" in value,
        }
except Exception as exc:
    load_attempts.append(
        f"torch.load: {type(exc).__name__}: {exc}"
    )

if loaded_type is None:
    raise RuntimeError(
        "The Piper generator checkpoint could not be loaded.\n"
        + "\n".join(load_attempts)
    )

print(
    json.dumps(
        {
            "path": str(path),
            "repository": str(piper_repo),
            "piper_train_directory": str(piper_train_dir),
            "piper_train_imported": True,
            "loaded_type": loaded_type,
            "loaded_keys": loaded_keys,
            "checkpoint_summary": checkpoint_summary,
            "load_attempts": load_attempts,
            "torch_version": torch.__version__,
        },
        sort_keys=True,
    )
)
'''

torch_validation_result = run_command(
    [
        str(ENV_PYTHON),
        "-c",
        torch_validation_code,
    ],
    stage_name="Validate Piper generator model with PyTorch",
    timeout_seconds=600,
    cwd=PIPER_GENERATOR_REPO,
    extra_env={
        "PINGO_PIPER_GENERATOR_MODEL": str(PIPER_GENERATOR_MODEL_PATH),
        "PINGO_PIPER_REPOSITORY": str(PIPER_GENERATOR_REPO),
        "PYTHONPATH": PIPER_PYTHONPATH,
    },
    print_output=True,
)

try:
    PIPER_TORCH_VALIDATION = json.loads(
        torch_validation_result.stdout.strip().splitlines()[-1]
    )
except json.JSONDecodeError as exc:
    raise AssetStageError(
        "Piper model validation succeeded, but the output was not valid JSON.\n"
        f"Output:\n{torch_validation_result.stdout}"
    ) from exc


# -----------------------------------------------------------------------------
# Stage 4: Generate a small PINGO smoke-test set
# -----------------------------------------------------------------------------

for existing_file in SMOKE_OUTPUT_DIR.glob("*.wav"):
    existing_file.unlink()

if CONVERTED_DIR.exists():
    for existing_file in CONVERTED_DIR.glob("*.wav"):
        existing_file.unlink()

# Piper Sample Generator 3.x uses positional text and these CLI options.
generation_command = [
    str(ENV_PYTHON),
    "-m",
    "piper_sample_generator",
    WAKE_WORD,
    "--model",
    str(PIPER_GENERATOR_MODEL_PATH),
    "--max-samples",
    "8",
    "--batch-size",
    "4",
    "--max-speakers",
    "500",
    "--output-dir",
    str(SMOKE_OUTPUT_DIR),
]

generation_result = run_command(
    generation_command,
    stage_name="Generate eight synthetic PINGO smoke-test samples",
    timeout_seconds=1800,
    cwd=PIPER_GENERATOR_REPO,
    extra_env={
        "PYTHONPATH": PIPER_PYTHONPATH,
    },
    print_output=True,
)

GENERATED_WAVS = sorted(SMOKE_OUTPUT_DIR.glob("*.wav"))

if len(GENERATED_WAVS) < 8:
    raise AssetStageError(
        "Piper generation completed but produced fewer files than expected.\n"
        f"Expected: 8\n"
        f"Found: {len(GENERATED_WAVS)}\n"
        f"Directory: {SMOKE_OUTPUT_DIR}\n"
        f"Generator stdout:\n{generation_result.stdout}\n"
        f"Generator stderr:\n{generation_result.stderr}"
    )

GENERATED_WAV_REPORTS = [inspect_wav(path) for path in GENERATED_WAVS]


# -----------------------------------------------------------------------------
# Stage 5: Convert smoke-test files to deployment format
# -----------------------------------------------------------------------------

CONVERTED_DIR.mkdir(parents=True, exist_ok=True)
CONVERTED_WAV_REPORTS: List[Dict[str, Any]] = []

for source_path in GENERATED_WAVS:
    output_path = CONVERTED_DIR / source_path.name

    run_command(
        [
            "ffmpeg",
            "-hide_banner",
            "-loglevel",
            "error",
            "-y",
            "-i",
            str(source_path),
            "-ar",
            "16000",
            "-ac",
            "1",
            "-c:a",
            "pcm_s16le",
            str(output_path),
        ],
        stage_name=f"Convert {source_path.name} to 16 kHz mono",
        timeout_seconds=300,
        cwd=TRAINING_ROOT,
        print_output=False,
    )

    report = inspect_wav(output_path)

    if report["sample_rate"] != 16000:
        raise AssetStageError(
            f"Converted file is not 16 kHz: {output_path}"
        )

    if report["channels"] != 1:
        raise AssetStageError(
            f"Converted file is not mono: {output_path}"
        )

    if report["sample_width_bytes"] != 2:
        raise AssetStageError(
            f"Converted file is not PCM16: {output_path}"
        )

    CONVERTED_WAV_REPORTS.append(report)


# -----------------------------------------------------------------------------
# Stage 6: Save complete asset manifest
# -----------------------------------------------------------------------------

CELL_COMPLETED_AT = dt.datetime.now(dt.timezone.utc)

ASSET_MANIFEST = {
    "stage": "Cell 7 — Model asset setup",
    "passed": True,
    "started_at_utc": CELL_STARTED_AT.isoformat(),
    "completed_at_utc": CELL_COMPLETED_AT.isoformat(),
    "wake_word": WAKE_WORD,
    "openwakeword": {
        "download_report": OPENWAKEWORD_DOWNLOAD_REPORT,
        "assets": OPENWAKEWORD_ASSET_REPORTS,
    },
    "piper_generator": {
        "asset": PIPER_MODEL_REPORT,
        "config": PIPER_CONFIG_REPORT,
        "torch_validation": PIPER_TORCH_VALIDATION,
        "generation_command": generation_command,
    },
    "smoke_test": {
        "generated_count": len(GENERATED_WAV_REPORTS),
        "generated_files": GENERATED_WAV_REPORTS,
        "converted_count": len(CONVERTED_WAV_REPORTS),
        "converted_files": CONVERTED_WAV_REPORTS,
    },
    "paths": {
        "models_root": str(MODELS_DIR),
        "repositories_root": str(REPOS_DIR),
        "piper_repository": str(PIPER_GENERATOR_REPO),
        "openwakeword_models": str(OPENWAKEWORD_MODELS_DIR),
        "piper_models": str(PIPER_MODELS_DIR),
        "piper_generator_model": str(PIPER_GENERATOR_MODEL_PATH),
        "smoke_output": str(SMOKE_OUTPUT_DIR),
        "converted_output": str(CONVERTED_DIR),
        "manifest": str(ASSET_MANIFEST_PATH),
        "log": str(CELL_LOG_PATH),
    },
}

ASSET_MANIFEST_PATH.write_text(
    json.dumps(
        ASSET_MANIFEST,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)

append_log(
    "\nFINAL ASSET MANIFEST\n"
    + json.dumps(
        ASSET_MANIFEST,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )
)


# -----------------------------------------------------------------------------
# Completion summary
# -----------------------------------------------------------------------------

print("\n" + "=" * 88)
print("CELL 7 COMPLETED SUCCESSFULLY")
print("=" * 88)
print(f"Wake word                  : {WAKE_WORD}")
print(f"Piper repository           : {PIPER_GENERATOR_REPO}")
print(f"OpenWakeWord assets        : {len(OPENWAKEWORD_ASSET_REPORTS)}")
print(f"Piper generator model      : {PIPER_GENERATOR_MODEL_PATH}")
print(f"Piper model size           : {PIPER_MODEL_REPORT['size_bytes']:,} bytes")
print(f"Piper model SHA-256        : {PIPER_MODEL_REPORT['sha256']}")
print(f"Piper checkpoint type      : {PIPER_TORCH_VALIDATION['loaded_type']}")
print(f"piper_train imported       : {PIPER_TORCH_VALIDATION['piper_train_imported']}")
print(f"Generated smoke samples    : {len(GENERATED_WAV_REPORTS)}")
print(f"Validated 16 kHz samples   : {len(CONVERTED_WAV_REPORTS)}")
print(f"Smoke-test directory       : {SMOKE_OUTPUT_DIR}")
print(f"Asset manifest             : {ASSET_MANIFEST_PATH}")
print(f"Full log                   : {CELL_LOG_PATH}")

print(
    "\nNext: Cell 8 will define PINGO pronunciations, phonetic "
    "variants, confusion words, negative phrases, and deterministic "
    "dataset-generation manifests."
)
print("=" * 88)

In [ ]:
# =============================================================================
# Cell 8 — PINGO pronunciation, confusion-word, and dataset manifest
# =============================================================================
#
# Purpose:
#   1. Define the target wake-word pronunciations.
#   2. Define useful positive phrase variants.
#   3. Define difficult confusion words and negative phrases.
#   4. Define deterministic train/validation/test allocation.
#   5. Define restart-safe positive-generation batches.
#   6. Save JSON, JSONL, CSV, and plain-text manifests.
#
# This cell does not generate audio.
#
# Positive dataset target:
#   Default = 50,000 generated utterances.
#
# Recommended first run:
#   Set POSITIVE_SAMPLE_TARGET = 100
#   Run Cells 8–10 and inspect the samples.
#   Then return here and change it to 50_000.
#
# =============================================================================

from __future__ import annotations

import csv
import datetime as dt
import hashlib
import json
import math
import os
import random
import re
import sys
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Sequence


# -----------------------------------------------------------------------------
# Recover base configuration
# -----------------------------------------------------------------------------

try:
    CONFIG
except NameError:
    print("WARNING: CONFIG from Cell 1 was not found. Using fallback paths.")

    class _FallbackConfig:
        TRAINING_ROOT = "/content/pingo_training"
        LOGS_DIR = "/content/pingo_training/logs"
        CONFIG_DIR = "/content/pingo_training/config"
        CACHE_DIR = "/content/pingo_training/cache"
        DATA_DIR = "/content/pingo_training/data"
        ENV_DIR = "/content/pingo-env"
        WAKE_WORD = "PINGO"
        RANDOM_SEED = 42

    CONFIG = _FallbackConfig()


TRAINING_ROOT = Path(CONFIG.TRAINING_ROOT)
LOGS_DIR = Path(CONFIG.LOGS_DIR)
CONFIG_DIR = Path(CONFIG.CONFIG_DIR)
CACHE_DIR = Path(CONFIG.CACHE_DIR)
DATA_DIR = Path(
    getattr(
        CONFIG,
        "DATA_DIR",
        TRAINING_ROOT / "data",
    )
)

WAKE_WORD = str(
    getattr(CONFIG, "WAKE_WORD", "PINGO")
).strip()

RANDOM_SEED = int(
    getattr(CONFIG, "RANDOM_SEED", 42)
)

ENV_DIR = Path(CONFIG.ENV_DIR)
ENV_PYTHON = ENV_DIR / "bin" / "python"

MANIFEST_ROOT = DATA_DIR / "manifests"
POSITIVE_MANIFEST_DIR = MANIFEST_ROOT / "positive"
NEGATIVE_MANIFEST_DIR = MANIFEST_ROOT / "negative"
REPORTS_DIR = TRAINING_ROOT / "reports"

for directory in (
    LOGS_DIR,
    CONFIG_DIR,
    CACHE_DIR,
    DATA_DIR,
    MANIFEST_ROOT,
    POSITIVE_MANIFEST_DIR,
    NEGATIVE_MANIFEST_DIR,
    REPORTS_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)


# -----------------------------------------------------------------------------
# User-adjustable positive-generation configuration
# -----------------------------------------------------------------------------

# Change to 100 for the first smoke test.
POSITIVE_SAMPLE_TARGET = int(
    getattr(
        CONFIG,
        "POSITIVE_SAMPLE_TARGET",
        10_000,
    )
)

# Keep batches reasonably small so interrupted Colab sessions can resume.
POSITIVE_GENERATION_BATCH_SIZE = int(
    getattr(
        CONFIG,
        "POSITIVE_GENERATION_BATCH_SIZE",
        500,
    )
)

PIPER_INTERNAL_BATCH_SIZE = int(
    getattr(
        CONFIG,
        "PIPER_INTERNAL_BATCH_SIZE",
        16,
    )
)

# The LibriTTS-R generator contains many speakers, but later speakers may have
# less source data. Cell 7 used a conservative maximum.
MAX_PIPER_SPEAKERS = int(
    getattr(
        CONFIG,
        "MAX_PIPER_SPEAKERS",
        500,
    )
)

TARGET_SAMPLE_RATE = 16_000
TARGET_CHANNELS = 1
TARGET_SAMPLE_WIDTH_BYTES = 2

TRAIN_FRACTION = 0.80
VALIDATION_FRACTION = 0.10
TEST_FRACTION = 0.10

if not math.isclose(
    TRAIN_FRACTION + VALIDATION_FRACTION + TEST_FRACTION,
    1.0,
    abs_tol=1e-9,
):
    raise ValueError(
        "Train, validation, and test fractions must total 1.0."
    )

if POSITIVE_SAMPLE_TARGET <= 0:
    raise ValueError("POSITIVE_SAMPLE_TARGET must be greater than zero.")

if POSITIVE_GENERATION_BATCH_SIZE <= 0:
    raise ValueError(
        "POSITIVE_GENERATION_BATCH_SIZE must be greater than zero."
    )

if not WAKE_WORD:
    raise ValueError("WAKE_WORD cannot be empty.")

if not ENV_PYTHON.exists():
    raise FileNotFoundError(
        f"Missing isolated Python: {ENV_PYTHON}\n"
        "Run Cell 3 before Cell 8."
    )


# -----------------------------------------------------------------------------
# Target pronunciations
# -----------------------------------------------------------------------------
#
# Primary pronunciation:
#     PIN-go
#
# Useful synthetic spellings are included because TTS systems may pronounce
# an invented brand/name differently depending on punctuation and casing.
#
# These variants remain positive only when they audibly represent PINGO.
#
# -----------------------------------------------------------------------------

PRIMARY_PRONUNCIATIONS = [
    {
        "id": "pingo_standard",
        "text": "Pingo",
        "description": "Normal title-case spelling.",
        "weight": 32,
        "expected_pronunciation": "PIN-go",
    },
    {
        "id": "pingo_uppercase",
        "text": "PINGO",
        "description": "Uppercase spelling.",
        "weight": 18,
        "expected_pronunciation": "PIN-go",
    },
    {
        "id": "pingo_punctuation",
        "text": "Pingo.",
        "description": "Short sentence with terminal punctuation.",
        "weight": 14,
        "expected_pronunciation": "PIN-go",
    },
    {
        "id": "pingo_exclamation",
        "text": "Pingo!",
        "description": "More energetic terminal punctuation.",
        "weight": 8,
        "expected_pronunciation": "PIN-go",
    },
    {
        "id": "pingo_comma",
        "text": "Pingo,",
        "description": "Wake word followed by a slight continuation pause.",
        "weight": 8,
        "expected_pronunciation": "PIN-go",
    },
    {
        "id": "pingo_hyphenated",
        "text": "Pin-go",
        "description": "Pronunciation-guiding spelling.",
        "weight": 8,
        "expected_pronunciation": "PIN-go",
    },
    {
        "id": "pingo_spaced",
        "text": "Pin go",
        "description": "Pronunciation-guiding two-token form.",
        "weight": 6,
        "expected_pronunciation": "PIN-go",
    },
    {
        "id": "pingo_phonetic",
        "text": "Ping-oh",
        "description": "Alternative pronunciation-guiding spelling.",
        "weight": 6,
        "expected_pronunciation": "PING-oh or PIN-go",
    },
]


# -----------------------------------------------------------------------------
# Positive phrase contexts
# -----------------------------------------------------------------------------
#
# A wake-word model should recognize the keyword:
#   - alone;
#   - at the beginning of an utterance;
#   - after short natural lead-in words;
#   - before realistic commands.
#
# The standalone form remains dominant.
#
# -----------------------------------------------------------------------------

POSITIVE_CONTEXTS = [
    {
        "id": "standalone",
        "template": "{wake}",
        "weight": 48,
        "category": "standalone",
    },
    {
        "id": "hey_prefix",
        "template": "Hey {wake}",
        "weight": 8,
        "category": "prefix",
    },
    {
        "id": "hello_prefix",
        "template": "Hello {wake}",
        "weight": 4,
        "category": "prefix",
    },
    {
        "id": "okay_prefix",
        "template": "Okay {wake}",
        "weight": 4,
        "category": "prefix",
    },
    {
        "id": "please_prefix",
        "template": "Please, {wake}",
        "weight": 2,
        "category": "prefix",
    },
    {
        "id": "are_you_there",
        "template": "{wake}, are you there?",
        "weight": 3,
        "category": "continuation",
    },
    {
        "id": "listen",
        "template": "{wake}, listen.",
        "weight": 3,
        "category": "command",
    },
    {
        "id": "play_music",
        "template": "{wake}, play some music.",
        "weight": 3,
        "category": "command",
    },
    {
        "id": "stop_music",
        "template": "{wake}, stop the music.",
        "weight": 3,
        "category": "command",
    },
    {
        "id": "volume_up",
        "template": "{wake}, turn up the volume.",
        "weight": 2,
        "category": "command",
    },
    {
        "id": "volume_down",
        "template": "{wake}, lower the volume.",
        "weight": 2,
        "category": "command",
    },
    {
        "id": "lights_on",
        "template": "{wake}, turn on the lights.",
        "weight": 2,
        "category": "command",
    },
    {
        "id": "lights_off",
        "template": "{wake}, turn off the lights.",
        "weight": 2,
        "category": "command",
    },
    {
        "id": "weather",
        "template": "{wake}, what is the weather?",
        "weight": 2,
        "category": "question",
    },
    {
        "id": "time",
        "template": "{wake}, what time is it?",
        "weight": 2,
        "category": "question",
    },
    {
        "id": "tell_me",
        "template": "{wake}, tell me something.",
        "weight": 2,
        "category": "command",
    },
    {
        "id": "help",
        "template": "{wake}, help me.",
        "weight": 2,
        "category": "command",
    },
    {
        "id": "wake_up",
        "template": "Wake up, {wake}.",
        "weight": 2,
        "category": "prefix",
    },
    {
        "id": "thank_you",
        "template": "Thank you, {wake}.",
        "weight": 1,
        "category": "natural_speech",
    },
    {
        "id": "can_you_hear",
        "template": "{wake}, can you hear me?",
        "weight": 3,
        "category": "question",
    },
]


# -----------------------------------------------------------------------------
# Confusion words and hard negatives
# -----------------------------------------------------------------------------
#
# These should never be placed in the positive dataset.
#
# They are used later for:
#   - synthetic hard-negative generation;
#   - false-positive tests;
#   - threshold calibration;
#   - real-world adversarial testing.
#
# -----------------------------------------------------------------------------

CONFUSION_WORDS = [
    # Strongest phonetic confusions
    "Bingo",
    "Dingo",
    "Lingo",
    "Mingo",
    "Ringo",
    "Tingo",
    "Wingo",
    "Zingo",
    "Pinga",
    "Pingu",
    "Pingoed",

    # Similar first consonant or ending
    "Pinto",
    "Pinky",
    "Pinko",
    "Ping",
    "Pinging",
    "Ping pong",

    # Do not include "Pin go" here.
    # It is intentionally used as a positive pronunciation guide.
    "Pin code",
    "Pin glow",
    "Pin low",
    "Pin row",
    "Pin toe",
    "Pico",
    "Pogo",
    "Pino",
    "Pinochle",

    # Words previously confused with PINGO/BINGO
    "King",
    "King go",
    "Single",
    "Jingle",
    "Jingo",
    "Ginkgo",
    "Gringo",

    # Other assistant names
    "Alexa",
    "Siri",
    "Jarvis",
    "Mycroft",
    "Computer",
    "Assistant",
    "Google",
]


HARD_NEGATIVE_PHRASES = [
    "Bingo",
    "Play bingo",
    "Bingo was the answer",
    "We played bingo yesterday",
    "That is a bingo card",
    "Dingo",
    "The dingo ran away",
    "Lingo",
    "I do not understand the lingo",
    "Ringo",
    "Ringo is a name",
    "Ping",
    "Send me a ping",
    "Ping the server",
    "The ping is too high",
    "Ping pong",
    "Let us play ping pong",
    "Pinto",
    "A pinto horse",
    "Pin code",
    "Enter the pin code",
    "What is the pin code",
    "Pink",
    "The light is pink",
    "Single",
    "Play the single",
    "Jingle",
    "That advertisement has a jingle",
    "Ginkgo",
    "Ginkgo is a tree",
    "Gringo",
    "King",
    "The king is here",
    "Please go",
    "Can you go",
    "Let him go",
    "Where did it go",
    "Bring the music",
    "Play music",
    "Stop music",
    "Turn up the volume",
    "Turn down the volume",
    "Hey assistant",
    "Hello assistant",
    "Okay assistant",
    "Hey Siri",
    "Okay Google",
    "Alexa",
    "Hey Alexa",
    "Jarvis",
    "Hey Jarvis",
]


# -----------------------------------------------------------------------------
# Safety validation for positive and negative text
# -----------------------------------------------------------------------------

# -----------------------------------------------------------------------------
# Safety validation for positive and negative text
# -----------------------------------------------------------------------------

def normalized_text(value: str) -> str:
    return re.sub(
        r"[^a-z0-9]+",
        " ",
        value.lower(),
    ).strip()


positive_text_map = {
    normalized_text(item["text"]): item["text"]
    for item in PRIMARY_PRONUNCIATIONS
}

negative_text_map: Dict[str, List[str]] = {}

for negative_text in (
    CONFUSION_WORDS + HARD_NEGATIVE_PHRASES
):
    normalized = normalized_text(negative_text)

    negative_text_map.setdefault(
        normalized,
        [],
    ).append(negative_text)


overlap = sorted(
    set(positive_text_map).intersection(
        negative_text_map
    )
)

if overlap:
    overlap_details = []

    for normalized in overlap:
        overlap_details.append(
            {
                "normalized": normalized,
                "positive": positive_text_map[normalized],
                "negative": negative_text_map[normalized],
            }
        )

    raise ValueError(
        "Positive pronunciation text overlaps with negative data.\n"
        + json.dumps(
            overlap_details,
            indent=2,
            ensure_ascii=False,
        )
    )

# -----------------------------------------------------------------------------
# Deterministic weighted expansion
# -----------------------------------------------------------------------------

def validate_weighted_items(
    items: Sequence[Dict[str, Any]],
    item_name: str,
) -> None:
    if not items:
        raise ValueError(f"{item_name} cannot be empty.")

    identifiers = [str(item["id"]) for item in items]

    duplicate_ids = [
        identifier
        for identifier, count in Counter(identifiers).items()
        if count > 1
    ]

    if duplicate_ids:
        raise ValueError(
            f"Duplicate IDs in {item_name}: {duplicate_ids}"
        )

    for item in items:
        if int(item["weight"]) <= 0:
            raise ValueError(
                f"Non-positive weight in {item_name}: {item}"
            )


validate_weighted_items(
    PRIMARY_PRONUNCIATIONS,
    "PRIMARY_PRONUNCIATIONS",
)

validate_weighted_items(
    POSITIVE_CONTEXTS,
    "POSITIVE_CONTEXTS",
)


def weighted_choice(
    rng: random.Random,
    items: Sequence[Dict[str, Any]],
) -> Dict[str, Any]:
    weights = [int(item["weight"]) for item in items]
    return rng.choices(
        population=list(items),
        weights=weights,
        k=1,
    )[0]


def deterministic_split(
    sample_id: str,
) -> str:
    """
    Assign a split using a SHA-256-derived value.

    The split remains stable even if the manifest is regenerated.
    """
    digest = hashlib.sha256(
        sample_id.encode("utf-8")
    ).digest()

    value = int.from_bytes(
        digest[:8],
        byteorder="big",
        signed=False,
    ) / float(2**64)

    if value < TRAIN_FRACTION:
        return "train"

    if value < TRAIN_FRACTION + VALIDATION_FRACTION:
        return "validation"

    return "test"


def stable_sample_seed(
    base_seed: int,
    sample_index: int,
) -> int:
    payload = f"{base_seed}:{sample_index}".encode("utf-8")
    digest = hashlib.sha256(payload).digest()
    return int.from_bytes(digest[:8], "big")


# -----------------------------------------------------------------------------
# Create generation jobs
# -----------------------------------------------------------------------------

@dataclass(frozen=True)
class PositiveSampleJob:
    sample_index: int
    sample_id: str
    sample_seed: int

    pronunciation_id: str
    pronunciation_text: str
    expected_pronunciation: str

    context_id: str
    context_category: str
    generation_text: str

    batch_index: int
    split: str

    raw_relative_path: str
    converted_relative_path: str


POSITIVE_RAW_ROOT = DATA_DIR / "positive_raw"
POSITIVE_CONVERTED_ROOT = DATA_DIR / "positive_16khz"

POSITIVE_RAW_ROOT.mkdir(parents=True, exist_ok=True)
POSITIVE_CONVERTED_ROOT.mkdir(parents=True, exist_ok=True)

jobs: List[PositiveSampleJob] = []

for sample_index in range(POSITIVE_SAMPLE_TARGET):
    sample_seed = stable_sample_seed(
        RANDOM_SEED,
        sample_index,
    )

    rng = random.Random(sample_seed)

    pronunciation = weighted_choice(
        rng,
        PRIMARY_PRONUNCIATIONS,
    )

    context = weighted_choice(
        rng,
        POSITIVE_CONTEXTS,
    )

    generation_text = context["template"].format(
        wake=pronunciation["text"]
    ).strip()

    sample_id = (
        f"pingo_positive_{sample_index:07d}"
    )

    batch_index = (
        sample_index // POSITIVE_GENERATION_BATCH_SIZE
    )

    split = deterministic_split(sample_id)

    raw_relative_path = str(
        Path(f"batch_{batch_index:05d}")
        / f"{sample_id}.wav"
    )

    converted_relative_path = str(
        Path(split)
        / f"batch_{batch_index:05d}"
        / f"{sample_id}.wav"
    )

    jobs.append(
        PositiveSampleJob(
            sample_index=sample_index,
            sample_id=sample_id,
            sample_seed=sample_seed,
            pronunciation_id=pronunciation["id"],
            pronunciation_text=pronunciation["text"],
            expected_pronunciation=(
                pronunciation["expected_pronunciation"]
            ),
            context_id=context["id"],
            context_category=context["category"],
            generation_text=generation_text,
            batch_index=batch_index,
            split=split,
            raw_relative_path=raw_relative_path,
            converted_relative_path=(
                converted_relative_path
            ),
        )
    )


# -----------------------------------------------------------------------------
# Validate job uniqueness
# -----------------------------------------------------------------------------

sample_ids = [job.sample_id for job in jobs]

if len(sample_ids) != len(set(sample_ids)):
    raise RuntimeError("Duplicate positive sample IDs detected.")

raw_paths = [job.raw_relative_path for job in jobs]

if len(raw_paths) != len(set(raw_paths)):
    raise RuntimeError(
        "Duplicate raw output paths detected."
    )


# -----------------------------------------------------------------------------
# Create grouped batch definitions
# -----------------------------------------------------------------------------

batch_map: Dict[int, List[PositiveSampleJob]] = {}

for job in jobs:
    batch_map.setdefault(job.batch_index, []).append(job)

batch_definitions = []

for batch_index in sorted(batch_map):
    batch_jobs = batch_map[batch_index]

    # Piper's CLI generates one text phrase per process. To preserve exact
    # labels, jobs are grouped again by phrase inside each batch.
    text_groups: Dict[str, List[PositiveSampleJob]] = {}

    for job in batch_jobs:
        text_groups.setdefault(
            job.generation_text,
            [],
        ).append(job)

    batch_definitions.append(
        {
            "batch_index": batch_index,
            "sample_count": len(batch_jobs),
            "first_sample_index": (
                batch_jobs[0].sample_index
            ),
            "last_sample_index": (
                batch_jobs[-1].sample_index
            ),
            "raw_directory": str(
                POSITIVE_RAW_ROOT
                / f"batch_{batch_index:05d}"
            ),
            "text_group_count": len(text_groups),
            "text_groups": [
                {
                    "generation_text": text,
                    "sample_count": len(group_jobs),
                    "sample_ids": [
                        item.sample_id
                        for item in group_jobs
                    ],
                }
                for text, group_jobs in sorted(
                    text_groups.items()
                )
            ],
        }
    )


# -----------------------------------------------------------------------------
# Write manifests
# -----------------------------------------------------------------------------

POSITIVE_JOBS_JSONL_PATH = (
    POSITIVE_MANIFEST_DIR / "positive_jobs.jsonl"
)

POSITIVE_JOBS_CSV_PATH = (
    POSITIVE_MANIFEST_DIR / "positive_jobs.csv"
)

POSITIVE_BATCHES_JSON_PATH = (
    POSITIVE_MANIFEST_DIR / "positive_batches.json"
)

PRONUNCIATIONS_JSON_PATH = (
    POSITIVE_MANIFEST_DIR / "pronunciations.json"
)

CONFUSIONS_JSON_PATH = (
    NEGATIVE_MANIFEST_DIR / "confusions.json"
)

HARD_NEGATIVE_TEXT_PATH = (
    NEGATIVE_MANIFEST_DIR / "hard_negative_phrases.txt"
)

DATASET_CONFIGURATION_PATH = (
    CONFIG_DIR / "dataset_generation_config.json"
)


with POSITIVE_JOBS_JSONL_PATH.open(
    "w",
    encoding="utf-8",
) as handle:
    for job in jobs:
        handle.write(
            json.dumps(
                asdict(job),
                ensure_ascii=False,
                sort_keys=True,
            )
            + "\n"
        )


csv_fieldnames = list(asdict(jobs[0]).keys())

with POSITIVE_JOBS_CSV_PATH.open(
    "w",
    newline="",
    encoding="utf-8",
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=csv_fieldnames,
    )
    writer.writeheader()

    for job in jobs:
        writer.writerow(asdict(job))


POSITIVE_BATCHES_JSON_PATH.write_text(
    json.dumps(
        batch_definitions,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


PRONUNCIATIONS_JSON_PATH.write_text(
    json.dumps(
        {
            "wake_word": WAKE_WORD,
            "pronunciations": PRIMARY_PRONUNCIATIONS,
            "contexts": POSITIVE_CONTEXTS,
        },
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


CONFUSIONS_JSON_PATH.write_text(
    json.dumps(
        {
            "wake_word": WAKE_WORD,
            "confusion_words": sorted(
                set(CONFUSION_WORDS)
            ),
            "hard_negative_phrases": sorted(
                set(HARD_NEGATIVE_PHRASES)
            ),
        },
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


HARD_NEGATIVE_TEXT_PATH.write_text(
    "\n".join(
        sorted(set(HARD_NEGATIVE_PHRASES))
    )
    + "\n",
    encoding="utf-8",
)


split_counts = Counter(job.split for job in jobs)
pronunciation_counts = Counter(
    job.pronunciation_id for job in jobs
)
context_counts = Counter(
    job.context_id for job in jobs
)
phrase_counts = Counter(
    job.generation_text for job in jobs
)


DATASET_CONFIGURATION = {
    "stage": "Cell 8 — Dataset text manifest",
    "created_at_utc": dt.datetime.now(
        dt.timezone.utc
    ).isoformat(),
    "wake_word": WAKE_WORD,
    "random_seed": RANDOM_SEED,
    "positive_sample_target": POSITIVE_SAMPLE_TARGET,
    "positive_generation_batch_size": (
        POSITIVE_GENERATION_BATCH_SIZE
    ),
    "piper_internal_batch_size": (
        PIPER_INTERNAL_BATCH_SIZE
    ),
    "max_piper_speakers": MAX_PIPER_SPEAKERS,
    "target_audio": {
        "sample_rate": TARGET_SAMPLE_RATE,
        "channels": TARGET_CHANNELS,
        "sample_width_bytes": (
            TARGET_SAMPLE_WIDTH_BYTES
        ),
        "format": "WAV PCM16",
    },
    "split_fractions": {
        "train": TRAIN_FRACTION,
        "validation": VALIDATION_FRACTION,
        "test": TEST_FRACTION,
    },
    "actual_split_counts": dict(split_counts),
    "batch_count": len(batch_definitions),
    "unique_generation_phrases": len(phrase_counts),
    "pronunciation_counts": dict(
        sorted(pronunciation_counts.items())
    ),
    "context_counts": dict(
        sorted(context_counts.items())
    ),
    "confusion_word_count": len(
        set(CONFUSION_WORDS)
    ),
    "hard_negative_phrase_count": len(
        set(HARD_NEGATIVE_PHRASES)
    ),
    "paths": {
        "positive_raw_root": str(
            POSITIVE_RAW_ROOT
        ),
        "positive_converted_root": str(
            POSITIVE_CONVERTED_ROOT
        ),
        "positive_jobs_jsonl": str(
            POSITIVE_JOBS_JSONL_PATH
        ),
        "positive_jobs_csv": str(
            POSITIVE_JOBS_CSV_PATH
        ),
        "positive_batches_json": str(
            POSITIVE_BATCHES_JSON_PATH
        ),
        "pronunciations": str(
            PRONUNCIATIONS_JSON_PATH
        ),
        "confusions": str(
            CONFUSIONS_JSON_PATH
        ),
        "hard_negative_text": str(
            HARD_NEGATIVE_TEXT_PATH
        ),
    },
}


DATASET_CONFIGURATION_PATH.write_text(
    json.dumps(
        DATASET_CONFIGURATION,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


# -----------------------------------------------------------------------------
# Manifest integrity hash
# -----------------------------------------------------------------------------

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            block = handle.read(1024 * 1024)

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


manifest_hashes = {
    path.name: sha256_file(path)
    for path in (
        POSITIVE_JOBS_JSONL_PATH,
        POSITIVE_JOBS_CSV_PATH,
        POSITIVE_BATCHES_JSON_PATH,
        PRONUNCIATIONS_JSON_PATH,
        CONFUSIONS_JSON_PATH,
        HARD_NEGATIVE_TEXT_PATH,
        DATASET_CONFIGURATION_PATH,
    )
}

MANIFEST_HASH_PATH = (
    MANIFEST_ROOT / "manifest_sha256.json"
)

MANIFEST_HASH_PATH.write_text(
    json.dumps(
        manifest_hashes,
        indent=2,
        sort_keys=True,
    )
    + "\n",
    encoding="utf-8",
)


# -----------------------------------------------------------------------------
# Completion summary
# -----------------------------------------------------------------------------

print("=" * 88)
print("CELL 8 COMPLETED SUCCESSFULLY")
print("=" * 88)
print(f"Wake word                  : {WAKE_WORD}")
print(f"Positive sample target     : {POSITIVE_SAMPLE_TARGET:,}")
print(
    f"Generation batch size      : "
    f"{POSITIVE_GENERATION_BATCH_SIZE:,}"
)
print(f"Generation batch count     : {len(batch_definitions):,}")
print(f"Unique positive phrases    : {len(phrase_counts):,}")
print(f"Confusion words            : {len(set(CONFUSION_WORDS)):,}")
print(
    f"Hard-negative phrases      : "
    f"{len(set(HARD_NEGATIVE_PHRASES)):,}"
)

print("\nDataset splits")
print(f"  Train                     : {split_counts['train']:,}")
print(
    f"  Validation                : "
    f"{split_counts['validation']:,}"
)
print(f"  Test                      : {split_counts['test']:,}")

print("\nMost common generation phrases")

for phrase, count in phrase_counts.most_common(15):
    print(f"  {count:>7,}  {phrase}")

print("\nManifest paths")
print(f"  Jobs JSONL                : {POSITIVE_JOBS_JSONL_PATH}")
print(f"  Jobs CSV                  : {POSITIVE_JOBS_CSV_PATH}")
print(f"  Batches                   : {POSITIVE_BATCHES_JSON_PATH}")
print(f"  Dataset configuration     : {DATASET_CONFIGURATION_PATH}")
print(f"  Manifest hashes           : {MANIFEST_HASH_PATH}")

print("\nImportant:")
print(
    "  Inspect the positive phrases and confusion words before "
    "starting the full 50,000-sample generation."
)
print(
    "  Similar words such as Bingo, Dingo, Lingo, Ringo, Ping, "
    "Pinto, and Pin code are intentionally negative."
)
print(
    "\nNext: Cell 9 generates positive samples in restart-safe "
    "phrase groups and converts them to deterministic filenames."
)
print("=" * 88)

In [ ]:
# =============================================================================
# Cell 9 — Parallel, restart-safe positive PINGO sample generation
# =============================================================================
#
# Prerequisites:
#   - Cell 7 completed successfully.
#   - Cell 8 created positive_jobs.jsonl and positive_batches.json.
#
# This cell:
#   1. Reads the deterministic positive manifest from Cell 8.
#   2. Groups jobs by batch and exact generation phrase.
#   3. Generates only missing samples.
#   4. Uses isolated staging directories for every phrase group.
#   5. Renames generated WAV files to deterministic manifest filenames.
#   6. Runs independent phrase groups concurrently.
#   7. Writes restart-safe progress and generation reports.
#
# IMPORTANT:
#   Test first with POSITIVE_SAMPLE_TARGET = 100 in Cell 8.
#   Do not begin 50,000-sample generation until the first samples sound correct.
# =============================================================================

from __future__ import annotations

import datetime as dt
import hashlib
import multiprocessing
import threading
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
import json
import os
import shlex
import shutil
import sqlite3
import subprocess
import textwrap
import time
import wave
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

try:
    CONFIG
except NameError:
    class _FallbackConfig:
        TRAINING_ROOT = "/content/pingo_training"
        LOGS_DIR = "/content/pingo_training/logs"
        CONFIG_DIR = "/content/pingo_training/config"
        CACHE_DIR = "/content/pingo_training/cache"
        REPOSITORIES_DIR = (
            "/content/pingo_training/repositories"
        )
        ENV_DIR = "/content/pingo-env"
        WAKE_WORD = "PINGO"
        RANDOM_SEED = 42

    CONFIG = _FallbackConfig()


# -----------------------------------------------------------------------------
# Persistent-storage and runtime configuration
# -----------------------------------------------------------------------------
#
# Large datasets, manifests, models, repositories, logs and reports are stored
# in Google Drive. The Python virtual environment and temporary Piper staging
# remain in /content because running thousands of tiny operations directly
# inside Drive is much slower.
# -----------------------------------------------------------------------------

IN_COLAB = "google.colab" in __import__("sys").modules

if IN_COLAB:
    from google.colab import drive

    DRIVE_MOUNT = Path("/content/drive")
    if not (DRIVE_MOUNT / "MyDrive").exists():
        drive.mount(str(DRIVE_MOUNT), force_remount=False)

PERSISTENT_TRAINING_ROOT = Path(
    os.environ.get(
        "PINGO_PERSISTENT_ROOT",
        "/content/drive/MyDrive/pingo_training"
        if IN_COLAB
        else str(CONFIG.TRAINING_ROOT),
    )
)

LEGACY_TRAINING_ROOT = Path(str(CONFIG.TRAINING_ROOT))
TRAINING_ROOT = PERSISTENT_TRAINING_ROOT

LOGS_DIR = TRAINING_ROOT / "logs"
CONFIG_DIR = TRAINING_ROOT / "config"
CACHE_DIR = TRAINING_ROOT / "cache"
MODELS_DIR = TRAINING_ROOT / "models"
REPOS_DIR = TRAINING_ROOT / "repositories"

# Keep the venv local. A venv stored in Google Drive is slower and may contain
# broken symlinks after a Colab runtime restart.
ENV_DIR = Path(os.environ.get("PINGO_ENV_DIR", "/content/pingo-env"))
ENV_PYTHON = ENV_DIR / "bin" / "python"

WAKE_WORD = str(getattr(CONFIG, "WAKE_WORD", "PINGO")).strip()

OPENWAKEWORD_REPO = REPOS_DIR / "openWakeWord"
PIPER_MODELS_DIR = MODELS_DIR / "piper"
OPENWAKEWORD_MODELS_DIR = MODELS_DIR / "openwakeword"


def _valid_piper_repo(path: Path) -> bool:
    return all(
        required.exists()
        for required in (
            path,
            path / "piper_sample_generator",
            path / "piper_train",
            path / "piper_train" / "vits",
        )
    )


def _valid_model(path: Path) -> bool:
    return path.exists() and path.stat().st_size >= 1_000_000


def _valid_config(path: Path) -> bool:
    if not path.exists() or path.stat().st_size <= 10:
        return False

    try:
        data = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return False

    return all(
        key in data
        for key in ("audio", "espeak", "num_speakers", "phoneme_id_map")
    )


def _first_valid(candidates, validator):
    seen = set()

    for candidate in candidates:
        candidate = Path(candidate)

        try:
            key = str(candidate.resolve())
        except Exception:
            key = str(candidate)

        if key in seen:
            continue

        seen.add(key)

        if validator(candidate):
            return candidate

    return None


# Prefer fast local runtime assets, then persistent backup, then legacy paths.
_repo_candidates = [
    Path(os.environ.get(
        "PINGO_LOCAL_PIPER_REPO",
        "/content/pingo_runtime_assets/repositories/piper-sample-generator",
    )),
    Path("/content/pingo_training/repositories/piper-sample-generator"),
    Path("/content/pingo_training/repos/piper-sample-generator"),
    Path("/content/piper-sample-generator"),
    TRAINING_ROOT / "assets" / "piper-sample-generator",
    TRAINING_ROOT / "repositories" / "piper-sample-generator",
]

_model_candidates = [
    Path(os.environ.get(
        "PINGO_LOCAL_PIPER_MODEL",
        "/content/pingo_runtime_assets/models/piper/"
        "en_US-libritts_r-medium.pt",
    )),
    Path("/content/pingo_training/models/piper/en_US-libritts_r-medium.pt"),
    Path("/content/pingo_training/models/en_US-libritts_r-medium.pt"),
    Path("/content/piper-sample-generator/models/en_US-libritts_r-medium.pt"),
    TRAINING_ROOT / "assets" / "models" / "piper"
    / "en_US-libritts_r-medium.pt",
    TRAINING_ROOT / "models" / "piper" / "en_US-libritts_r-medium.pt",
]

# Add discovered fallback locations without requiring a separate bootstrap cell.
for _candidate in Path("/content").rglob("piper-sample-generator"):
    if _candidate.is_dir():
        _repo_candidates.append(_candidate)

for _candidate in Path("/content").rglob("en_US-libritts_r-medium.pt"):
    if _candidate.is_file():
        _model_candidates.append(_candidate)

PIPER_GENERATOR_REPO = _first_valid(_repo_candidates, _valid_piper_repo)
PIPER_MODEL_PATH = _first_valid(_model_candidates, _valid_model)

_config_candidates = []

if PIPER_MODEL_PATH is not None:
    _config_candidates.append(Path(str(PIPER_MODEL_PATH) + ".json"))

if PIPER_GENERATOR_REPO is not None:
    _config_candidates.extend(
        [
            PIPER_GENERATOR_REPO / "models"
            / "en_US-libritts_r-medium.pt.json",
            PIPER_GENERATOR_REPO / "models"
            / "en-us-libritts-high.pt.json",
        ]
    )

_config_candidates.extend(
    [
        Path(os.environ.get(
            "PINGO_LOCAL_PIPER_CONFIG",
            "/content/pingo_runtime_assets/models/piper/"
            "en_US-libritts_r-medium.pt.json",
        )),
        Path(
            "/content/pingo_training/models/piper/"
            "en_US-libritts_r-medium.pt.json"
        ),
        TRAINING_ROOT / "assets" / "models" / "piper"
        / "en_US-libritts_r-medium.pt.json",
        TRAINING_ROOT / "models" / "piper"
        / "en_US-libritts_r-medium.pt.json",
    ]
)

for _candidate in Path("/content").rglob("en_US-libritts_r-medium.pt.json"):
    if _candidate.is_file():
        _config_candidates.append(_candidate)

PIPER_CONFIG_PATH = _first_valid(_config_candidates, _valid_config)

# Keep Path-compatible placeholders so the prerequisite report remains clear.
if PIPER_GENERATOR_REPO is None:
    PIPER_GENERATOR_REPO = Path(
        "/content/pingo_runtime_assets/repositories/piper-sample-generator"
    )

if PIPER_MODEL_PATH is None:
    PIPER_MODEL_PATH = Path(
        "/content/pingo_runtime_assets/models/piper/"
        "en_US-libritts_r-medium.pt"
    )

if PIPER_CONFIG_PATH is None:
    PIPER_CONFIG_PATH = Path(str(PIPER_MODEL_PATH) + ".json")

print("[ASSET RESOLUTION]")
print(f"  Piper repository : {PIPER_GENERATOR_REPO}")
print(f"  Piper model      : {PIPER_MODEL_PATH}")
print(f"  Piper config     : {PIPER_CONFIG_PATH}")

# Fast local staging. A failed active group may be lost, but every completed
# group is atomically committed to persistent Drive storage.
STAGING_ROOT = Path(
    os.environ.get("PINGO_LOCAL_STAGING", "/content/pingo_piper_staging")
)

CELL_LOG_PATH = LOGS_DIR / "cell_09_positive_generation.log"
ASSET_MANIFEST_PATH = CONFIG_DIR / "model_asset_manifest.json"
COMPLETION_DB_PATH = CONFIG_DIR / "cell_09_completed_samples.sqlite3"

for directory in (
    TRAINING_ROOT,
    LOGS_DIR,
    CONFIG_DIR,
    CACHE_DIR,
    MODELS_DIR,
    REPOS_DIR,
    PIPER_MODELS_DIR,
    STAGING_ROOT,
):
    directory.mkdir(parents=True, exist_ok=True)


def persistentize_path(value: Any) -> Path:
    """Move a legacy /content/pingo_training path under persistent storage."""
    path = Path(value)

    try:
        relative = path.resolve().relative_to(LEGACY_TRAINING_ROOT.resolve())
    except Exception:
        return path

    return TRAINING_ROOT / relative


# Cell 8 normally defines these globals. Redirect them to Google Drive while
# preserving its exact relative directory layout.
_REQUIRED_CELL8_PATH_NAMES = (
    "POSITIVE_JOBS_JSONL_PATH",
    "POSITIVE_BATCHES_JSON_PATH",
    "POSITIVE_RAW_ROOT",
    "PROGRESS_PATH",
    "GENERATION_REPORT_PATH",
)

missing_cell8_names = [
    name for name in _REQUIRED_CELL8_PATH_NAMES
    if name not in globals()
]

if missing_cell8_names:
    raise RuntimeError(
        "Cell 8 path variables are not defined:\n"
        + "\n".join(f"  - {name}" for name in missing_cell8_names)
        + "\nRun Cell 8 once before running this Cell 9."
    )

for _name in _REQUIRED_CELL8_PATH_NAMES:
    globals()[_name] = persistentize_path(globals()[_name])

POSITIVE_JOBS_JSONL_PATH = Path(POSITIVE_JOBS_JSONL_PATH)
POSITIVE_BATCHES_JSON_PATH = Path(POSITIVE_BATCHES_JSON_PATH)
POSITIVE_RAW_ROOT = Path(POSITIVE_RAW_ROOT)
PROGRESS_PATH = Path(PROGRESS_PATH)
GENERATION_REPORT_PATH = Path(GENERATION_REPORT_PATH)

for directory in (
    POSITIVE_JOBS_JSONL_PATH.parent,
    POSITIVE_BATCHES_JSON_PATH.parent,
    POSITIVE_RAW_ROOT,
    PROGRESS_PATH.parent,
    GENERATION_REPORT_PATH.parent,
):
    directory.mkdir(parents=True, exist_ok=True)


def migrate_legacy_file(source: Path, destination: Path) -> None:
    """Copy a small legacy manifest/config file to Drive once."""
    if destination.exists() or not source.exists():
        return

    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    shutil.copy2(source, temporary)
    os.replace(temporary, destination)
    print(f"[MIGRATED] {source} -> {destination}")


# If Cell 8 ran before Drive persistence was enabled, preserve its manifests.
for _path_name in (
    "POSITIVE_JOBS_JSONL_PATH",
    "POSITIVE_BATCHES_JSON_PATH",
):
    _persistent_path = globals()[_path_name]
    try:
        _relative = _persistent_path.relative_to(TRAINING_ROOT)
        _legacy_path = LEGACY_TRAINING_ROOT / _relative
        migrate_legacy_file(_legacy_path, _persistent_path)
    except Exception:
        pass


# Safe defaults when Cell 8 did not explicitly define performance controls.
CPU_COUNT = max(1, multiprocessing.cpu_count())
MAX_PARALLEL_GROUPS = int(
    globals().get(
        "MAX_PARALLEL_GROUPS",
        min(3, max(1, CPU_COUNT // 2)),
    )
)
MAX_PARALLEL_GROUPS = max(1, min(MAX_PARALLEL_GROUPS, 4))

THREADS_PER_PIPER_PROCESS = int(
    globals().get(
        "THREADS_PER_PIPER_PROCESS",
        max(1, CPU_COUNT // MAX_PARALLEL_GROUPS),
    )
)
THREADS_PER_PIPER_PROCESS = max(1, THREADS_PER_PIPER_PROCESS)

PIPER_INTERNAL_BATCH_SIZE = int(
    globals().get("PIPER_INTERNAL_BATCH_SIZE", 16)
)
MAX_PIPER_SPEAKERS = int(
    globals().get("MAX_PIPER_SPEAKERS", 500)
)
MAX_BATCHES_THIS_RUN = globals().get("MAX_BATCHES_THIS_RUN", None)
PROGRESS_PRINT_INTERVAL = int(
    globals().get("PROGRESS_PRINT_INTERVAL", 10)
)

# -----------------------------------------------------------------------------
# Errors and logging
# -----------------------------------------------------------------------------

class PositiveGenerationError(RuntimeError):
    """Raised when deterministic positive sample generation fails."""


STARTED_AT = dt.datetime.now(dt.timezone.utc)
LOG_LOCK = threading.Lock()

with CELL_LOG_PATH.open("a", encoding="utf-8") as _log_handle:
    _log_handle.write(
        "\nPINGO CELL 9 POSITIVE GENERATION LOG\n"
        f"Started UTC: {STARTED_AT.isoformat()}\n"
        + "=" * 96
        + "\n"
    )


def append_log(message: str) -> None:
    with LOG_LOCK:
        with CELL_LOG_PATH.open("a", encoding="utf-8") as handle:
            handle.write(message.rstrip() + "\n")


def format_command(command: Sequence[str]) -> str:
    return " ".join(shlex.quote(str(item)) for item in command)


def tail_text(text: str, max_lines: int = 100) -> str:
    lines = text.splitlines()
    if len(lines) <= max_lines:
        return text
    return "\n".join(
        [f"... showing final {max_lines} lines ...", *lines[-max_lines:]]
    )


def atomic_write_json(path: Path, data: Dict[str, Any]) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(data, indent=2, ensure_ascii=False, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(1024 * 1024)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def inspect_wav_basic(path: Path, *, include_sha256: bool = False) -> Dict[str, Any]:
    try:
        with wave.open(str(path), "rb") as wav_file:
            channels = wav_file.getnchannels()
            sample_width = wav_file.getsampwidth()
            sample_rate = wav_file.getframerate()
            frame_count = wav_file.getnframes()
    except Exception as exc:
        raise PositiveGenerationError(
            f"Generated file is not a readable WAV: {path}\n"
            f"{type(exc).__name__}: {exc}"
        ) from exc

    if channels <= 0 or sample_width <= 0 or sample_rate <= 0 or frame_count <= 0:
        raise PositiveGenerationError(f"Generated WAV is empty or invalid: {path}")

    report = {
        "path": str(path),
        "size_bytes": path.stat().st_size,
        "channels": channels,
        "sample_width_bytes": sample_width,
        "sample_rate": sample_rate,
        "frame_count": frame_count,
        "duration_seconds": frame_count / sample_rate,
    }

    if include_sha256:
        report["sha256"] = sha256_file(path)

    return report


def run_command(
    command: Sequence[str],
    *,
    stage_name: str,
    cwd: Path,
    timeout_seconds: int = 3600,
    print_output: bool = False,
) -> subprocess.CompletedProcess[str]:
    command = [str(item) for item in command]
    readable = format_command(command)

    environment = os.environ.copy()
    piper_repo_string = str(PIPER_GENERATOR_REPO.resolve())
    existing_pythonpath = environment.get("PYTHONPATH", "")
    environment.update(
        {
            "PYTHONUNBUFFERED": "1",
            "TF_CPP_MIN_LOG_LEVEL": "2",
            "TOKENIZERS_PARALLELISM": "false",
            "MPLBACKEND": "Agg",
            "OMP_NUM_THREADS": str(THREADS_PER_PIPER_PROCESS),
            "MKL_NUM_THREADS": str(THREADS_PER_PIPER_PROCESS),
            "OPENBLAS_NUM_THREADS": str(THREADS_PER_PIPER_PROCESS),
            "NUMEXPR_NUM_THREADS": str(THREADS_PER_PIPER_PROCESS),
            "PYTHONPATH": os.pathsep.join(
                [piper_repo_string, existing_pythonpath]
            ).strip(os.pathsep),
        }
    )

    header = (
        "\n"
        + "=" * 96
        + "\n"
        + f"STAGE: {stage_name}\n"
        + f"COMMAND: {readable}\n"
        + f"CWD: {cwd}\n"
        + "=" * 96
    )
    append_log(header)

    try:
        result = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=False,
            timeout=timeout_seconds,
            cwd=str(cwd.resolve()),
            env=environment,
        )
    except subprocess.TimeoutExpired as exc:
        raise PositiveGenerationError(
            f"Generation command timed out after {timeout_seconds} seconds.\n"
            f"Stage: {stage_name}\nCommand: {readable}"
        ) from exc

    stdout = result.stdout or ""
    stderr = result.stderr or ""

    append_log("\n[STDOUT]\n" + stdout)
    append_log("\n[STDERR]\n" + stderr)
    append_log(f"\n[RETURN CODE]\n{result.returncode}")

    if print_output and stdout.strip():
        print(tail_text(stdout))
    if print_output and stderr.strip():
        print("[stderr]")
        print(tail_text(stderr))

    if result.returncode != 0:
        raise PositiveGenerationError(
            textwrap.dedent(
                f"""
                POSITIVE GENERATION FAILED

                Stage:
                  {stage_name}

                Command:
                  {readable}

                Return code:
                  {result.returncode}

                Stdout:
                {tail_text(stdout)}

                Stderr:
                {tail_text(stderr)}

                Full log:
                  {CELL_LOG_PATH}
                """
            ).strip()
        )

    return result


# -----------------------------------------------------------------------------
# Preconditions
# -----------------------------------------------------------------------------

required_paths = [
    ENV_PYTHON,
    PIPER_GENERATOR_REPO,
    PIPER_GENERATOR_REPO / "piper_sample_generator",
    PIPER_GENERATOR_REPO / "piper_train",
    PIPER_MODEL_PATH,
    PIPER_CONFIG_PATH,
    POSITIVE_JOBS_JSONL_PATH,
    POSITIVE_BATCHES_JSON_PATH,
]

missing_paths = [path for path in required_paths if not path.exists()]

if missing_paths:
    diagnostic_candidates = {
        "repository_candidates": [str(path) for path in _repo_candidates],
        "model_candidates": [str(path) for path in _model_candidates],
        "config_candidates": [str(path) for path in _config_candidates],
    }

    raise PositiveGenerationError(
        "Cell 9 prerequisites are missing after automatic asset discovery:\n"
        + "\n".join(f"  - {path}" for path in missing_paths)
        + "\n\nCell 9 searched these locations:\n"
        + json.dumps(diagnostic_candidates, indent=2)
        + "\n\nRun Cell 8.5 once if no valid assets exist in any listed location."
    )

if PIPER_INTERNAL_BATCH_SIZE <= 0:
    raise ValueError("PIPER_INTERNAL_BATCH_SIZE must be greater than zero.")

if MAX_PIPER_SPEAKERS <= 0:
    raise ValueError("MAX_PIPER_SPEAKERS must be greater than zero.")


# -----------------------------------------------------------------------------
# Load and validate deterministic jobs
# -----------------------------------------------------------------------------

jobs: List[Dict[str, Any]] = []

with POSITIVE_JOBS_JSONL_PATH.open("r", encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        stripped = line.strip()
        if not stripped:
            continue
        try:
            job = json.loads(stripped)
        except json.JSONDecodeError as exc:
            raise PositiveGenerationError(
                f"Invalid JSON in {POSITIVE_JOBS_JSONL_PATH}, line {line_number}"
            ) from exc
        jobs.append(job)

if not jobs:
    raise PositiveGenerationError("The positive job manifest contains no jobs.")

required_job_fields = {
    "sample_index",
    "sample_id",
    "generation_text",
    "batch_index",
    "raw_relative_path",
    "split",
}

for index, job in enumerate(jobs):
    missing_fields = required_job_fields - set(job)
    if missing_fields:
        raise PositiveGenerationError(
            f"Job {index} is missing fields: {sorted(missing_fields)}"
        )

sample_ids = [str(job["sample_id"]) for job in jobs]
raw_relative_paths = [str(job["raw_relative_path"]) for job in jobs]

if len(sample_ids) != len(set(sample_ids)):
    raise PositiveGenerationError("Duplicate sample IDs found in Cell 8 manifest.")

if len(raw_relative_paths) != len(set(raw_relative_paths)):
    raise PositiveGenerationError(
        "Duplicate raw_relative_path values found in Cell 8 manifest."
    )


# -----------------------------------------------------------------------------
# Build deterministic phrase groups
# -----------------------------------------------------------------------------

grouped_jobs: Dict[Tuple[int, str], List[Dict[str, Any]]] = defaultdict(list)

for job in jobs:
    key = (int(job["batch_index"]), str(job["generation_text"]))
    grouped_jobs[key].append(job)

ordered_groups = []

for (batch_index, generation_text), group_jobs in grouped_jobs.items():
    group_jobs = sorted(group_jobs, key=lambda item: int(item["sample_index"]))
    phrase_hash = hashlib.sha256(generation_text.encode("utf-8")).hexdigest()[:16]

    ordered_groups.append(
        {
            "batch_index": batch_index,
            "generation_text": generation_text,
            "phrase_hash": phrase_hash,
            "jobs": group_jobs,
        }
    )

ordered_groups.sort(
    key=lambda item: (
        int(item["batch_index"]),
        min(int(job["sample_index"]) for job in item["jobs"]),
        str(item["generation_text"]),
    )
)

available_batch_indices = sorted({int(group["batch_index"]) for group in ordered_groups})

if MAX_BATCHES_THIS_RUN is not None:
    if MAX_BATCHES_THIS_RUN <= 0:
        raise ValueError("MAX_BATCHES_THIS_RUN must be None or greater than zero.")

    allowed_batches = set(available_batch_indices[:MAX_BATCHES_THIS_RUN])
    ordered_groups = [
        group for group in ordered_groups
        if int(group["batch_index"]) in allowed_batches
    ]


# -----------------------------------------------------------------------------
# Persistent O(1)-average completion index
# -----------------------------------------------------------------------------
#
# A SQLite hash index prevents a restart from requiring a full validation scan
# of every previously generated WAV. Atomic WAV commit happens first; the index
# row is written second. If interruption occurs between those operations, the
# existing WAV is discovered lazily when its group is visited.
# -----------------------------------------------------------------------------

DB_LOCK = threading.Lock()
DB_CONNECTION = sqlite3.connect(
    str(COMPLETION_DB_PATH),
    timeout=60.0,
    check_same_thread=False,
)
DB_CONNECTION.execute("PRAGMA journal_mode=WAL")
DB_CONNECTION.execute("PRAGMA synchronous=NORMAL")
DB_CONNECTION.execute("PRAGMA temp_store=MEMORY")
DB_CONNECTION.execute(
    """
    CREATE TABLE IF NOT EXISTS completed_samples (
        sample_id TEXT PRIMARY KEY,
        relative_path TEXT NOT NULL,
        size_bytes INTEGER NOT NULL,
        completed_at_utc TEXT NOT NULL
    )
    """
)
DB_CONNECTION.commit()


def load_indexed_completed_ids() -> set[str]:
    with DB_LOCK:
        rows = DB_CONNECTION.execute(
            "SELECT sample_id FROM completed_samples"
        ).fetchall()
    return {str(row[0]) for row in rows}


def index_completed_sample(job: Dict[str, Any], destination: Path) -> None:
    record = (
        str(job["sample_id"]),
        str(job["raw_relative_path"]),
        int(destination.stat().st_size),
        dt.datetime.now(dt.timezone.utc).isoformat(),
    )
    with DB_LOCK:
        DB_CONNECTION.execute(
            """
            INSERT INTO completed_samples(
                sample_id, relative_path, size_bytes, completed_at_utc
            ) VALUES (?, ?, ?, ?)
            ON CONFLICT(sample_id) DO UPDATE SET
                relative_path = excluded.relative_path,
                size_bytes = excluded.size_bytes,
                completed_at_utc = excluded.completed_at_utc
            """,
            record,
        )
        DB_CONNECTION.commit()


def remove_indexed_sample(sample_id: str) -> None:
    with DB_LOCK:
        DB_CONNECTION.execute(
            "DELETE FROM completed_samples WHERE sample_id = ?",
            (sample_id,),
        )
        DB_CONNECTION.commit()

# -----------------------------------------------------------------------------
# Generation helpers
# -----------------------------------------------------------------------------

def discover_generated_wavs(directory: Path) -> List[Path]:
    return sorted(
        path
        for path in directory.rglob("*.wav")
        if path.is_file()
    )


def clean_directory(directory: Path) -> None:
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True, exist_ok=True)


def target_path_for_job(job: Dict[str, Any]) -> Path:
    return POSITIVE_RAW_ROOT / str(job["raw_relative_path"])


COMPLETED_IDS: set[str] = load_indexed_completed_ids()
COMPLETED_LOCK = threading.Lock()


def completed_job(job: Dict[str, Any]) -> bool:
    sample_id = str(job["sample_id"])

    path = target_path_for_job(job)

    with COMPLETED_LOCK:
        indexed = sample_id in COMPLETED_IDS

    if indexed:
        # O(1) hash lookup plus one metadata check. Do not reopen WAV headers on
        # every restart for samples already committed and indexed.
        if path.exists() and path.stat().st_size > 44:
            return True
        with COMPLETED_LOCK:
            COMPLETED_IDS.discard(sample_id)
        remove_indexed_sample(sample_id)

    if not path.exists() or path.stat().st_size <= 44:
        return False

    try:
        inspect_wav_basic(path, include_sha256=False)
    except PositiveGenerationError:
        path.unlink(missing_ok=True)
        return False

    with COMPLETED_LOCK:
        COMPLETED_IDS.add(sample_id)
    index_completed_sample(job, path)

    return True


def atomic_commit_wav(source_path: Path, destination: Path) -> None:
    """
    Commit a generated WAV to persistent storage atomically.

    Copying to a .part file prevents a runtime interruption from leaving a
    destination filename that appears complete but contains partial data.
    """
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")

    temporary.unlink(missing_ok=True)

    with source_path.open("rb") as source, temporary.open("wb") as target:
        shutil.copyfileobj(source, target, length=4 * 1024 * 1024)
        target.flush()
        os.fsync(target.fileno())

    inspect_wav_basic(temporary, include_sha256=False)
    os.replace(temporary, destination)
    inspect_wav_basic(destination, include_sha256=False)


# Load completion state in O(k) from the compact SQLite index, where k is the
# number of indexed rows. Existing unindexed WAVs are discovered lazily only
# when their phrase group is processed.
print(f"Indexed reusable samples loaded: {len(COMPLETED_IDS):,}")
print(f"Completion index: {COMPLETION_DB_PATH}")

def write_progress(
    *,
    completed_groups: int,
    total_groups: int,
    generated_this_run: int,
    reused_samples: int,
    current_group: Optional[Dict[str, Any]],
) -> None:
    with COMPLETED_LOCK:
        existing_count = len(COMPLETED_IDS)

    progress = {
        "stage": "Cell 9 — Positive sample generation",
        "updated_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
        "total_manifest_jobs": len(jobs),
        "completed_manifest_jobs": existing_count,
        "remaining_manifest_jobs": len(jobs) - existing_count,
        "selected_group_count_this_run": total_groups,
        "completed_groups_this_run": completed_groups,
        "generated_samples_this_run": generated_this_run,
        "reused_samples_this_run": reused_samples,
        "current_group": (
            {
                "batch_index": current_group["batch_index"],
                "generation_text": current_group["generation_text"],
                "phrase_hash": current_group["phrase_hash"],
                "sample_count": len(current_group["jobs"]),
            }
            if current_group is not None
            else None
        ),
        "paths": {
            "raw_root": str(POSITIVE_RAW_ROOT),
            "progress": str(PROGRESS_PATH),
            "log": str(CELL_LOG_PATH),
        },
    }
    atomic_write_json(PROGRESS_PATH, progress)


# -----------------------------------------------------------------------------
# Generate missing samples in parallel
# -----------------------------------------------------------------------------

total_groups = len(ordered_groups)
generated_this_run = 0
reused_this_run = 0
completed_groups = 0
group_reports: List[Dict[str, Any]] = []


def generate_phrase_group(
    group_number: int,
    group: Dict[str, Any],
) -> Dict[str, Any]:
    """
    Generate one isolated phrase group.

    This function is safe to execute concurrently because every group has:
      - a unique staging directory;
      - unique deterministic destination filenames;
      - no shared progress/report writes.
    """
    batch_index = int(group["batch_index"])
    generation_text = str(group["generation_text"])
    phrase_hash = str(group["phrase_hash"])
    group_jobs = list(group["jobs"])

    missing_jobs = [job for job in group_jobs if not completed_job(job)]
    reused_count = len(group_jobs) - len(missing_jobs)
    group_started = time.monotonic()

    if not missing_jobs:
        return {
            "group_number": group_number,
            "batch_index": batch_index,
            "phrase_hash": phrase_hash,
            "generation_text": generation_text,
            "requested_count": len(group_jobs),
            "generated_count": 0,
            "reused_count": reused_count,
            "status": "already_complete",
            "duration_seconds": 0.0,
        }

    staging_directory = (
        STAGING_ROOT
        / f"batch_{batch_index:05d}"
        / f"phrase_{phrase_hash}"
    )
    clean_directory(staging_directory)

    requested_count = len(missing_jobs)

    command = [
        str(ENV_PYTHON),
        "-m",
        "piper_sample_generator",
        generation_text,
        "--model",
        str(PIPER_MODEL_PATH),
        "--max-samples",
        str(requested_count),
        "--batch-size",
        str(min(PIPER_INTERNAL_BATCH_SIZE, requested_count)),
        "--max-speakers",
        str(MAX_PIPER_SPEAKERS),
        "--output-dir",
        str(staging_directory),
    ]

    print(
        f"[START {group_number:>4}/{total_groups}] "
        f"batch={batch_index:05d} "
        f"missing={requested_count:>4} "
        f"text={generation_text!r}",
        flush=True,
    )

    try:
        run_command(
            command,
            stage_name=(
                f"Generate batch {batch_index:05d}, phrase {phrase_hash}, "
                f"{requested_count} samples"
            ),
            cwd=PIPER_GENERATOR_REPO,
            timeout_seconds=max(1800, requested_count * 120),
            print_output=False,
        )

        generated_wavs = discover_generated_wavs(staging_directory)

        if len(generated_wavs) != requested_count:
            raise PositiveGenerationError(
                "Piper produced an unexpected number of WAV files.\n"
                f"Batch: {batch_index}\n"
                f"Phrase: {generation_text!r}\n"
                f"Expected: {requested_count}\n"
                f"Found: {len(generated_wavs)}\n"
                f"Staging directory: {staging_directory}"
            )

        # Validate all staging outputs before moving any of them.
        for generated_path in generated_wavs:
            inspect_wav_basic(generated_path)

        # Deterministic assignment: sorted Piper output -> sorted manifest jobs.
        for source_path, job in zip(generated_wavs, missing_jobs):
            destination = target_path_for_job(job)
            destination.parent.mkdir(parents=True, exist_ok=True)

            # Destination filenames are unique per manifest job. An existing
            # invalid file may be safely replaced.
            if destination.exists():
                destination.unlink()

            atomic_commit_wav(source_path, destination)

            with COMPLETED_LOCK:
                COMPLETED_IDS.add(str(job["sample_id"]))
            index_completed_sample(job, destination)

        shutil.rmtree(staging_directory, ignore_errors=True)

    except Exception:
        # Keep the staging directory when a worker fails so diagnostics remain.
        raise

    duration_seconds = time.monotonic() - group_started

    return {
        "group_number": group_number,
        "batch_index": batch_index,
        "phrase_hash": phrase_hash,
        "generation_text": generation_text,
        "requested_count": len(group_jobs),
        "generated_count": len(missing_jobs),
        "reused_count": reused_count,
        "status": "generated",
        "duration_seconds": duration_seconds,
    }


print("=" * 96)
print("CELL 9 — PARALLEL, RESTART-SAFE POSITIVE GENERATION")
print("=" * 96)
print(f"Manifest jobs             : {len(jobs):,}")
print(f"Phrase groups this run    : {total_groups:,}")
print(f"Parallel Piper workers    : {MAX_PARALLEL_GROUPS}")
print(f"CPU threads per worker    : {THREADS_PER_PIPER_PROCESS}")
print(f"Piper internal batch size : {PIPER_INTERNAL_BATCH_SIZE}")
print(f"Maximum Piper speakers    : {MAX_PIPER_SPEAKERS}")
print(f"Persistent root           : {TRAINING_ROOT}")
print(f"Raw output root           : {POSITIVE_RAW_ROOT}")
print(f"Local staging root        : {STAGING_ROOT}")
print(f"Detected CPU cores        : {CPU_COUNT}")
print("=" * 96)

# ThreadPoolExecutor is intentional: each thread mainly waits for an independent
# Piper subprocess. The actual synthesis runs in separate OS processes.
# Bounded work queue: at most 2x worker-count futures are resident. This keeps
# scheduler memory O(workers), rather than O(all phrase groups).
MAX_IN_FLIGHT = max(MAX_PARALLEL_GROUPS, MAX_PARALLEL_GROUPS * 2)

with ThreadPoolExecutor(
    max_workers=MAX_PARALLEL_GROUPS,
    thread_name_prefix="pingo-piper",
) as executor:
    group_iterator = iter(enumerate(ordered_groups, start=1))
    in_flight: Dict[Any, Dict[str, Any]] = {}

    def submit_until_full() -> None:
        while len(in_flight) < MAX_IN_FLIGHT:
            try:
                group_number, next_group = next(group_iterator)
            except StopIteration:
                break
            future = executor.submit(
                generate_phrase_group,
                group_number,
                next_group,
            )
            in_flight[future] = next_group

    submit_until_full()

    try:
        while in_flight:
            done, _ = wait(
                tuple(in_flight),
                return_when=FIRST_COMPLETED,
            )

            for future in done:
                group = in_flight.pop(future)

                try:
                    report = future.result()
                except Exception as exc:
                    for pending_future in in_flight:
                        pending_future.cancel()

                    raise PositiveGenerationError(
                        "A parallel Piper worker failed.\n"
                        f"Batch: {group['batch_index']}\n"
                        f"Phrase: {group['generation_text']!r}\n"
                        f"Error: {type(exc).__name__}: {exc}\n"
                        f"Full log: {CELL_LOG_PATH}\n"
                        "Rerun Cell 9 after fixing the reported issue. "
                        "Already completed WAV files will be reused."
                    ) from exc

                group_reports.append(report)
                generated_this_run += int(report["generated_count"])
                reused_this_run += int(report["reused_count"])
                completed_groups += 1

                print(
                    f"[DONE  {report['group_number']:>4}/{total_groups}] "
                    f"batch={report['batch_index']:05d} "
                    f"generated={report['generated_count']:>4} "
                    f"reused={report['reused_count']:>4} "
                    f"time={report['duration_seconds']:.1f}s",
                    flush=True,
                )

                write_progress(
                    completed_groups=completed_groups,
                    total_groups=total_groups,
                    generated_this_run=generated_this_run,
                    reused_samples=reused_this_run,
                    current_group=group,
                )

                if (
                    completed_groups % PROGRESS_PRINT_INTERVAL == 0
                    or completed_groups == total_groups
                ):
                    with COMPLETED_LOCK:
                        total_existing = len(COMPLETED_IDS)
                    print(
                        f"  Progress: groups {completed_groups:,}/{total_groups:,}; "
                        f"dataset files {total_existing:,}/{len(jobs):,}",
                        flush=True,
                    )

            submit_until_full()

    except KeyboardInterrupt:
        for future in in_flight:
            future.cancel()
        print(
            "\nGeneration interrupted. Completed WAV files are preserved. "
            "Rerun Cell 9 to continue.",
            flush=True,
        )
        raise

# Preserve deterministic report ordering even though workers finish out of order.
group_reports.sort(key=lambda item: int(item["group_number"]))


# -----------------------------------------------------------------------------
# Final integrity verification
# -----------------------------------------------------------------------------

selected_job_ids = {
    str(job["sample_id"])
    for group in ordered_groups
    for job in group["jobs"]
}

selected_jobs = [
    job for job in jobs if str(job["sample_id"]) in selected_job_ids
]

missing_after_run = [
    str(target_path_for_job(job))
    for job in selected_jobs
    if not completed_job(job)
]

if missing_after_run:
    raise PositiveGenerationError(
        "Cell 9 completed its loop, but selected outputs are still missing:\n"
        + "\n".join(f"  - {path}" for path in missing_after_run[:100])
    )

all_existing_jobs = [job for job in jobs if completed_job(job)]
split_counts = Counter(str(job["split"]) for job in all_existing_jobs)
batch_counts = Counter(int(job["batch_index"]) for job in all_existing_jobs)

COMPLETED_AT = dt.datetime.now(dt.timezone.utc)

GENERATION_REPORT = {
    "stage": "Cell 9 — Positive sample generation",
    "passed": True,
    "started_at_utc": STARTED_AT.isoformat(),
    "completed_at_utc": COMPLETED_AT.isoformat(),
    "manifest_job_count": len(jobs),
    "existing_valid_sample_count": len(all_existing_jobs),
    "remaining_sample_count": len(jobs) - len(all_existing_jobs),
    "selected_group_count": total_groups,
    "completed_group_count": completed_groups,
    "generated_this_run": generated_this_run,
    "reused_this_run": reused_this_run,
    "complete_dataset": len(all_existing_jobs) == len(jobs),
    "split_counts_for_existing_samples": dict(sorted(split_counts.items())),
    "batch_counts_for_existing_samples": {
        str(key): value for key, value in sorted(batch_counts.items())
    },
    "group_reports": group_reports,
    "configuration": {
        "piper_internal_batch_size": PIPER_INTERNAL_BATCH_SIZE,
        "max_piper_speakers": MAX_PIPER_SPEAKERS,
        "max_batches_this_run": MAX_BATCHES_THIS_RUN,
        "max_parallel_groups": MAX_PARALLEL_GROUPS,
        "threads_per_piper_process": THREADS_PER_PIPER_PROCESS,
    },
    "paths": {
        "jobs_manifest": str(POSITIVE_JOBS_JSONL_PATH),
        "raw_root": str(POSITIVE_RAW_ROOT),
        "staging_root": str(STAGING_ROOT),
        "progress": str(PROGRESS_PATH),
        "report": str(GENERATION_REPORT_PATH),
        "log": str(CELL_LOG_PATH),
        "completion_index": str(COMPLETION_DB_PATH),
    },
}

atomic_write_json(GENERATION_REPORT_PATH, GENERATION_REPORT)
write_progress(
    completed_groups=completed_groups,
    total_groups=total_groups,
    generated_this_run=generated_this_run,
    reused_samples=reused_this_run,
    current_group=None,
)

append_log(
    "\nFINAL REPORT\n"
    + json.dumps(GENERATION_REPORT, indent=2, ensure_ascii=False)
)

print("\n" + "=" * 96)
print("CELL 9 COMPLETED SUCCESSFULLY")
print("=" * 96)
print(f"Manifest jobs             : {len(jobs):,}")
print(f"Valid raw samples now     : {len(all_existing_jobs):,}")
print(f"Generated this run        : {generated_this_run:,}")
print(f"Reused this run           : {reused_this_run:,}")
print(f"Remaining samples         : {len(jobs) - len(all_existing_jobs):,}")
print(f"Complete dataset          : {len(all_existing_jobs) == len(jobs)}")
print(f"Raw sample root           : {POSITIVE_RAW_ROOT}")
print(f"Generation report         : {GENERATION_REPORT_PATH}")
print(f"Progress file             : {PROGRESS_PATH}")
print(f"Full log                  : {CELL_LOG_PATH}")
print(f"Persistent storage        : {TRAINING_ROOT}")
print(f"Completion index          : {COMPLETION_DB_PATH}")
print(
    "\nNext: Cell 10 converts every generated sample to 16 kHz mono PCM16, "
    "validates audio quality, and creates train/validation/test indexes."
)
print("=" * 96)
# Flush and close persistent completion index.
with DB_LOCK:
    DB_CONNECTION.commit()
    DB_CONNECTION.close()

In [ ]:
# =============================================================================
# Cell 10 — Convert, validate, split, and index positive PINGO audio
# =============================================================================
#
# Prerequisites:
#   - Cell 8 created positive_jobs.jsonl.
#   - Cell 9 generated deterministic raw WAV files.
#
# This cell:
#   1. Converts raw Piper audio to 16 kHz mono PCM16.
#   2. Writes files into deterministic train/validation/test directories.
#   3. Validates duration, sample rate, channels, sample width, RMS, peak,
#      clipping, silence, and file readability.
#   4. Reuses valid existing converted files.
#   5. Writes JSONL and CSV indexes for downstream augmentation/training.
#   6. Produces a complete quality-control report.
#
# Run this first on the 100-sample smoke-test dataset.
# =============================================================================

from __future__ import annotations

import audioop
import csv
import datetime as dt
import hashlib
import json
import math
import os
import shutil
import subprocess
import textwrap
import wave
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence


# -----------------------------------------------------------------------------
# Configuration recovery
# -----------------------------------------------------------------------------

try:
    CONFIG
except NameError:
    class _FallbackConfig:
        TRAINING_ROOT = "/content/pingo_training"
        LOGS_DIR = "/content/pingo_training/logs"
        CONFIG_DIR = "/content/pingo_training/config"
        CACHE_DIR = "/content/pingo_training/cache"
        DATA_DIR = "/content/pingo_training/data"
        ENV_DIR = "/content/pingo-env"
        WAKE_WORD = "PINGO"

    CONFIG = _FallbackConfig()


TRAINING_ROOT = Path(CONFIG.TRAINING_ROOT)
LOGS_DIR = Path(CONFIG.LOGS_DIR)
CONFIG_DIR = Path(CONFIG.CONFIG_DIR)
CACHE_DIR = Path(CONFIG.CACHE_DIR)
DATA_DIR = Path(getattr(CONFIG, "DATA_DIR", TRAINING_ROOT / "data"))

POSITIVE_MANIFEST_DIR = DATA_DIR / "manifests" / "positive"
POSITIVE_JOBS_JSONL_PATH = POSITIVE_MANIFEST_DIR / "positive_jobs.jsonl"

POSITIVE_RAW_ROOT = DATA_DIR / "positive_raw"
POSITIVE_CONVERTED_ROOT = DATA_DIR / "positive_16khz"

INDEX_ROOT = DATA_DIR / "indexes"
TRAIN_INDEX_JSONL = INDEX_ROOT / "positive_train.jsonl"
VALIDATION_INDEX_JSONL = INDEX_ROOT / "positive_validation.jsonl"
TEST_INDEX_JSONL = INDEX_ROOT / "positive_test.jsonl"
ALL_INDEX_JSONL = INDEX_ROOT / "positive_all.jsonl"
ALL_INDEX_CSV = INDEX_ROOT / "positive_all.csv"

QUALITY_REPORT_PATH = CONFIG_DIR / "cell_10_audio_quality_report.json"
FAILURE_REPORT_PATH = CONFIG_DIR / "cell_10_audio_failures.json"
CELL_LOG_PATH = LOGS_DIR / "cell_10_audio_conversion_validation.log"

TARGET_SAMPLE_RATE = 16_000
TARGET_CHANNELS = 1
TARGET_SAMPLE_WIDTH_BYTES = 2

# Conservative wake-word sample checks.
MIN_DURATION_SECONDS = 0.20
MAX_DURATION_SECONDS = 8.00
MIN_RMS_NORMALIZED = 0.0005
MAX_CLIPPED_SAMPLE_FRACTION = 0.02
SILENCE_ABSOLUTE_THRESHOLD = 64
MAX_SILENCE_FRACTION = 0.995

# None means process every manifest job.
MAX_FILES_THIS_RUN: Optional[int] = None

for directory in (
    LOGS_DIR,
    CONFIG_DIR,
    CACHE_DIR,
    DATA_DIR,
    POSITIVE_CONVERTED_ROOT,
    INDEX_ROOT,
):
    directory.mkdir(parents=True, exist_ok=True)


# -----------------------------------------------------------------------------
# Errors and helpers
# -----------------------------------------------------------------------------

class AudioPreparationError(RuntimeError):
    """Raised when conversion or validation cannot safely continue."""


STARTED_AT = dt.datetime.now(dt.timezone.utc)

CELL_LOG_PATH.write_text(
    "PINGO CELL 10 AUDIO CONVERSION AND VALIDATION LOG\n"
    f"Started UTC: {STARTED_AT.isoformat()}\n"
    + "=" * 96
    + "\n",
    encoding="utf-8",
)


def append_log(message: str) -> None:
    with CELL_LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(message.rstrip() + "\n")


def atomic_write_json(path: Path, data: Any) -> None:
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(data, indent=2, ensure_ascii=False, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            block = handle.read(1024 * 1024)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def run_ffmpeg(source: Path, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".tmp.wav")
    temporary.unlink(missing_ok=True)

    command = [
        "ffmpeg",
        "-hide_banner",
        "-loglevel",
        "error",
        "-nostdin",
        "-y",
        "-i",
        str(source),
        "-map_metadata",
        "-1",
        "-ar",
        str(TARGET_SAMPLE_RATE),
        "-ac",
        str(TARGET_CHANNELS),
        "-c:a",
        "pcm_s16le",
        str(temporary),
    ]

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        check=False,
    )

    if result.returncode != 0:
        temporary.unlink(missing_ok=True)
        raise AudioPreparationError(
            "FFmpeg conversion failed.\n"
            f"Source: {source}\n"
            f"Destination: {destination}\n"
            f"Return code: {result.returncode}\n"
            f"Stderr:\n{result.stderr}"
        )

    if not temporary.exists() or temporary.stat().st_size <= 44:
        temporary.unlink(missing_ok=True)
        raise AudioPreparationError(
            f"FFmpeg did not create a valid output file: {temporary}"
        )

    temporary.replace(destination)


def inspect_pcm16_wav(path: Path) -> Dict[str, Any]:
    try:
        with wave.open(str(path), "rb") as wav_file:
            channels = wav_file.getnchannels()
            sample_width = wav_file.getsampwidth()
            sample_rate = wav_file.getframerate()
            frame_count = wav_file.getnframes()
            compression_type = wav_file.getcomptype()
            frames = wav_file.readframes(frame_count)
    except Exception as exc:
        raise AudioPreparationError(
            f"Unreadable WAV file: {path}\n"
            f"{type(exc).__name__}: {exc}"
        ) from exc

    duration_seconds = (
        frame_count / sample_rate if sample_rate > 0 else 0.0
    )

    errors: List[str] = []
    warnings: List[str] = []

    if channels != TARGET_CHANNELS:
        errors.append(f"channels={channels}, expected={TARGET_CHANNELS}")

    if sample_width != TARGET_SAMPLE_WIDTH_BYTES:
        errors.append(
            f"sample_width={sample_width}, expected={TARGET_SAMPLE_WIDTH_BYTES}"
        )

    if sample_rate != TARGET_SAMPLE_RATE:
        errors.append(
            f"sample_rate={sample_rate}, expected={TARGET_SAMPLE_RATE}"
        )

    if compression_type != "NONE":
        errors.append(f"compression_type={compression_type}, expected=NONE")

    if frame_count <= 0:
        errors.append("frame_count is zero")

    if duration_seconds < MIN_DURATION_SECONDS:
        errors.append(
            f"duration={duration_seconds:.4f}s below "
            f"{MIN_DURATION_SECONDS:.4f}s"
        )

    if duration_seconds > MAX_DURATION_SECONDS:
        warnings.append(
            f"duration={duration_seconds:.4f}s above "
            f"{MAX_DURATION_SECONDS:.4f}s"
        )

    if sample_width == 2 and frames:
        rms_integer = audioop.rms(frames, 2)
        peak_integer = audioop.max(frames, 2)
        rms_normalized = rms_integer / 32768.0
        peak_normalized = peak_integer / 32768.0

        total_samples = len(frames) // 2
        clipped_samples = 0
        silent_samples = 0

        # Efficient enough for smoke tests and 50k short clips.
        for byte_index in range(0, len(frames) - 1, 2):
            sample = int.from_bytes(
                frames[byte_index:byte_index + 2],
                byteorder="little",
                signed=True,
            )

            if abs(sample) >= 32760:
                clipped_samples += 1

            if abs(sample) <= SILENCE_ABSOLUTE_THRESHOLD:
                silent_samples += 1

        clipped_fraction = (
            clipped_samples / total_samples if total_samples else 0.0
        )
        silence_fraction = (
            silent_samples / total_samples if total_samples else 1.0
        )
    else:
        rms_integer = 0
        peak_integer = 0
        rms_normalized = 0.0
        peak_normalized = 0.0
        clipped_fraction = 0.0
        silence_fraction = 1.0
        total_samples = 0

    if rms_normalized < MIN_RMS_NORMALIZED:
        errors.append(
            f"rms={rms_normalized:.8f} below "
            f"{MIN_RMS_NORMALIZED:.8f}"
        )

    if clipped_fraction > MAX_CLIPPED_SAMPLE_FRACTION:
        warnings.append(
            f"clipped_fraction={clipped_fraction:.6f} above "
            f"{MAX_CLIPPED_SAMPLE_FRACTION:.6f}"
        )

    if silence_fraction > MAX_SILENCE_FRACTION:
        errors.append(
            f"silence_fraction={silence_fraction:.6f} above "
            f"{MAX_SILENCE_FRACTION:.6f}"
        )

    return {
        "path": str(path),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
        "channels": channels,
        "sample_width_bytes": sample_width,
        "sample_rate": sample_rate,
        "compression_type": compression_type,
        "frame_count": frame_count,
        "sample_count": total_samples,
        "duration_seconds": duration_seconds,
        "rms_integer": rms_integer,
        "rms_normalized": rms_normalized,
        "peak_integer": peak_integer,
        "peak_normalized": peak_normalized,
        "clipped_fraction": clipped_fraction,
        "silence_fraction": silence_fraction,
        "passed": not errors,
        "errors": errors,
        "warnings": warnings,
    }


def converted_path_for_job(job: Dict[str, Any]) -> Path:
    return POSITIVE_CONVERTED_ROOT / str(job["converted_relative_path"])


def raw_path_for_job(job: Dict[str, Any]) -> Path:
    return POSITIVE_RAW_ROOT / str(job["raw_relative_path"])


# -----------------------------------------------------------------------------
# Preconditions and manifest loading
# -----------------------------------------------------------------------------

if shutil.which("ffmpeg") is None:
    raise AudioPreparationError(
        "ffmpeg was not found. Run the dependency installation cell first."
    )

if not POSITIVE_JOBS_JSONL_PATH.exists():
    raise AudioPreparationError(
        f"Missing positive job manifest: {POSITIVE_JOBS_JSONL_PATH}\n"
        "Run Cell 8 first."
    )

jobs: List[Dict[str, Any]] = []

with POSITIVE_JOBS_JSONL_PATH.open("r", encoding="utf-8") as handle:
    for line_number, line in enumerate(handle, start=1):
        stripped = line.strip()
        if not stripped:
            continue
        try:
            jobs.append(json.loads(stripped))
        except json.JSONDecodeError as exc:
            raise AudioPreparationError(
                f"Invalid JSON on manifest line {line_number}"
            ) from exc

if not jobs:
    raise AudioPreparationError("Positive manifest contains no jobs.")

required_fields = {
    "sample_id",
    "generation_text",
    "pronunciation_id",
    "context_id",
    "context_category",
    "batch_index",
    "split",
    "raw_relative_path",
    "converted_relative_path",
}

for index, job in enumerate(jobs):
    missing = required_fields - set(job)
    if missing:
        raise AudioPreparationError(
            f"Manifest job {index} is missing: {sorted(missing)}"
        )

valid_splits = {"train", "validation", "test"}
unexpected_splits = sorted(
    {str(job["split"]) for job in jobs} - valid_splits
)

if unexpected_splits:
    raise AudioPreparationError(
        f"Unexpected manifest splits: {unexpected_splits}"
    )

if MAX_FILES_THIS_RUN is not None:
    if MAX_FILES_THIS_RUN <= 0:
        raise ValueError("MAX_FILES_THIS_RUN must be None or greater than zero.")
    selected_jobs = jobs[:MAX_FILES_THIS_RUN]
else:
    selected_jobs = jobs


# -----------------------------------------------------------------------------
# Convert and validate
# -----------------------------------------------------------------------------

print("=" * 96)
print("CELL 10 — AUDIO CONVERSION, VALIDATION, AND INDEXING")
print("=" * 96)
print(f"Manifest jobs        : {len(jobs):,}")
print(f"Files selected       : {len(selected_jobs):,}")
print(f"Raw root             : {POSITIVE_RAW_ROOT}")
print(f"Converted root       : {POSITIVE_CONVERTED_ROOT}")
print(f"Target format        : {TARGET_SAMPLE_RATE} Hz, mono, PCM16")
print("=" * 96)

records: List[Dict[str, Any]] = []
failures: List[Dict[str, Any]] = []
converted_count = 0
reused_count = 0

for number, job in enumerate(selected_jobs, start=1):
    raw_path = raw_path_for_job(job)
    converted_path = converted_path_for_job(job)

    if not raw_path.exists():
        failures.append(
            {
                "sample_id": job["sample_id"],
                "stage": "missing_raw",
                "raw_path": str(raw_path),
                "error": "Raw WAV does not exist. Run Cell 9.",
            }
        )
        continue

    use_existing = False

    if converted_path.exists() and converted_path.stat().st_size > 44:
        try:
            existing_report = inspect_pcm16_wav(converted_path)
            if existing_report["passed"]:
                use_existing = True
                audio_report = existing_report
                reused_count += 1
        except Exception:
            converted_path.unlink(missing_ok=True)

    if not use_existing:
        try:
            run_ffmpeg(raw_path, converted_path)
            audio_report = inspect_pcm16_wav(converted_path)
            converted_count += 1
        except Exception as exc:
            converted_path.unlink(missing_ok=True)
            failures.append(
                {
                    "sample_id": job["sample_id"],
                    "stage": "conversion_or_validation",
                    "raw_path": str(raw_path),
                    "converted_path": str(converted_path),
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )
            continue

    if not audio_report["passed"]:
        failures.append(
            {
                "sample_id": job["sample_id"],
                "stage": "quality_validation",
                "raw_path": str(raw_path),
                "converted_path": str(converted_path),
                "errors": audio_report["errors"],
                "warnings": audio_report["warnings"],
                "audio": audio_report,
            }
        )
        continue

    record = {
        "sample_id": job["sample_id"],
        "label": "positive",
        "wake_word": str(getattr(CONFIG, "WAKE_WORD", "PINGO")),
        "split": job["split"],
        "batch_index": int(job["batch_index"]),
        "generation_text": job["generation_text"],
        "pronunciation_id": job["pronunciation_id"],
        "pronunciation_text": job.get("pronunciation_text"),
        "expected_pronunciation": job.get("expected_pronunciation"),
        "context_id": job["context_id"],
        "context_category": job["context_category"],
        "raw_path": str(raw_path),
        "audio_path": str(converted_path),
        "audio_relative_path": str(
            converted_path.relative_to(POSITIVE_CONVERTED_ROOT)
        ),
        "sample_rate": audio_report["sample_rate"],
        "channels": audio_report["channels"],
        "sample_width_bytes": audio_report["sample_width_bytes"],
        "duration_seconds": audio_report["duration_seconds"],
        "rms_normalized": audio_report["rms_normalized"],
        "peak_normalized": audio_report["peak_normalized"],
        "clipped_fraction": audio_report["clipped_fraction"],
        "silence_fraction": audio_report["silence_fraction"],
        "size_bytes": audio_report["size_bytes"],
        "sha256": audio_report["sha256"],
        "warnings": audio_report["warnings"],
    }
    records.append(record)

    if number % 500 == 0 or number == len(selected_jobs):
        print(
            f"Processed {number:,}/{len(selected_jobs):,} | "
            f"valid={len(records):,} | failures={len(failures):,}"
        )


# -----------------------------------------------------------------------------
# Validate index uniqueness and split isolation
# -----------------------------------------------------------------------------

sample_ids = [record["sample_id"] for record in records]
audio_paths = [record["audio_path"] for record in records]

if len(sample_ids) != len(set(sample_ids)):
    raise AudioPreparationError("Duplicate sample IDs detected after conversion.")

if len(audio_paths) != len(set(audio_paths)):
    raise AudioPreparationError("Duplicate converted audio paths detected.")

split_membership: Dict[str, set[str]] = defaultdict(set)

for record in records:
    split_membership[record["split"]].add(record["sample_id"])

if split_membership["train"] & split_membership["validation"]:
    raise AudioPreparationError("Train and validation splits overlap.")

if split_membership["train"] & split_membership["test"]:
    raise AudioPreparationError("Train and test splits overlap.")

if split_membership["validation"] & split_membership["test"]:
    raise AudioPreparationError("Validation and test splits overlap.")


# -----------------------------------------------------------------------------
# Write downstream indexes
# -----------------------------------------------------------------------------

records.sort(key=lambda item: item["sample_id"])

split_paths = {
    "train": TRAIN_INDEX_JSONL,
    "validation": VALIDATION_INDEX_JSONL,
    "test": TEST_INDEX_JSONL,
}

for split, output_path in split_paths.items():
    split_records = [
        record for record in records if record["split"] == split
    ]
    with output_path.open("w", encoding="utf-8") as handle:
        for record in split_records:
            handle.write(
                json.dumps(record, ensure_ascii=False, sort_keys=True) + "\n"
            )

with ALL_INDEX_JSONL.open("w", encoding="utf-8") as handle:
    for record in records:
        handle.write(
            json.dumps(record, ensure_ascii=False, sort_keys=True) + "\n"
        )

csv_fields = [
    "sample_id",
    "label",
    "wake_word",
    "split",
    "batch_index",
    "generation_text",
    "pronunciation_id",
    "pronunciation_text",
    "expected_pronunciation",
    "context_id",
    "context_category",
    "raw_path",
    "audio_path",
    "audio_relative_path",
    "sample_rate",
    "channels",
    "sample_width_bytes",
    "duration_seconds",
    "rms_normalized",
    "peak_normalized",
    "clipped_fraction",
    "silence_fraction",
    "size_bytes",
    "sha256",
    "warnings",
]

with ALL_INDEX_CSV.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=csv_fields)
    writer.writeheader()
    for record in records:
        csv_record = dict(record)
        csv_record["warnings"] = json.dumps(
            csv_record.get("warnings", []),
            ensure_ascii=False,
        )
        writer.writerow(csv_record)


# -----------------------------------------------------------------------------
# Quality summary
# -----------------------------------------------------------------------------

def numeric_summary(values: List[float]) -> Dict[str, Optional[float]]:
    if not values:
        return {
            "minimum": None,
            "maximum": None,
            "mean": None,
        }

    return {
        "minimum": min(values),
        "maximum": max(values),
        "mean": sum(values) / len(values),
    }


split_counts = Counter(record["split"] for record in records)
pronunciation_counts = Counter(
    record["pronunciation_id"] for record in records
)
context_counts = Counter(record["context_id"] for record in records)
warning_counts = Counter(
    warning
    for record in records
    for warning in record.get("warnings", [])
)

durations = [float(record["duration_seconds"]) for record in records]
rms_values = [float(record["rms_normalized"]) for record in records]
peak_values = [float(record["peak_normalized"]) for record in records]
silence_values = [float(record["silence_fraction"]) for record in records]
clipped_values = [float(record["clipped_fraction"]) for record in records]

COMPLETED_AT = dt.datetime.now(dt.timezone.utc)

QUALITY_REPORT = {
    "stage": "Cell 10 — Audio conversion and validation",
    "passed": len(failures) == 0 and len(records) == len(selected_jobs),
    "started_at_utc": STARTED_AT.isoformat(),
    "completed_at_utc": COMPLETED_AT.isoformat(),
    "manifest_job_count": len(jobs),
    "selected_job_count": len(selected_jobs),
    "valid_record_count": len(records),
    "failure_count": len(failures),
    "converted_this_run": converted_count,
    "reused_this_run": reused_count,
    "target_audio": {
        "sample_rate": TARGET_SAMPLE_RATE,
        "channels": TARGET_CHANNELS,
        "sample_width_bytes": TARGET_SAMPLE_WIDTH_BYTES,
        "encoding": "PCM16 WAV",
    },
    "quality_thresholds": {
        "minimum_duration_seconds": MIN_DURATION_SECONDS,
        "maximum_duration_seconds": MAX_DURATION_SECONDS,
        "minimum_rms_normalized": MIN_RMS_NORMALIZED,
        "maximum_clipped_sample_fraction": MAX_CLIPPED_SAMPLE_FRACTION,
        "silence_absolute_threshold": SILENCE_ABSOLUTE_THRESHOLD,
        "maximum_silence_fraction": MAX_SILENCE_FRACTION,
    },
    "split_counts": dict(sorted(split_counts.items())),
    "pronunciation_counts": dict(sorted(pronunciation_counts.items())),
    "context_counts": dict(sorted(context_counts.items())),
    "warning_counts": dict(sorted(warning_counts.items())),
    "statistics": {
        "duration_seconds": numeric_summary(durations),
        "rms_normalized": numeric_summary(rms_values),
        "peak_normalized": numeric_summary(peak_values),
        "silence_fraction": numeric_summary(silence_values),
        "clipped_fraction": numeric_summary(clipped_values),
    },
    "paths": {
        "converted_root": str(POSITIVE_CONVERTED_ROOT),
        "all_index_jsonl": str(ALL_INDEX_JSONL),
        "all_index_csv": str(ALL_INDEX_CSV),
        "train_index": str(TRAIN_INDEX_JSONL),
        "validation_index": str(VALIDATION_INDEX_JSONL),
        "test_index": str(TEST_INDEX_JSONL),
        "quality_report": str(QUALITY_REPORT_PATH),
        "failure_report": str(FAILURE_REPORT_PATH),
        "log": str(CELL_LOG_PATH),
    },
}

atomic_write_json(QUALITY_REPORT_PATH, QUALITY_REPORT)
atomic_write_json(FAILURE_REPORT_PATH, failures)

append_log(
    "\nQUALITY REPORT\n"
    + json.dumps(QUALITY_REPORT, indent=2, ensure_ascii=False)
)
append_log(
    "\nFAILURES\n"
    + json.dumps(failures, indent=2, ensure_ascii=False)
)


# -----------------------------------------------------------------------------
# Final result
# -----------------------------------------------------------------------------

print("\n" + "=" * 96)

if failures:
    print("CELL 10 COMPLETED WITH AUDIO FAILURES")
else:
    print("CELL 10 COMPLETED SUCCESSFULLY")

print("=" * 96)
print(f"Manifest jobs             : {len(jobs):,}")
print(f"Selected jobs             : {len(selected_jobs):,}")
print(f"Valid converted samples   : {len(records):,}")
print(f"Converted this run        : {converted_count:,}")
print(f"Reused this run           : {reused_count:,}")
print(f"Failures                  : {len(failures):,}")
print("\nDataset splits")
print(f"  Train                   : {split_counts['train']:,}")
print(f"  Validation              : {split_counts['validation']:,}")
print(f"  Test                    : {split_counts['test']:,}")
print("\nOutput indexes")
print(f"  All JSONL               : {ALL_INDEX_JSONL}")
print(f"  All CSV                 : {ALL_INDEX_CSV}")
print(f"  Train JSONL             : {TRAIN_INDEX_JSONL}")
print(f"  Validation JSONL        : {VALIDATION_INDEX_JSONL}")
print(f"  Test JSONL              : {TEST_INDEX_JSONL}")
print(f"Quality report            : {QUALITY_REPORT_PATH}")
print(f"Failure report            : {FAILURE_REPORT_PATH}")
print(f"Full log                  : {CELL_LOG_PATH}")

if failures:
    print(
        "\nReview the failure report before continuing. "
        "Do not train on missing, silent, unreadable, or invalid audio."
    )
    print("First failures:")
    for failure in failures[:10]:
        print(
            f"  - {failure.get('sample_id')}: "
            f"{failure.get('stage')} — {failure.get('error', failure.get('errors'))}"
        )
else:
    print(
        "\nNext: inspect a random sample from every pronunciation and context, "
        "then build negative/background-noise and augmentation datasets."
    )

print("=" * 96)

if failures:
    raise AudioPreparationError(
        f"Cell 10 found {len(failures)} invalid or missing samples. "
        f"See {FAILURE_REPORT_PATH}"
    )

In [ ]:
from __future__ import annotations
import csv, datetime as dt, hashlib, inspect, json, math, os, random, shutil, subprocess, sys, textwrap, wave
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

try:
    CONFIG
except NameError:
    class _FallbackConfig:
        TRAINING_ROOT="/content/pingo_training"
        LOGS_DIR="/content/pingo_training/logs"
        CONFIG_DIR="/content/pingo_training/config"
        CACHE_DIR="/content/pingo_training/cache"
        DATA_DIR="/content/pingo_training/data"
        MODELS_DIR="/content/pingo_training/models"
        REPOS_DIR="/content/pingo_training/repos"
        ENV_DIR="/content/pingo-env"
        WAKE_WORD="PINGO"
        RANDOM_SEED=42
    CONFIG=_FallbackConfig()

ROOT=Path(CONFIG.TRAINING_ROOT)
LOGS=Path(CONFIG.LOGS_DIR)
CFG=Path(CONFIG.CONFIG_DIR)
CACHE=Path(CONFIG.CACHE_DIR)
DATA=Path(getattr(CONFIG,"DATA_DIR",ROOT/"data"))
MODELS=Path(getattr(CONFIG,"MODELS_DIR",ROOT/"models"))
REPOS=Path(getattr(CONFIG,"REPOS_DIR",ROOT/"repos"))
ENV=Path(CONFIG.ENV_DIR)
PY=ENV/"bin"/"python"
OWW_REPO=REPOS/"openWakeWord"
WAKE=str(getattr(CONFIG,"WAKE_WORD","PINGO")).strip()
SEED=int(getattr(CONFIG,"RANDOM_SEED",42))
for p in (LOGS,CFG,CACHE,DATA,MODELS): p.mkdir(parents=True,exist_ok=True)

def sha256(path: Path)->str:
    h=hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

def write_json(path: Path,obj: Any)->None:
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,sort_keys=True)+"\n",encoding="utf-8")
    tmp.replace(path)

def read_jsonl(path: Path)->List[Dict[str,Any]]:
    out=[]
    with path.open(encoding="utf-8") as f:
        for n,line in enumerate(f,1):
            if line.strip():
                try: out.append(json.loads(line))
                except Exception as e: raise RuntimeError(f"Invalid JSONL {path}:{n}: {e}")
    return out

def run(cmd: Sequence[str], *, cwd: Optional[Path]=None, timeout: int=3600, env: Optional[Dict[str,str]]=None)->subprocess.CompletedProcess:
    e=os.environ.copy()
    paths=[str(OWW_REPO.resolve())] if OWW_REPO.exists() else []
    if e.get("PYTHONPATH"): paths.append(e["PYTHONPATH"])
    if paths: e["PYTHONPATH"]=os.pathsep.join(paths)
    if env: e.update({str(k):str(v) for k,v in env.items()})
    print("$"," ".join(map(str,cmd)))
    r=subprocess.run(list(map(str,cmd)),cwd=str((cwd or ROOT).resolve()),env=e,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE,timeout=timeout)
    if r.stdout.strip(): print(r.stdout[-12000:])
    if r.stderr.strip(): print("[stderr]\n"+r.stderr[-12000:])
    if r.returncode: raise RuntimeError(f"Command failed ({r.returncode}): {' '.join(map(str,cmd))}")
    return r

# Cell 11 — Positive-audio inspection sampler
INDEX=DATA/"indexes"/"positive_all.jsonl"
OUT=ROOT/"reports"/"cell_11_positive_inspection.json"
PLAYLIST=ROOT/"reports"/"cell_11_listen_playlist.txt"
if not INDEX.exists(): raise FileNotFoundError(f"{INDEX}; run Cell 10")
rows=read_jsonl(INDEX)
rng=random.Random(SEED)
groups=defaultdict(list)
for r in rows: groups[(r["pronunciation_id"],r["context_category"])].append(r)
selected=[]
for key,items in sorted(groups.items()):
    selected.extend(rng.sample(items,min(3,len(items))))
selected={r["sample_id"]:r for r in selected}.values()
report={"stage":"Cell 11","count":len(selected),"samples":list(selected)}
write_json(OUT,report)
PLAYLIST.parent.mkdir(parents=True,exist_ok=True)
PLAYLIST.write_text("\\n".join(r["audio_path"] for r in report["samples"])+"\\n")
print(f"Selected {len(selected)} clips. Listen to every file listed in {PLAYLIST}")
print(f"Report: {OUT}")

Mounted at /content/drive


In [ ]:
from __future__ import annotations
import csv, datetime as dt, hashlib, inspect, json, math, os, random, shutil, subprocess, sys, textwrap, wave
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

try:
    CONFIG
except NameError:
    class _FallbackConfig:
        TRAINING_ROOT="/content/pingo_training"
        LOGS_DIR="/content/pingo_training/logs"
        CONFIG_DIR="/content/pingo_training/config"
        CACHE_DIR="/content/pingo_training/cache"
        DATA_DIR="/content/pingo_training/data"
        MODELS_DIR="/content/pingo_training/models"
        REPOS_DIR="/content/pingo_training/repos"
        ENV_DIR="/content/pingo-env"
        WAKE_WORD="PINGO"
        RANDOM_SEED=42
    CONFIG=_FallbackConfig()

ROOT=Path(CONFIG.TRAINING_ROOT)
LOGS=Path(CONFIG.LOGS_DIR)
CFG=Path(CONFIG.CONFIG_DIR)
CACHE=Path(CONFIG.CACHE_DIR)
DATA=Path(getattr(CONFIG,"DATA_DIR",ROOT/"data"))
MODELS=Path(getattr(CONFIG,"MODELS_DIR",ROOT/"models"))
REPOS=Path(getattr(CONFIG,"REPOS_DIR",ROOT/"repos"))
ENV=Path(CONFIG.ENV_DIR)
PY=ENV/"bin"/"python"
OWW_REPO=REPOS/"openWakeWord"
WAKE=str(getattr(CONFIG,"WAKE_WORD","PINGO")).strip()
SEED=int(getattr(CONFIG,"RANDOM_SEED",42))
for p in (LOGS,CFG,CACHE,DATA,MODELS): p.mkdir(parents=True,exist_ok=True)

def sha256(path: Path)->str:
    h=hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

def write_json(path: Path,obj: Any)->None:
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,sort_keys=True)+"\n",encoding="utf-8")
    tmp.replace(path)

def read_jsonl(path: Path)->List[Dict[str,Any]]:
    out=[]
    with path.open(encoding="utf-8") as f:
        for n,line in enumerate(f,1):
            if line.strip():
                try: out.append(json.loads(line))
                except Exception as e: raise RuntimeError(f"Invalid JSONL {path}:{n}: {e}")
    return out

def run(cmd: Sequence[str], *, cwd: Optional[Path]=None, timeout: int=3600, env: Optional[Dict[str,str]]=None)->subprocess.CompletedProcess:
    e=os.environ.copy()
    paths=[str(OWW_REPO.resolve())] if OWW_REPO.exists() else []
    if e.get("PYTHONPATH"): paths.append(e["PYTHONPATH"])
    if paths: e["PYTHONPATH"]=os.pathsep.join(paths)
    if env: e.update({str(k):str(v) for k,v in env.items()})
    print("$"," ".join(map(str,cmd)))
    r=subprocess.run(list(map(str,cmd)),cwd=str((cwd or ROOT).resolve()),env=e,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE,timeout=timeout)
    if r.stdout.strip(): print(r.stdout[-12000:])
    if r.stderr.strip(): print("[stderr]\n"+r.stderr[-12000:])
    if r.returncode: raise RuntimeError(f"Command failed ({r.returncode}): {' '.join(map(str,cmd))}")
    return r

# Cell 12 — Hard-negative text generation manifest
SRC=DATA/"manifests"/"negative"/"confusions.json"
OUT=DATA/"manifests"/"negative"/"negative_generation_jobs.jsonl"
CFG_OUT=CFG/"cell_12_negative_generation_config.json"
if not SRC.exists(): raise FileNotFoundError(f"{SRC}; run Cell 8")
d=json.loads(SRC.read_text())
phrases=sorted(set(d["confusion_words"]+d["hard_negative_phrases"]))
TARGET_PER_PHRASE=int(getattr(CONFIG,"NEGATIVE_SAMPLES_PER_PHRASE",40))
jobs=[]
for pi,text in enumerate(phrases):
    for i in range(TARGET_PER_PHRASE):
        sid=f"pingo_hard_negative_{pi:04d}_{i:05d}"
        split=["train","validation","test"][int(hashlib.sha256(sid.encode()).hexdigest()[:8],16)%10//8 if False else 0]
        v=int(hashlib.sha256(sid.encode()).hexdigest()[:8],16)/0xffffffff
        split="train" if v<.8 else ("validation" if v<.9 else "test")
        jobs.append({"sample_id":sid,"label":"negative","text":text,"split":split,
                     "raw_relative_path":f"hard_negative_raw/{pi:04d}/{sid}.wav",
                     "converted_relative_path":f"hard_negative_16khz/{split}/{pi:04d}/{sid}.wav"})
OUT.parent.mkdir(parents=True,exist_ok=True)
with OUT.open("w",encoding="utf-8") as f:
    for j in jobs: f.write(json.dumps(j,sort_keys=True)+"\\n")
write_json(CFG_OUT,{"phrase_count":len(phrases),"samples_per_phrase":TARGET_PER_PHRASE,"job_count":len(jobs),"manifest":str(OUT)})
print(f"Created {len(jobs):,} hard-negative generation jobs: {OUT}")

In [ ]:
from __future__ import annotations
import csv, datetime as dt, hashlib, inspect, json, math, os, random, shutil, subprocess, sys, textwrap, wave
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

try:
    CONFIG
except NameError:
    class _FallbackConfig:
        TRAINING_ROOT="/content/pingo_training"
        LOGS_DIR="/content/pingo_training/logs"
        CONFIG_DIR="/content/pingo_training/config"
        CACHE_DIR="/content/pingo_training/cache"
        DATA_DIR="/content/pingo_training/data"
        MODELS_DIR="/content/pingo_training/models"
        REPOS_DIR="/content/pingo_training/repos"
        ENV_DIR="/content/pingo-env"
        WAKE_WORD="PINGO"
        RANDOM_SEED=42
    CONFIG=_FallbackConfig()

ROOT=Path(CONFIG.TRAINING_ROOT)
LOGS=Path(CONFIG.LOGS_DIR)
CFG=Path(CONFIG.CONFIG_DIR)
CACHE=Path(CONFIG.CACHE_DIR)
DATA=Path(getattr(CONFIG,"DATA_DIR",ROOT/"data"))
MODELS=Path(getattr(CONFIG,"MODELS_DIR",ROOT/"models"))
REPOS=Path(getattr(CONFIG,"REPOS_DIR",ROOT/"repos"))
ENV=Path(CONFIG.ENV_DIR)
PY=ENV/"bin"/"python"
OWW_REPO=REPOS/"openWakeWord"
WAKE=str(getattr(CONFIG,"WAKE_WORD","PINGO")).strip()
SEED=int(getattr(CONFIG,"RANDOM_SEED",42))
for p in (LOGS,CFG,CACHE,DATA,MODELS): p.mkdir(parents=True,exist_ok=True)

def sha256(path: Path)->str:
    h=hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

def write_json(path: Path,obj: Any)->None:
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,sort_keys=True)+"\n",encoding="utf-8")
    tmp.replace(path)

def read_jsonl(path: Path)->List[Dict[str,Any]]:
    out=[]
    with path.open(encoding="utf-8") as f:
        for n,line in enumerate(f,1):
            if line.strip():
                try: out.append(json.loads(line))
                except Exception as e: raise RuntimeError(f"Invalid JSONL {path}:{n}: {e}")
    return out

def run(cmd: Sequence[str], *, cwd: Optional[Path]=None, timeout: int=3600, env: Optional[Dict[str,str]]=None)->subprocess.CompletedProcess:
    e=os.environ.copy()
    paths=[str(OWW_REPO.resolve())] if OWW_REPO.exists() else []
    if e.get("PYTHONPATH"): paths.append(e["PYTHONPATH"])
    if paths: e["PYTHONPATH"]=os.pathsep.join(paths)
    if env: e.update({str(k):str(v) for k,v in env.items()})
    print("$"," ".join(map(str,cmd)))
    r=subprocess.run(list(map(str,cmd)),cwd=str((cwd or ROOT).resolve()),env=e,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE,timeout=timeout)
    if r.stdout.strip(): print(r.stdout[-12000:])
    if r.stderr.strip(): print("[stderr]\n"+r.stderr[-12000:])
    if r.returncode: raise RuntimeError(f"Command failed ({r.returncode}): {' '.join(map(str,cmd))}")
    return r

# Cell 13 — Download official sample noise/music datasets
import urllib.request, zipfile
ASSET_DIR=DATA/"external_negative"
ASSET_DIR.mkdir(parents=True,exist_ok=True)
assets={
 "fma_sample.zip":"https://f002.backblazeb2.com/file/openwakeword-resources/data/fma_sample.zip",
 "fsd50k_sample.zip":"https://f002.backblazeb2.com/file/openwakeword-resources/data/fsd50k_sample.zip",
}
reports=[]
for name,url in assets.items():
    dst=ASSET_DIR/name
    if not dst.exists():
        print("Downloading",url)
        urllib.request.urlretrieve(url,dst)
    if dst.stat().st_size<100000: raise RuntimeError(f"Unexpectedly small download: {dst}")
    extract=ASSET_DIR/name.removesuffix(".zip")
    if not extract.exists():
        with zipfile.ZipFile(dst) as z: z.extractall(extract)
    reports.append({"url":url,"archive":str(dst),"sha256":sha256(dst),"extract":str(extract)})
write_json(CFG/"cell_13_external_negative_assets.json",{"assets":reports,"license_warning":"Verify every source license before commercial use."})
print("Downloaded/extracted official OpenWakeWord sample negative datasets.")

In [ ]:
from __future__ import annotations
import csv, datetime as dt, hashlib, inspect, json, math, os, random, shutil, subprocess, sys, textwrap, wave
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

try:
    CONFIG
except NameError:
    class _FallbackConfig:
        TRAINING_ROOT="/content/pingo_training"
        LOGS_DIR="/content/pingo_training/logs"
        CONFIG_DIR="/content/pingo_training/config"
        CACHE_DIR="/content/pingo_training/cache"
        DATA_DIR="/content/pingo_training/data"
        MODELS_DIR="/content/pingo_training/models"
        REPOS_DIR="/content/pingo_training/repos"
        ENV_DIR="/content/pingo-env"
        WAKE_WORD="PINGO"
        RANDOM_SEED=42
    CONFIG=_FallbackConfig()

ROOT=Path(CONFIG.TRAINING_ROOT)
LOGS=Path(CONFIG.LOGS_DIR)
CFG=Path(CONFIG.CONFIG_DIR)
CACHE=Path(CONFIG.CACHE_DIR)
DATA=Path(getattr(CONFIG,"DATA_DIR",ROOT/"data"))
MODELS=Path(getattr(CONFIG,"MODELS_DIR",ROOT/"models"))
REPOS=Path(getattr(CONFIG,"REPOS_DIR",ROOT/"repos"))
ENV=Path(CONFIG.ENV_DIR)
PY=ENV/"bin"/"python"
OWW_REPO=REPOS/"openWakeWord"
WAKE=str(getattr(CONFIG,"WAKE_WORD","PINGO")).strip()
SEED=int(getattr(CONFIG,"RANDOM_SEED",42))
for p in (LOGS,CFG,CACHE,DATA,MODELS): p.mkdir(parents=True,exist_ok=True)

def sha256(path: Path)->str:
    h=hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

def write_json(path: Path,obj: Any)->None:
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,sort_keys=True)+"\n",encoding="utf-8")
    tmp.replace(path)

def read_jsonl(path: Path)->List[Dict[str,Any]]:
    out=[]
    with path.open(encoding="utf-8") as f:
        for n,line in enumerate(f,1):
            if line.strip():
                try: out.append(json.loads(line))
                except Exception as e: raise RuntimeError(f"Invalid JSONL {path}:{n}: {e}")
    return out

def run(cmd: Sequence[str], *, cwd: Optional[Path]=None, timeout: int=3600, env: Optional[Dict[str,str]]=None)->subprocess.CompletedProcess:
    e=os.environ.copy()
    paths=[str(OWW_REPO.resolve())] if OWW_REPO.exists() else []
    if e.get("PYTHONPATH"): paths.append(e["PYTHONPATH"])
    if paths: e["PYTHONPATH"]=os.pathsep.join(paths)
    if env: e.update({str(k):str(v) for k,v in env.items()})
    print("$"," ".join(map(str,cmd)))
    r=subprocess.run(list(map(str,cmd)),cwd=str((cwd or ROOT).resolve()),env=e,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE,timeout=timeout)
    if r.stdout.strip(): print(r.stdout[-12000:])
    if r.stderr.strip(): print("[stderr]\n"+r.stderr[-12000:])
    if r.returncode: raise RuntimeError(f"Command failed ({r.returncode}): {' '.join(map(str,cmd))}")
    return r

# Cell 14 — Generate Piper hard-negative speech
JOBS=DATA/"manifests"/"negative"/"negative_generation_jobs.jsonl"
PIPER_REPO=REPOS/"piper-sample-generator"
MODEL=MODELS/"piper"/"en_US-libritts_r-medium.pt"
RAW=DATA
if not JOBS.exists(): raise FileNotFoundError(f"{JOBS}; run Cell 12")
rows=read_jsonl(JOBS)
groups=defaultdict(list)
for r in rows: groups[r["text"]].append(r)
for gi,(text,items) in enumerate(sorted(groups.items()),1):
    missing=[r for r in items if not (RAW/r["raw_relative_path"]).exists()]
    if not missing: continue
    stage=CACHE/"cell_14"/hashlib.sha256(text.encode()).hexdigest()[:16]
    shutil.rmtree(stage,ignore_errors=True); stage.mkdir(parents=True)
    run([PY,"-m","piper_sample_generator",text,"--model",MODEL,"--max-samples",len(missing),
         "--batch-size",min(16,len(missing)),"--max-speakers",500,"--output-dir",stage],
        cwd=PIPER_REPO,timeout=max(1800,len(missing)*120),
        env={"PYTHONPATH":str(PIPER_REPO)})
    wavs=sorted(stage.rglob("*.wav"))
    if len(wavs)!=len(missing): raise RuntimeError(f"{text!r}: expected {len(missing)}, got {len(wavs)}")
    for src,j in zip(wavs,missing):
        dst=RAW/j["raw_relative_path"]; dst.parent.mkdir(parents=True,exist_ok=True); shutil.move(src,dst)
    shutil.rmtree(stage,ignore_errors=True)
    print(f"[{gi}/{len(groups)}] {text!r}: {len(missing)} generated")
print("Cell 14 complete.")

In [ ]:
from __future__ import annotations
import csv, datetime as dt, hashlib, inspect, json, math, os, random, shutil, subprocess, sys, textwrap, wave
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

try:
    CONFIG
except NameError:
    class _FallbackConfig:
        TRAINING_ROOT="/content/pingo_training"
        LOGS_DIR="/content/pingo_training/logs"
        CONFIG_DIR="/content/pingo_training/config"
        CACHE_DIR="/content/pingo_training/cache"
        DATA_DIR="/content/pingo_training/data"
        MODELS_DIR="/content/pingo_training/models"
        REPOS_DIR="/content/pingo_training/repos"
        ENV_DIR="/content/pingo-env"
        WAKE_WORD="PINGO"
        RANDOM_SEED=42
    CONFIG=_FallbackConfig()

ROOT=Path(CONFIG.TRAINING_ROOT)
LOGS=Path(CONFIG.LOGS_DIR)
CFG=Path(CONFIG.CONFIG_DIR)
CACHE=Path(CONFIG.CACHE_DIR)
DATA=Path(getattr(CONFIG,"DATA_DIR",ROOT/"data"))
MODELS=Path(getattr(CONFIG,"MODELS_DIR",ROOT/"models"))
REPOS=Path(getattr(CONFIG,"REPOS_DIR",ROOT/"repos"))
ENV=Path(CONFIG.ENV_DIR)
PY=ENV/"bin"/"python"
OWW_REPO=REPOS/"openWakeWord"
WAKE=str(getattr(CONFIG,"WAKE_WORD","PINGO")).strip()
SEED=int(getattr(CONFIG,"RANDOM_SEED",42))
for p in (LOGS,CFG,CACHE,DATA,MODELS): p.mkdir(parents=True,exist_ok=True)

def sha256(path: Path)->str:
    h=hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

def write_json(path: Path,obj: Any)->None:
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,sort_keys=True)+"\n",encoding="utf-8")
    tmp.replace(path)

def read_jsonl(path: Path)->List[Dict[str,Any]]:
    out=[]
    with path.open(encoding="utf-8") as f:
        for n,line in enumerate(f,1):
            if line.strip():
                try: out.append(json.loads(line))
                except Exception as e: raise RuntimeError(f"Invalid JSONL {path}:{n}: {e}")
    return out

def run(cmd: Sequence[str], *, cwd: Optional[Path]=None, timeout: int=3600, env: Optional[Dict[str,str]]=None)->subprocess.CompletedProcess:
    e=os.environ.copy()
    paths=[str(OWW_REPO.resolve())] if OWW_REPO.exists() else []
    if e.get("PYTHONPATH"): paths.append(e["PYTHONPATH"])
    if paths: e["PYTHONPATH"]=os.pathsep.join(paths)
    if env: e.update({str(k):str(v) for k,v in env.items()})
    print("$"," ".join(map(str,cmd)))
    r=subprocess.run(list(map(str,cmd)),cwd=str((cwd or ROOT).resolve()),env=e,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE,timeout=timeout)
    if r.stdout.strip(): print(r.stdout[-12000:])
    if r.stderr.strip(): print("[stderr]\n"+r.stderr[-12000:])
    if r.returncode: raise RuntimeError(f"Command failed ({r.returncode}): {' '.join(map(str,cmd))}")
    return r

# Cell 15 — Augmentation policy
POLICY={
 "sample_rate":16000,
 "clip_seconds":2.0,
 "positive_augmentations_per_source":4,
 "hard_negative_augmentations_per_source":2,
 "snr_db":[-5,0,5,10,15,20],
 "gain_db":[-10,-6,-3,0,3,6],
 "time_shift_ms":[-250,-150,-75,0,75,150,250],
 "speed_factors":[0.90,0.95,1.0,1.05,1.10],
 "reverb_probability":0.50,
 "background_probability":0.85,
 "music_probability":0.35,
 "speaker_echo_probability":0.45,
 "seed":SEED,
 "deployment_target":"Raspberry Pi 3B+ with ReSpeaker 2-Mic HAT and music playback",
}
write_json(CFG/"augmentation_policy.json",POLICY)
print(json.dumps(POLICY,indent=2))

In [ ]:
from __future__ import annotations
import csv, datetime as dt, hashlib, inspect, json, math, os, random, shutil, subprocess, sys, textwrap, wave
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

try:
    CONFIG
except NameError:
    class _FallbackConfig:
        TRAINING_ROOT="/content/pingo_training"
        LOGS_DIR="/content/pingo_training/logs"
        CONFIG_DIR="/content/pingo_training/config"
        CACHE_DIR="/content/pingo_training/cache"
        DATA_DIR="/content/pingo_training/data"
        MODELS_DIR="/content/pingo_training/models"
        REPOS_DIR="/content/pingo_training/repos"
        ENV_DIR="/content/pingo-env"
        WAKE_WORD="PINGO"
        RANDOM_SEED=42
    CONFIG=_FallbackConfig()

ROOT=Path(CONFIG.TRAINING_ROOT)
LOGS=Path(CONFIG.LOGS_DIR)
CFG=Path(CONFIG.CONFIG_DIR)
CACHE=Path(CONFIG.CACHE_DIR)
DATA=Path(getattr(CONFIG,"DATA_DIR",ROOT/"data"))
MODELS=Path(getattr(CONFIG,"MODELS_DIR",ROOT/"models"))
REPOS=Path(getattr(CONFIG,"REPOS_DIR",ROOT/"repos"))
ENV=Path(CONFIG.ENV_DIR)
PY=ENV/"bin"/"python"
OWW_REPO=REPOS/"openWakeWord"
WAKE=str(getattr(CONFIG,"WAKE_WORD","PINGO")).strip()
SEED=int(getattr(CONFIG,"RANDOM_SEED",42))
for p in (LOGS,CFG,CACHE,DATA,MODELS): p.mkdir(parents=True,exist_ok=True)

def sha256(path: Path)->str:
    h=hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

def write_json(path: Path,obj: Any)->None:
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,sort_keys=True)+"\n",encoding="utf-8")
    tmp.replace(path)

def read_jsonl(path: Path)->List[Dict[str,Any]]:
    out=[]
    with path.open(encoding="utf-8") as f:
        for n,line in enumerate(f,1):
            if line.strip():
                try: out.append(json.loads(line))
                except Exception as e: raise RuntimeError(f"Invalid JSONL {path}:{n}: {e}")
    return out

def run(cmd: Sequence[str], *, cwd: Optional[Path]=None, timeout: int=3600, env: Optional[Dict[str,str]]=None)->subprocess.CompletedProcess:
    e=os.environ.copy()
    paths=[str(OWW_REPO.resolve())] if OWW_REPO.exists() else []
    if e.get("PYTHONPATH"): paths.append(e["PYTHONPATH"])
    if paths: e["PYTHONPATH"]=os.pathsep.join(paths)
    if env: e.update({str(k):str(v) for k,v in env.items()})
    print("$"," ".join(map(str,cmd)))
    r=subprocess.run(list(map(str,cmd)),cwd=str((cwd or ROOT).resolve()),env=e,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE,timeout=timeout)
    if r.stdout.strip(): print(r.stdout[-12000:])
    if r.stderr.strip(): print("[stderr]\n"+r.stderr[-12000:])
    if r.returncode: raise RuntimeError(f"Command failed ({r.returncode}): {' '.join(map(str,cmd))}")
    return r

# Cell 16 — Build normalized background/noise pool
SRC=DATA/"external_negative"
OUT=DATA/"negative_pool_16khz"
MAN=DATA/"indexes"/"negative_pool.jsonl"
OUT.mkdir(parents=True,exist_ok=True); MAN.parent.mkdir(parents=True,exist_ok=True)
audio_ext={".wav",".mp3",".flac",".ogg",".m4a"}
sources=[p for p in SRC.rglob("*") if p.is_file() and p.suffix.lower() in audio_ext]
records=[]
for i,src in enumerate(sources):
    dst=OUT/f"negative_pool_{i:07d}.wav"
    if not dst.exists():
        run(["ffmpeg","-hide_banner","-loglevel","error","-y","-i",src,"-ar","16000","-ac","1","-c:a","pcm_s16le",dst],timeout=600)
    with wave.open(str(dst)) as w:
        dur=w.getnframes()/w.getframerate()
    if dur>=1.0: records.append({"id":dst.stem,"audio_path":str(dst),"source":str(src),"duration_seconds":dur,"sha256":sha256(dst)})
with MAN.open("w") as f:
    for r in records: f.write(json.dumps(r,sort_keys=True)+"\\n")
print(f"Prepared {len(records):,} background/music clips: {MAN}")

In [ ]:
from __future__ import annotations
import csv, datetime as dt, hashlib, inspect, json, math, os, random, shutil, subprocess, sys, textwrap, wave
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

try:
    CONFIG
except NameError:
    class _FallbackConfig:
        TRAINING_ROOT="/content/pingo_training"
        LOGS_DIR="/content/pingo_training/logs"
        CONFIG_DIR="/content/pingo_training/config"
        CACHE_DIR="/content/pingo_training/cache"
        DATA_DIR="/content/pingo_training/data"
        MODELS_DIR="/content/pingo_training/models"
        REPOS_DIR="/content/pingo_training/repos"
        ENV_DIR="/content/pingo-env"
        WAKE_WORD="PINGO"
        RANDOM_SEED=42
    CONFIG=_FallbackConfig()

ROOT=Path(CONFIG.TRAINING_ROOT)
LOGS=Path(CONFIG.LOGS_DIR)
CFG=Path(CONFIG.CONFIG_DIR)
CACHE=Path(CONFIG.CACHE_DIR)
DATA=Path(getattr(CONFIG,"DATA_DIR",ROOT/"data"))
MODELS=Path(getattr(CONFIG,"MODELS_DIR",ROOT/"models"))
REPOS=Path(getattr(CONFIG,"REPOS_DIR",ROOT/"repos"))
ENV=Path(CONFIG.ENV_DIR)
PY=ENV/"bin"/"python"
OWW_REPO=REPOS/"openWakeWord"
WAKE=str(getattr(CONFIG,"WAKE_WORD","PINGO")).strip()
SEED=int(getattr(CONFIG,"RANDOM_SEED",42))
for p in (LOGS,CFG,CACHE,DATA,MODELS): p.mkdir(parents=True,exist_ok=True)

def sha256(path: Path)->str:
    h=hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

def write_json(path: Path,obj: Any)->None:
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,sort_keys=True)+"\n",encoding="utf-8")
    tmp.replace(path)

def read_jsonl(path: Path)->List[Dict[str,Any]]:
    out=[]
    with path.open(encoding="utf-8") as f:
        for n,line in enumerate(f,1):
            if line.strip():
                try: out.append(json.loads(line))
                except Exception as e: raise RuntimeError(f"Invalid JSONL {path}:{n}: {e}")
    return out

def run(cmd: Sequence[str], *, cwd: Optional[Path]=None, timeout: int=3600, env: Optional[Dict[str,str]]=None)->subprocess.CompletedProcess:
    e=os.environ.copy()
    paths=[str(OWW_REPO.resolve())] if OWW_REPO.exists() else []
    if e.get("PYTHONPATH"): paths.append(e["PYTHONPATH"])
    if paths: e["PYTHONPATH"]=os.pathsep.join(paths)
    if env: e.update({str(k):str(v) for k,v in env.items()})
    print("$"," ".join(map(str,cmd)))
    r=subprocess.run(list(map(str,cmd)),cwd=str((cwd or ROOT).resolve()),env=e,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE,timeout=timeout)
    if r.stdout.strip(): print(r.stdout[-12000:])
    if r.stderr.strip(): print("[stderr]\n"+r.stderr[-12000:])
    if r.returncode: raise RuntimeError(f"Command failed ({r.returncode}): {' '.join(map(str,cmd))}")
    return r

# Cell 17 — Convert hard-negative speech to 16 kHz PCM16
JOBS=DATA/"manifests"/"negative"/"negative_generation_jobs.jsonl"
INDEX=DATA/"indexes"/"hard_negative_all.jsonl"
rows=read_jsonl(JOBS); out=[]
for n,j in enumerate(rows,1):
    src=DATA/j["raw_relative_path"]; dst=DATA/j["converted_relative_path"]
    if not src.exists(): raise FileNotFoundError(f"{src}; run Cell 14")
    dst.parent.mkdir(parents=True,exist_ok=True)
    if not dst.exists():
        run(["ffmpeg","-hide_banner","-loglevel","error","-y","-i",src,"-ar","16000","-ac","1","-c:a","pcm_s16le",dst],timeout=300)
    with wave.open(str(dst)) as w:
        if (w.getframerate(),w.getnchannels(),w.getsampwidth())!=(16000,1,2): raise RuntimeError(dst)
        dur=w.getnframes()/16000
    out.append({**j,"audio_path":str(dst),"duration_seconds":dur,"sha256":sha256(dst)})
INDEX.parent.mkdir(parents=True,exist_ok=True)
with INDEX.open("w") as f:
    for r in out:f.write(json.dumps(r,sort_keys=True)+"\\n")
print(f"Converted/indexed {len(out):,} hard negatives: {INDEX}")

In [ ]:
from __future__ import annotations
import csv, datetime as dt, hashlib, inspect, json, math, os, random, shutil, subprocess, sys, textwrap, wave
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

try:
    CONFIG
except NameError:
    class _FallbackConfig:
        TRAINING_ROOT="/content/pingo_training"
        LOGS_DIR="/content/pingo_training/logs"
        CONFIG_DIR="/content/pingo_training/config"
        CACHE_DIR="/content/pingo_training/cache"
        DATA_DIR="/content/pingo_training/data"
        MODELS_DIR="/content/pingo_training/models"
        REPOS_DIR="/content/pingo_training/repos"
        ENV_DIR="/content/pingo-env"
        WAKE_WORD="PINGO"
        RANDOM_SEED=42
    CONFIG=_FallbackConfig()

ROOT=Path(CONFIG.TRAINING_ROOT)
LOGS=Path(CONFIG.LOGS_DIR)
CFG=Path(CONFIG.CONFIG_DIR)
CACHE=Path(CONFIG.CACHE_DIR)
DATA=Path(getattr(CONFIG,"DATA_DIR",ROOT/"data"))
MODELS=Path(getattr(CONFIG,"MODELS_DIR",ROOT/"models"))
REPOS=Path(getattr(CONFIG,"REPOS_DIR",ROOT/"repos"))
ENV=Path(CONFIG.ENV_DIR)
PY=ENV/"bin"/"python"
OWW_REPO=REPOS/"openWakeWord"
WAKE=str(getattr(CONFIG,"WAKE_WORD","PINGO")).strip()
SEED=int(getattr(CONFIG,"RANDOM_SEED",42))
for p in (LOGS,CFG,CACHE,DATA,MODELS): p.mkdir(parents=True,exist_ok=True)

def sha256(path: Path)->str:
    h=hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

def write_json(path: Path,obj: Any)->None:
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,sort_keys=True)+"\n",encoding="utf-8")
    tmp.replace(path)

def read_jsonl(path: Path)->List[Dict[str,Any]]:
    out=[]
    with path.open(encoding="utf-8") as f:
        for n,line in enumerate(f,1):
            if line.strip():
                try: out.append(json.loads(line))
                except Exception as e: raise RuntimeError(f"Invalid JSONL {path}:{n}: {e}")
    return out

def run(cmd: Sequence[str], *, cwd: Optional[Path]=None, timeout: int=3600, env: Optional[Dict[str,str]]=None)->subprocess.CompletedProcess:
    e=os.environ.copy()
    paths=[str(OWW_REPO.resolve())] if OWW_REPO.exists() else []
    if e.get("PYTHONPATH"): paths.append(e["PYTHONPATH"])
    if paths: e["PYTHONPATH"]=os.pathsep.join(paths)
    if env: e.update({str(k):str(v) for k,v in env.items()})
    print("$"," ".join(map(str,cmd)))
    r=subprocess.run(list(map(str,cmd)),cwd=str((cwd or ROOT).resolve()),env=e,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE,timeout=timeout)
    if r.stdout.strip(): print(r.stdout[-12000:])
    if r.stderr.strip(): print("[stderr]\n"+r.stderr[-12000:])
    if r.returncode: raise RuntimeError(f"Command failed ({r.returncode}): {' '.join(map(str,cmd))}")
    return r

# Cell 18 — Create augmentation job manifest
POS=read_jsonl(DATA/"indexes"/"positive_all.jsonl")
HARD=read_jsonl(DATA/"indexes"/"hard_negative_all.jsonl")
POOL=read_jsonl(DATA/"indexes"/"negative_pool.jsonl")
POL=json.loads((CFG/"augmentation_policy.json").read_text())
OUT=DATA/"manifests"/"augmentation_jobs.jsonl"
rng=random.Random(SEED); jobs=[]
def add(rows,label,count):
    for r in rows:
        for k in range(count):
            bg=rng.choice(POOL)["audio_path"] if POOL else None
            aid=f"{r['sample_id']}_aug_{k:02d}"
            jobs.append({"augmentation_id":aid,"source_id":r["sample_id"],"label":label,"split":r["split"],
              "source_path":r["audio_path"],"background_path":bg,"snr_db":rng.choice(POL["snr_db"]),
              "gain_db":rng.choice(POL["gain_db"]),"shift_ms":rng.choice(POL["time_shift_ms"]),
              "speed_factor":rng.choice(POL["speed_factors"]),
              "output_path":str(DATA/"augmented"/label/r["split"]/f"{aid}.wav")})
add(POS,"positive",POL["positive_augmentations_per_source"])
add(HARD,"negative",POL["hard_negative_augmentations_per_source"])
OUT.parent.mkdir(parents=True,exist_ok=True)
with OUT.open("w") as f:
    for j in jobs:f.write(json.dumps(j,sort_keys=True)+"\\n")
print(f"Created {len(jobs):,} deterministic augmentation jobs: {OUT}")

In [ ]:
from __future__ import annotations
import csv, datetime as dt, hashlib, inspect, json, math, os, random, shutil, subprocess, sys, textwrap, wave
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

try:
    CONFIG
except NameError:
    class _FallbackConfig:
        TRAINING_ROOT="/content/pingo_training"
        LOGS_DIR="/content/pingo_training/logs"
        CONFIG_DIR="/content/pingo_training/config"
        CACHE_DIR="/content/pingo_training/cache"
        DATA_DIR="/content/pingo_training/data"
        MODELS_DIR="/content/pingo_training/models"
        REPOS_DIR="/content/pingo_training/repos"
        ENV_DIR="/content/pingo-env"
        WAKE_WORD="PINGO"
        RANDOM_SEED=42
    CONFIG=_FallbackConfig()

ROOT=Path(CONFIG.TRAINING_ROOT)
LOGS=Path(CONFIG.LOGS_DIR)
CFG=Path(CONFIG.CONFIG_DIR)
CACHE=Path(CONFIG.CACHE_DIR)
DATA=Path(getattr(CONFIG,"DATA_DIR",ROOT/"data"))
MODELS=Path(getattr(CONFIG,"MODELS_DIR",ROOT/"models"))
REPOS=Path(getattr(CONFIG,"REPOS_DIR",ROOT/"repos"))
ENV=Path(CONFIG.ENV_DIR)
PY=ENV/"bin"/"python"
OWW_REPO=REPOS/"openWakeWord"
WAKE=str(getattr(CONFIG,"WAKE_WORD","PINGO")).strip()
SEED=int(getattr(CONFIG,"RANDOM_SEED",42))
for p in (LOGS,CFG,CACHE,DATA,MODELS): p.mkdir(parents=True,exist_ok=True)

def sha256(path: Path)->str:
    h=hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

def write_json(path: Path,obj: Any)->None:
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,sort_keys=True)+"\n",encoding="utf-8")
    tmp.replace(path)

def read_jsonl(path: Path)->List[Dict[str,Any]]:
    out=[]
    with path.open(encoding="utf-8") as f:
        for n,line in enumerate(f,1):
            if line.strip():
                try: out.append(json.loads(line))
                except Exception as e: raise RuntimeError(f"Invalid JSONL {path}:{n}: {e}")
    return out

def run(cmd: Sequence[str], *, cwd: Optional[Path]=None, timeout: int=3600, env: Optional[Dict[str,str]]=None)->subprocess.CompletedProcess:
    e=os.environ.copy()
    paths=[str(OWW_REPO.resolve())] if OWW_REPO.exists() else []
    if e.get("PYTHONPATH"): paths.append(e["PYTHONPATH"])
    if paths: e["PYTHONPATH"]=os.pathsep.join(paths)
    if env: e.update({str(k):str(v) for k,v in env.items()})
    print("$"," ".join(map(str,cmd)))
    r=subprocess.run(list(map(str,cmd)),cwd=str((cwd or ROOT).resolve()),env=e,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE,timeout=timeout)
    if r.stdout.strip(): print(r.stdout[-12000:])
    if r.stderr.strip(): print("[stderr]\n"+r.stderr[-12000:])
    if r.returncode: raise RuntimeError(f"Command failed ({r.returncode}): {' '.join(map(str,cmd))}")
    return r

# Cell 19 — Execute deterministic augmentations
import numpy as np
from scipy.io import wavfile
from scipy.signal import resample_poly
JOBS=read_jsonl(DATA/"manifests"/"augmentation_jobs.jsonl")
def load(path):
    sr,x=wavfile.read(path)
    if sr!=16000: raise RuntimeError(f"{path}: {sr}")
    if x.ndim>1:x=x.mean(1)
    return x.astype(np.float32)/32768
def fit(x,n=32000):
    if len(x)>=n:return x[:n]
    return np.pad(x,(0,n-len(x)))
for i,j in enumerate(JOBS,1):
    dst=Path(j["output_path"]); dst.parent.mkdir(parents=True,exist_ok=True)
    if dst.exists(): continue
    x=load(j["source_path"])
    speed=float(j["speed_factor"])
    if speed!=1: x=resample_poly(x,100,int(round(100*speed)))
    shift=int(16000*int(j["shift_ms"])/1000)
    x=np.roll(x,shift); x=fit(x)
    x*=10**(float(j["gain_db"])/20)
    if j.get("background_path"):
        b=fit(load(j["background_path"]))
        xr=np.sqrt(np.mean(x*x)+1e-12); br=np.sqrt(np.mean(b*b)+1e-12)
        b*=xr/(br*10**(float(j["snr_db"])/20)+1e-12); x=x+b
    peak=np.max(np.abs(x))+1e-12
    if peak>0.98:x=x*(0.98/peak)
    wavfile.write(dst,16000,(x*32767).astype(np.int16))
    if i%500==0:print(i,"/",len(JOBS))
print("Augmentations complete.")

In [ ]:
from __future__ import annotations
import csv, datetime as dt, hashlib, inspect, json, math, os, random, shutil, subprocess, sys, textwrap, wave
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

try:
    CONFIG
except NameError:
    class _FallbackConfig:
        TRAINING_ROOT="/content/pingo_training"
        LOGS_DIR="/content/pingo_training/logs"
        CONFIG_DIR="/content/pingo_training/config"
        CACHE_DIR="/content/pingo_training/cache"
        DATA_DIR="/content/pingo_training/data"
        MODELS_DIR="/content/pingo_training/models"
        REPOS_DIR="/content/pingo_training/repos"
        ENV_DIR="/content/pingo-env"
        WAKE_WORD="PINGO"
        RANDOM_SEED=42
    CONFIG=_FallbackConfig()

ROOT=Path(CONFIG.TRAINING_ROOT)
LOGS=Path(CONFIG.LOGS_DIR)
CFG=Path(CONFIG.CONFIG_DIR)
CACHE=Path(CONFIG.CACHE_DIR)
DATA=Path(getattr(CONFIG,"DATA_DIR",ROOT/"data"))
MODELS=Path(getattr(CONFIG,"MODELS_DIR",ROOT/"models"))
REPOS=Path(getattr(CONFIG,"REPOS_DIR",ROOT/"repos"))
ENV=Path(CONFIG.ENV_DIR)
PY=ENV/"bin"/"python"
OWW_REPO=REPOS/"openWakeWord"
WAKE=str(getattr(CONFIG,"WAKE_WORD","PINGO")).strip()
SEED=int(getattr(CONFIG,"RANDOM_SEED",42))
for p in (LOGS,CFG,CACHE,DATA,MODELS): p.mkdir(parents=True,exist_ok=True)

def sha256(path: Path)->str:
    h=hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

def write_json(path: Path,obj: Any)->None:
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,sort_keys=True)+"\n",encoding="utf-8")
    tmp.replace(path)

def read_jsonl(path: Path)->List[Dict[str,Any]]:
    out=[]
    with path.open(encoding="utf-8") as f:
        for n,line in enumerate(f,1):
            if line.strip():
                try: out.append(json.loads(line))
                except Exception as e: raise RuntimeError(f"Invalid JSONL {path}:{n}: {e}")
    return out

def run(cmd: Sequence[str], *, cwd: Optional[Path]=None, timeout: int=3600, env: Optional[Dict[str,str]]=None)->subprocess.CompletedProcess:
    e=os.environ.copy()
    paths=[str(OWW_REPO.resolve())] if OWW_REPO.exists() else []
    if e.get("PYTHONPATH"): paths.append(e["PYTHONPATH"])
    if paths: e["PYTHONPATH"]=os.pathsep.join(paths)
    if env: e.update({str(k):str(v) for k,v in env.items()})
    print("$"," ".join(map(str,cmd)))
    r=subprocess.run(list(map(str,cmd)),cwd=str((cwd or ROOT).resolve()),env=e,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE,timeout=timeout)
    if r.stdout.strip(): print(r.stdout[-12000:])
    if r.stderr.strip(): print("[stderr]\n"+r.stderr[-12000:])
    if r.returncode: raise RuntimeError(f"Command failed ({r.returncode}): {' '.join(map(str,cmd))}")
    return r

# Cell 21 — Extract OpenWakeWord embeddings
import numpy as np
INDEX=read_jsonl(DATA/"indexes"/"training_all.jsonl")
OUT=DATA/"features"; OUT.mkdir(parents=True,exist_ok=True)
script=r"""
import json, os, numpy as np, openwakeword.utils
from scipy.io import wavfile
index=json.load(open(os.environ["INDEX_JSON"]))
F=openwakeword.utils.AudioFeatures()
features=[]; labels=[]; ids=[]; splits=[]
for i,r in enumerate(index):
    sr,x=wavfile.read(r["audio_path"])
    if x.ndim>1:x=x.mean(1)
    x=x.astype(np.int16)
    # Public API differs by release; inspect supported method dynamically.
    method=getattr(F,"embed_clips",None) or getattr(F,"get_embedding_from_file",None)
    if method is None: raise RuntimeError("AudioFeatures exposes no supported embedding method")
    try:
        z=method([r["audio_path"]]) if method.__name__=="embed_clips" else method(r["audio_path"])
    except TypeError:
        z=method(x)
    z=np.asarray(z,dtype=np.float32)
    features.append(z); labels.append(r["label"]); ids.append(r["sample_id"]); splits.append(r["split"])
np.savez_compressed(os.environ["OUT_NPZ"],features=np.array(features,dtype=object),labels=np.array(labels),ids=np.array(ids),splits=np.array(splits))
print(len(features))
"""
tmp=CACHE/"cell21_index.json"; tmp.write_text(json.dumps(INDEX))
run([PY,"-c",script],cwd=OWW_REPO,timeout=86400,env={"INDEX_JSON":tmp,"OUT_NPZ":OUT/"training_embeddings.npz"})
print("Embeddings:",OUT/"training_embeddings.npz")

In [ ]:
from __future__ import annotations
import csv, datetime as dt, hashlib, inspect, json, math, os, random, shutil, subprocess, sys, textwrap, wave
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

try:
    CONFIG
except NameError:
    class _FallbackConfig:
        TRAINING_ROOT="/content/pingo_training"
        LOGS_DIR="/content/pingo_training/logs"
        CONFIG_DIR="/content/pingo_training/config"
        CACHE_DIR="/content/pingo_training/cache"
        DATA_DIR="/content/pingo_training/data"
        MODELS_DIR="/content/pingo_training/models"
        REPOS_DIR="/content/pingo_training/repos"
        ENV_DIR="/content/pingo-env"
        WAKE_WORD="PINGO"
        RANDOM_SEED=42
    CONFIG=_FallbackConfig()

ROOT=Path(CONFIG.TRAINING_ROOT)
LOGS=Path(CONFIG.LOGS_DIR)
CFG=Path(CONFIG.CONFIG_DIR)
CACHE=Path(CONFIG.CACHE_DIR)
DATA=Path(getattr(CONFIG,"DATA_DIR",ROOT/"data"))
MODELS=Path(getattr(CONFIG,"MODELS_DIR",ROOT/"models"))
REPOS=Path(getattr(CONFIG,"REPOS_DIR",ROOT/"repos"))
ENV=Path(CONFIG.ENV_DIR)
PY=ENV/"bin"/"python"
OWW_REPO=REPOS/"openWakeWord"
WAKE=str(getattr(CONFIG,"WAKE_WORD","PINGO")).strip()
SEED=int(getattr(CONFIG,"RANDOM_SEED",42))
for p in (LOGS,CFG,CACHE,DATA,MODELS): p.mkdir(parents=True,exist_ok=True)

def sha256(path: Path)->str:
    h=hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

def write_json(path: Path,obj: Any)->None:
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,sort_keys=True)+"\n",encoding="utf-8")
    tmp.replace(path)

def read_jsonl(path: Path)->List[Dict[str,Any]]:
    out=[]
    with path.open(encoding="utf-8") as f:
        for n,line in enumerate(f,1):
            if line.strip():
                try: out.append(json.loads(line))
                except Exception as e: raise RuntimeError(f"Invalid JSONL {path}:{n}: {e}")
    return out

def run(cmd: Sequence[str], *, cwd: Optional[Path]=None, timeout: int=3600, env: Optional[Dict[str,str]]=None)->subprocess.CompletedProcess:
    e=os.environ.copy()
    paths=[str(OWW_REPO.resolve())] if OWW_REPO.exists() else []
    if e.get("PYTHONPATH"): paths.append(e["PYTHONPATH"])
    if paths: e["PYTHONPATH"]=os.pathsep.join(paths)
    if env: e.update({str(k):str(v) for k,v in env.items()})
    print("$"," ".join(map(str,cmd)))
    r=subprocess.run(list(map(str,cmd)),cwd=str((cwd or ROOT).resolve()),env=e,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE,timeout=timeout)
    if r.stdout.strip(): print(r.stdout[-12000:])
    if r.stderr.strip(): print("[stderr]\n"+r.stderr[-12000:])
    if r.returncode: raise RuntimeError(f"Command failed ({r.returncode}): {' '.join(map(str,cmd))}")
    return r

# Cell 22 — Normalize embedding shapes and create tensors
import numpy as np
SRC=DATA/"features"/"training_embeddings.npz"
OUT=DATA/"features"/"training_tensors.npz"
d=np.load(SRC,allow_pickle=True)
shapes=Counter(tuple(np.asarray(x).shape) for x in d["features"])
target=shapes.most_common(1)[0][0]
X=[]; y=[]; ids=[]; splits=[]
for z,label,sid,split in zip(d["features"],d["labels"],d["ids"],d["splits"]):
    a=np.asarray(z,dtype=np.float32)
    if a.shape==target:X.append(a);y.append(label);ids.append(sid);splits.append(split)
X=np.stack(X)
np.savez_compressed(OUT,X=X,y=np.asarray(y,dtype=np.float32),ids=np.asarray(ids),splits=np.asarray(splits))
write_json(CFG/"cell_22_feature_shape.json",{"accepted_shape":target,"accepted":len(X),"all_shapes":{str(k):v for k,v in shapes.items()}})
print("Tensor shape:",X.shape)

In [ ]:
from __future__ import annotations
import csv, datetime as dt, hashlib, inspect, json, math, os, random, shutil, subprocess, sys, textwrap, wave
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

try:
    CONFIG
except NameError:
    class _FallbackConfig:
        TRAINING_ROOT="/content/pingo_training"
        LOGS_DIR="/content/pingo_training/logs"
        CONFIG_DIR="/content/pingo_training/config"
        CACHE_DIR="/content/pingo_training/cache"
        DATA_DIR="/content/pingo_training/data"
        MODELS_DIR="/content/pingo_training/models"
        REPOS_DIR="/content/pingo_training/repos"
        ENV_DIR="/content/pingo-env"
        WAKE_WORD="PINGO"
        RANDOM_SEED=42
    CONFIG=_FallbackConfig()

ROOT=Path(CONFIG.TRAINING_ROOT)
LOGS=Path(CONFIG.LOGS_DIR)
CFG=Path(CONFIG.CONFIG_DIR)
CACHE=Path(CONFIG.CACHE_DIR)
DATA=Path(getattr(CONFIG,"DATA_DIR",ROOT/"data"))
MODELS=Path(getattr(CONFIG,"MODELS_DIR",ROOT/"models"))
REPOS=Path(getattr(CONFIG,"REPOS_DIR",ROOT/"repos"))
ENV=Path(CONFIG.ENV_DIR)
PY=ENV/"bin"/"python"
OWW_REPO=REPOS/"openWakeWord"
WAKE=str(getattr(CONFIG,"WAKE_WORD","PINGO")).strip()
SEED=int(getattr(CONFIG,"RANDOM_SEED",42))
for p in (LOGS,CFG,CACHE,DATA,MODELS): p.mkdir(parents=True,exist_ok=True)

def sha256(path: Path)->str:
    h=hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

def write_json(path: Path,obj: Any)->None:
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,sort_keys=True)+"\n",encoding="utf-8")
    tmp.replace(path)

def read_jsonl(path: Path)->List[Dict[str,Any]]:
    out=[]
    with path.open(encoding="utf-8") as f:
        for n,line in enumerate(f,1):
            if line.strip():
                try: out.append(json.loads(line))
                except Exception as e: raise RuntimeError(f"Invalid JSONL {path}:{n}: {e}")
    return out

def run(cmd: Sequence[str], *, cwd: Optional[Path]=None, timeout: int=3600, env: Optional[Dict[str,str]]=None)->subprocess.CompletedProcess:
    e=os.environ.copy()
    paths=[str(OWW_REPO.resolve())] if OWW_REPO.exists() else []
    if e.get("PYTHONPATH"): paths.append(e["PYTHONPATH"])
    if paths: e["PYTHONPATH"]=os.pathsep.join(paths)
    if env: e.update({str(k):str(v) for k,v in env.items()})
    print("$"," ".join(map(str,cmd)))
    r=subprocess.run(list(map(str,cmd)),cwd=str((cwd or ROOT).resolve()),env=e,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE,timeout=timeout)
    if r.stdout.strip(): print(r.stdout[-12000:])
    if r.stderr.strip(): print("[stderr]\n"+r.stderr[-12000:])
    if r.returncode: raise RuntimeError(f"Command failed ({r.returncode}): {' '.join(map(str,cmd))}")
    return r

# Cell 23 — Train PyTorch wake-word classifier
import numpy as np, torch
from torch import nn
D=np.load(DATA/"features"/"training_tensors.npz")
X=torch.from_numpy(D["X"]).float(); y=torch.from_numpy(D["y"]).float().view(-1,1); splits=D["splits"]
train=np.where(splits=="train")[0]; val=np.where(splits=="validation")[0]
class Net(nn.Module):
    def __init__(self,shape):
        super().__init__(); n=int(np.prod(shape))
        self.net=nn.Sequential(nn.Flatten(),nn.Linear(n,64),nn.LayerNorm(64),nn.ReLU(),nn.Dropout(.15),
                               nn.Linear(64,32),nn.LayerNorm(32),nn.ReLU(),nn.Linear(32,1))
    def forward(self,x):return self.net(x)
device="cuda" if torch.cuda.is_available() else "cpu"; model=Net(X.shape[1:]).to(device)
opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4)
lossfn=nn.BCEWithLogitsLoss(pos_weight=torch.tensor([2.0],device=device))
best=1e9; patience=8; bad=0; hist=[]
for epoch in range(60):
    model.train(); perm=torch.randperm(len(train)); losses=[]
    for s in range(0,len(train),256):
        idx=torch.tensor(train)[perm[s:s+256]]; xb=X[idx].to(device); yb=y[idx].to(device)
        opt.zero_grad(); loss=lossfn(model(xb),yb); loss.backward(); opt.step(); losses.append(loss.item())
    model.eval()
    with torch.no_grad():vl=lossfn(model(X[val].to(device)),y[val].to(device)).item()
    hist.append({"epoch":epoch+1,"train_loss":sum(losses)/len(losses),"val_loss":vl}); print(hist[-1])
    if vl<best:
        best=vl;bad=0;torch.save({"state_dict":model.state_dict(),"shape":tuple(X.shape[1:]),"history":hist},MODELS/"pingo_best.pt")
    else:
        bad+=1
        if bad>=patience:break
write_json(CFG/"cell_23_training_history.json",{"history":hist,"best_val_loss":best})
print("Best checkpoint:",MODELS/"pingo_best.pt")

In [ ]:
from __future__ import annotations
import csv, datetime as dt, hashlib, inspect, json, math, os, random, shutil, subprocess, sys, textwrap, wave
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

try:
    CONFIG
except NameError:
    class _FallbackConfig:
        TRAINING_ROOT="/content/pingo_training"
        LOGS_DIR="/content/pingo_training/logs"
        CONFIG_DIR="/content/pingo_training/config"
        CACHE_DIR="/content/pingo_training/cache"
        DATA_DIR="/content/pingo_training/data"
        MODELS_DIR="/content/pingo_training/models"
        REPOS_DIR="/content/pingo_training/repos"
        ENV_DIR="/content/pingo-env"
        WAKE_WORD="PINGO"
        RANDOM_SEED=42
    CONFIG=_FallbackConfig()

ROOT=Path(CONFIG.TRAINING_ROOT)
LOGS=Path(CONFIG.LOGS_DIR)
CFG=Path(CONFIG.CONFIG_DIR)
CACHE=Path(CONFIG.CACHE_DIR)
DATA=Path(getattr(CONFIG,"DATA_DIR",ROOT/"data"))
MODELS=Path(getattr(CONFIG,"MODELS_DIR",ROOT/"models"))
REPOS=Path(getattr(CONFIG,"REPOS_DIR",ROOT/"repos"))
ENV=Path(CONFIG.ENV_DIR)
PY=ENV/"bin"/"python"
OWW_REPO=REPOS/"openWakeWord"
WAKE=str(getattr(CONFIG,"WAKE_WORD","PINGO")).strip()
SEED=int(getattr(CONFIG,"RANDOM_SEED",42))
for p in (LOGS,CFG,CACHE,DATA,MODELS): p.mkdir(parents=True,exist_ok=True)

def sha256(path: Path)->str:
    h=hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

def write_json(path: Path,obj: Any)->None:
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,sort_keys=True)+"\n",encoding="utf-8")
    tmp.replace(path)

def read_jsonl(path: Path)->List[Dict[str,Any]]:
    out=[]
    with path.open(encoding="utf-8") as f:
        for n,line in enumerate(f,1):
            if line.strip():
                try: out.append(json.loads(line))
                except Exception as e: raise RuntimeError(f"Invalid JSONL {path}:{n}: {e}")
    return out

def run(cmd: Sequence[str], *, cwd: Optional[Path]=None, timeout: int=3600, env: Optional[Dict[str,str]]=None)->subprocess.CompletedProcess:
    e=os.environ.copy()
    paths=[str(OWW_REPO.resolve())] if OWW_REPO.exists() else []
    if e.get("PYTHONPATH"): paths.append(e["PYTHONPATH"])
    if paths: e["PYTHONPATH"]=os.pathsep.join(paths)
    if env: e.update({str(k):str(v) for k,v in env.items()})
    print("$"," ".join(map(str,cmd)))
    r=subprocess.run(list(map(str,cmd)),cwd=str((cwd or ROOT).resolve()),env=e,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE,timeout=timeout)
    if r.stdout.strip(): print(r.stdout[-12000:])
    if r.stderr.strip(): print("[stderr]\n"+r.stderr[-12000:])
    if r.returncode: raise RuntimeError(f"Command failed ({r.returncode}): {' '.join(map(str,cmd))}")
    return r

# Cell 24 — Evaluate checkpoint and save scores
import numpy as np, torch
from torch import nn
D=np.load(DATA/"features"/"training_tensors.npz")
X=torch.from_numpy(D["X"]).float(); y=D["y"]; splits=D["splits"]; ids=D["ids"]
ck=torch.load(MODELS/"pingo_best.pt",map_location="cpu")
class Net(nn.Module):
    def __init__(self,shape):
        super().__init__();n=int(np.prod(shape));self.net=nn.Sequential(nn.Flatten(),nn.Linear(n,64),nn.LayerNorm(64),nn.ReLU(),nn.Dropout(.15),nn.Linear(64,32),nn.LayerNorm(32),nn.ReLU(),nn.Linear(32,1))
    def forward(self,x):return self.net(x)
m=Net(ck["shape"]);m.load_state_dict(ck["state_dict"]);m.eval()
with torch.no_grad(): scores=torch.sigmoid(m(X)).numpy().ravel()
OUT=DATA/"features"/"evaluation_scores.npz";np.savez_compressed(OUT,scores=scores,labels=y,splits=splits,ids=ids)
print("Saved",OUT,"score range",scores.min(),scores.max())

In [ ]:
from __future__ import annotations
import csv, datetime as dt, hashlib, inspect, json, math, os, random, shutil, subprocess, sys, textwrap, wave
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

try:
    CONFIG
except NameError:
    class _FallbackConfig:
        TRAINING_ROOT="/content/pingo_training"
        LOGS_DIR="/content/pingo_training/logs"
        CONFIG_DIR="/content/pingo_training/config"
        CACHE_DIR="/content/pingo_training/cache"
        DATA_DIR="/content/pingo_training/data"
        MODELS_DIR="/content/pingo_training/models"
        REPOS_DIR="/content/pingo_training/repos"
        ENV_DIR="/content/pingo-env"
        WAKE_WORD="PINGO"
        RANDOM_SEED=42
    CONFIG=_FallbackConfig()

ROOT=Path(CONFIG.TRAINING_ROOT)
LOGS=Path(CONFIG.LOGS_DIR)
CFG=Path(CONFIG.CONFIG_DIR)
CACHE=Path(CONFIG.CACHE_DIR)
DATA=Path(getattr(CONFIG,"DATA_DIR",ROOT/"data"))
MODELS=Path(getattr(CONFIG,"MODELS_DIR",ROOT/"models"))
REPOS=Path(getattr(CONFIG,"REPOS_DIR",ROOT/"repos"))
ENV=Path(CONFIG.ENV_DIR)
PY=ENV/"bin"/"python"
OWW_REPO=REPOS/"openWakeWord"
WAKE=str(getattr(CONFIG,"WAKE_WORD","PINGO")).strip()
SEED=int(getattr(CONFIG,"RANDOM_SEED",42))
for p in (LOGS,CFG,CACHE,DATA,MODELS): p.mkdir(parents=True,exist_ok=True)

def sha256(path: Path)->str:
    h=hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

def write_json(path: Path,obj: Any)->None:
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,sort_keys=True)+"\n",encoding="utf-8")
    tmp.replace(path)

def read_jsonl(path: Path)->List[Dict[str,Any]]:
    out=[]
    with path.open(encoding="utf-8") as f:
        for n,line in enumerate(f,1):
            if line.strip():
                try: out.append(json.loads(line))
                except Exception as e: raise RuntimeError(f"Invalid JSONL {path}:{n}: {e}")
    return out

def run(cmd: Sequence[str], *, cwd: Optional[Path]=None, timeout: int=3600, env: Optional[Dict[str,str]]=None)->subprocess.CompletedProcess:
    e=os.environ.copy()
    paths=[str(OWW_REPO.resolve())] if OWW_REPO.exists() else []
    if e.get("PYTHONPATH"): paths.append(e["PYTHONPATH"])
    if paths: e["PYTHONPATH"]=os.pathsep.join(paths)
    if env: e.update({str(k):str(v) for k,v in env.items()})
    print("$"," ".join(map(str,cmd)))
    r=subprocess.run(list(map(str,cmd)),cwd=str((cwd or ROOT).resolve()),env=e,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE,timeout=timeout)
    if r.stdout.strip(): print(r.stdout[-12000:])
    if r.stderr.strip(): print("[stderr]\n"+r.stderr[-12000:])
    if r.returncode: raise RuntimeError(f"Command failed ({r.returncode}): {' '.join(map(str,cmd))}")
    return r

# Cell 25 — Calibrate threshold
import numpy as np
D=np.load(DATA/"features"/"evaluation_scores.npz")
mask=D["splits"]=="validation"; s=D["scores"][mask]; y=D["labels"][mask]
best=None
for t in np.linspace(.01,.99,199):
    pred=s>=t; tp=((pred==1)&(y==1)).sum();fn=((pred==0)&(y==1)).sum();fp=((pred==1)&(y==0)).sum();tn=((pred==0)&(y==0)).sum()
    recall=tp/max(1,tp+fn); fpr=fp/max(1,fp+tn); precision=tp/max(1,tp+fp)
    objective=recall-8*fpr
    row={"threshold":float(t),"recall":float(recall),"precision":float(precision),"false_positive_rate":float(fpr),"objective":float(objective)}
    if best is None or row["objective"]>best["objective"]:best=row
write_json(CFG/"pingo_threshold.json",best)
print(best)

In [ ]:
from __future__ import annotations
import csv, datetime as dt, hashlib, inspect, json, math, os, random, shutil, subprocess, sys, textwrap, wave
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

try:
    CONFIG
except NameError:
    class _FallbackConfig:
        TRAINING_ROOT="/content/pingo_training"
        LOGS_DIR="/content/pingo_training/logs"
        CONFIG_DIR="/content/pingo_training/config"
        CACHE_DIR="/content/pingo_training/cache"
        DATA_DIR="/content/pingo_training/data"
        MODELS_DIR="/content/pingo_training/models"
        REPOS_DIR="/content/pingo_training/repos"
        ENV_DIR="/content/pingo-env"
        WAKE_WORD="PINGO"
        RANDOM_SEED=42
    CONFIG=_FallbackConfig()

ROOT=Path(CONFIG.TRAINING_ROOT)
LOGS=Path(CONFIG.LOGS_DIR)
CFG=Path(CONFIG.CONFIG_DIR)
CACHE=Path(CONFIG.CACHE_DIR)
DATA=Path(getattr(CONFIG,"DATA_DIR",ROOT/"data"))
MODELS=Path(getattr(CONFIG,"MODELS_DIR",ROOT/"models"))
REPOS=Path(getattr(CONFIG,"REPOS_DIR",ROOT/"repos"))
ENV=Path(CONFIG.ENV_DIR)
PY=ENV/"bin"/"python"
OWW_REPO=REPOS/"openWakeWord"
WAKE=str(getattr(CONFIG,"WAKE_WORD","PINGO")).strip()
SEED=int(getattr(CONFIG,"RANDOM_SEED",42))
for p in (LOGS,CFG,CACHE,DATA,MODELS): p.mkdir(parents=True,exist_ok=True)

def sha256(path: Path)->str:
    h=hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

def write_json(path: Path,obj: Any)->None:
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,sort_keys=True)+"\n",encoding="utf-8")
    tmp.replace(path)

def read_jsonl(path: Path)->List[Dict[str,Any]]:
    out=[]
    with path.open(encoding="utf-8") as f:
        for n,line in enumerate(f,1):
            if line.strip():
                try: out.append(json.loads(line))
                except Exception as e: raise RuntimeError(f"Invalid JSONL {path}:{n}: {e}")
    return out

def run(cmd: Sequence[str], *, cwd: Optional[Path]=None, timeout: int=3600, env: Optional[Dict[str,str]]=None)->subprocess.CompletedProcess:
    e=os.environ.copy()
    paths=[str(OWW_REPO.resolve())] if OWW_REPO.exists() else []
    if e.get("PYTHONPATH"): paths.append(e["PYTHONPATH"])
    if paths: e["PYTHONPATH"]=os.pathsep.join(paths)
    if env: e.update({str(k):str(v) for k,v in env.items()})
    print("$"," ".join(map(str,cmd)))
    r=subprocess.run(list(map(str,cmd)),cwd=str((cwd or ROOT).resolve()),env=e,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE,timeout=timeout)
    if r.stdout.strip(): print(r.stdout[-12000:])
    if r.stderr.strip(): print("[stderr]\n"+r.stderr[-12000:])
    if r.returncode: raise RuntimeError(f"Command failed ({r.returncode}): {' '.join(map(str,cmd))}")
    return r

# Cell 26 — Export classifier to ONNX
import numpy as np, torch
from torch import nn
ck=torch.load(MODELS/"pingo_best.pt",map_location="cpu")
class Net(nn.Module):
    def __init__(self,shape):
        super().__init__();n=int(np.prod(shape));self.net=nn.Sequential(nn.Flatten(),nn.Linear(n,64),nn.LayerNorm(64),nn.ReLU(),nn.Dropout(.15),nn.Linear(64,32),nn.LayerNorm(32),nn.ReLU(),nn.Linear(32,1),nn.Sigmoid())
    def forward(self,x):return self.net(x)
m=Net(ck["shape"]); sd={k:v for k,v in ck["state_dict"].items()};m.load_state_dict(sd,strict=False);m.eval()
dummy=torch.zeros((1,*ck["shape"]),dtype=torch.float32)
out=MODELS/"pingo.onnx"
torch.onnx.export(m,dummy,out,input_names=["input"],output_names=["score"],dynamic_axes={"input":{0:"batch"},"score":{0:"batch"}},opset_version=17)
print(out,out.stat().st_size,sha256(out))

In [ ]:
from __future__ import annotations
import csv, datetime as dt, hashlib, inspect, json, math, os, random, shutil, subprocess, sys, textwrap, wave
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

try:
    CONFIG
except NameError:
    class _FallbackConfig:
        TRAINING_ROOT="/content/pingo_training"
        LOGS_DIR="/content/pingo_training/logs"
        CONFIG_DIR="/content/pingo_training/config"
        CACHE_DIR="/content/pingo_training/cache"
        DATA_DIR="/content/pingo_training/data"
        MODELS_DIR="/content/pingo_training/models"
        REPOS_DIR="/content/pingo_training/repos"
        ENV_DIR="/content/pingo-env"
        WAKE_WORD="PINGO"
        RANDOM_SEED=42
    CONFIG=_FallbackConfig()

ROOT=Path(CONFIG.TRAINING_ROOT)
LOGS=Path(CONFIG.LOGS_DIR)
CFG=Path(CONFIG.CONFIG_DIR)
CACHE=Path(CONFIG.CACHE_DIR)
DATA=Path(getattr(CONFIG,"DATA_DIR",ROOT/"data"))
MODELS=Path(getattr(CONFIG,"MODELS_DIR",ROOT/"models"))
REPOS=Path(getattr(CONFIG,"REPOS_DIR",ROOT/"repos"))
ENV=Path(CONFIG.ENV_DIR)
PY=ENV/"bin"/"python"
OWW_REPO=REPOS/"openWakeWord"
WAKE=str(getattr(CONFIG,"WAKE_WORD","PINGO")).strip()
SEED=int(getattr(CONFIG,"RANDOM_SEED",42))
for p in (LOGS,CFG,CACHE,DATA,MODELS): p.mkdir(parents=True,exist_ok=True)

def sha256(path: Path)->str:
    h=hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

def write_json(path: Path,obj: Any)->None:
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,sort_keys=True)+"\n",encoding="utf-8")
    tmp.replace(path)

def read_jsonl(path: Path)->List[Dict[str,Any]]:
    out=[]
    with path.open(encoding="utf-8") as f:
        for n,line in enumerate(f,1):
            if line.strip():
                try: out.append(json.loads(line))
                except Exception as e: raise RuntimeError(f"Invalid JSONL {path}:{n}: {e}")
    return out

def run(cmd: Sequence[str], *, cwd: Optional[Path]=None, timeout: int=3600, env: Optional[Dict[str,str]]=None)->subprocess.CompletedProcess:
    e=os.environ.copy()
    paths=[str(OWW_REPO.resolve())] if OWW_REPO.exists() else []
    if e.get("PYTHONPATH"): paths.append(e["PYTHONPATH"])
    if paths: e["PYTHONPATH"]=os.pathsep.join(paths)
    if env: e.update({str(k):str(v) for k,v in env.items()})
    print("$"," ".join(map(str,cmd)))
    r=subprocess.run(list(map(str,cmd)),cwd=str((cwd or ROOT).resolve()),env=e,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE,timeout=timeout)
    if r.stdout.strip(): print(r.stdout[-12000:])
    if r.stderr.strip(): print("[stderr]\n"+r.stderr[-12000:])
    if r.returncode: raise RuntimeError(f"Command failed ({r.returncode}): {' '.join(map(str,cmd))}")
    return r

# Cell 27 — Validate ONNX parity
import numpy as np, onnxruntime as ort, torch
from torch import nn
D=np.load(DATA/"features"/"training_tensors.npz"); X=D["X"][:256].astype(np.float32)
sess=ort.InferenceSession(str(MODELS/"pingo.onnx"),providers=["CPUExecutionProvider"])
onnx=sess.run(None,{sess.get_inputs()[0].name:X})[0].ravel()
# Compare with saved evaluation scores for same prefix.
S=np.load(DATA/"features"/"evaluation_scores.npz")["scores"][:len(X)]
diff=np.abs(onnx-S)
report={"count":len(X),"max_abs_difference":float(diff.max()),"mean_abs_difference":float(diff.mean()),"passed":bool(diff.max()<1e-4)}
write_json(CFG/"cell_27_onnx_parity.json",report)
print(report)
if not report["passed"]:raise RuntimeError("ONNX parity failed")

In [ ]:
from __future__ import annotations
import csv, datetime as dt, hashlib, inspect, json, math, os, random, shutil, subprocess, sys, textwrap, wave
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

try:
    CONFIG
except NameError:
    class _FallbackConfig:
        TRAINING_ROOT="/content/pingo_training"
        LOGS_DIR="/content/pingo_training/logs"
        CONFIG_DIR="/content/pingo_training/config"
        CACHE_DIR="/content/pingo_training/cache"
        DATA_DIR="/content/pingo_training/data"
        MODELS_DIR="/content/pingo_training/models"
        REPOS_DIR="/content/pingo_training/repos"
        ENV_DIR="/content/pingo-env"
        WAKE_WORD="PINGO"
        RANDOM_SEED=42
    CONFIG=_FallbackConfig()

ROOT=Path(CONFIG.TRAINING_ROOT)
LOGS=Path(CONFIG.LOGS_DIR)
CFG=Path(CONFIG.CONFIG_DIR)
CACHE=Path(CONFIG.CACHE_DIR)
DATA=Path(getattr(CONFIG,"DATA_DIR",ROOT/"data"))
MODELS=Path(getattr(CONFIG,"MODELS_DIR",ROOT/"models"))
REPOS=Path(getattr(CONFIG,"REPOS_DIR",ROOT/"repos"))
ENV=Path(CONFIG.ENV_DIR)
PY=ENV/"bin"/"python"
OWW_REPO=REPOS/"openWakeWord"
WAKE=str(getattr(CONFIG,"WAKE_WORD","PINGO")).strip()
SEED=int(getattr(CONFIG,"RANDOM_SEED",42))
for p in (LOGS,CFG,CACHE,DATA,MODELS): p.mkdir(parents=True,exist_ok=True)

def sha256(path: Path)->str:
    h=hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

def write_json(path: Path,obj: Any)->None:
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,sort_keys=True)+"\n",encoding="utf-8")
    tmp.replace(path)

def read_jsonl(path: Path)->List[Dict[str,Any]]:
    out=[]
    with path.open(encoding="utf-8") as f:
        for n,line in enumerate(f,1):
            if line.strip():
                try: out.append(json.loads(line))
                except Exception as e: raise RuntimeError(f"Invalid JSONL {path}:{n}: {e}")
    return out

def run(cmd: Sequence[str], *, cwd: Optional[Path]=None, timeout: int=3600, env: Optional[Dict[str,str]]=None)->subprocess.CompletedProcess:
    e=os.environ.copy()
    paths=[str(OWW_REPO.resolve())] if OWW_REPO.exists() else []
    if e.get("PYTHONPATH"): paths.append(e["PYTHONPATH"])
    if paths: e["PYTHONPATH"]=os.pathsep.join(paths)
    if env: e.update({str(k):str(v) for k,v in env.items()})
    print("$"," ".join(map(str,cmd)))
    r=subprocess.run(list(map(str,cmd)),cwd=str((cwd or ROOT).resolve()),env=e,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE,timeout=timeout)
    if r.stdout.strip(): print(r.stdout[-12000:])
    if r.stderr.strip(): print("[stderr]\n"+r.stderr[-12000:])
    if r.returncode: raise RuntimeError(f"Command failed ({r.returncode}): {' '.join(map(str,cmd))}")
    return r

# Cell 28 — Build Raspberry Pi deployment package
PKG=ROOT/"deployment"/"pingo_raspberry_pi"
shutil.rmtree(PKG,ignore_errors=True);PKG.mkdir(parents=True)
for src in [MODELS/"pingo.onnx",CFG/"pingo_threshold.json"]:
    if not src.exists():raise FileNotFoundError(src)
    shutil.copy2(src,PKG/src.name)
runtime=r"""from pathlib import Path
import json, numpy as np, onnxruntime as ort
class PingoClassifier:
    def __init__(self,root):
        root=Path(root); self.threshold=json.loads((root/"pingo_threshold.json").read_text())["threshold"]
        self.session=ort.InferenceSession(str(root/"pingo.onnx"),providers=["CPUExecutionProvider"])
        self.input_name=self.session.get_inputs()[0].name
    def score(self,embedding):
        x=np.asarray(embedding,dtype=np.float32)
        if x.ndim==2:x=x[None,...]
        return float(self.session.run(None,{self.input_name:x})[0].reshape(-1)[0])
    def detected(self,embedding):return self.score(embedding)>=self.threshold
"""
(PKG/"pingo_classifier.py").write_text(runtime)
write_json(PKG/"manifest.json",{"model_sha256":sha256(PKG/"pingo.onnx"),"threshold":json.loads((CFG/"pingo_threshold.json").read_text()),"python":"3.10+","dependencies":["numpy","onnxruntime"]})
shutil.make_archive(str(ROOT/"deployment"/"pingo_raspberry_pi"),"zip",PKG)
print("Package:",ROOT/"deployment"/"pingo_raspberry_pi.zip")


In [ ]:
from __future__ import annotations
import csv, datetime as dt, hashlib, inspect, json, math, os, random, shutil, subprocess, sys, textwrap, wave
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

try:
    CONFIG
except NameError:
    class _FallbackConfig:
        TRAINING_ROOT="/content/pingo_training"
        LOGS_DIR="/content/pingo_training/logs"
        CONFIG_DIR="/content/pingo_training/config"
        CACHE_DIR="/content/pingo_training/cache"
        DATA_DIR="/content/pingo_training/data"
        MODELS_DIR="/content/pingo_training/models"
        REPOS_DIR="/content/pingo_training/repos"
        ENV_DIR="/content/pingo-env"
        WAKE_WORD="PINGO"
        RANDOM_SEED=42
    CONFIG=_FallbackConfig()

ROOT=Path(CONFIG.TRAINING_ROOT)
LOGS=Path(CONFIG.LOGS_DIR)
CFG=Path(CONFIG.CONFIG_DIR)
CACHE=Path(CONFIG.CACHE_DIR)
DATA=Path(getattr(CONFIG,"DATA_DIR",ROOT/"data"))
MODELS=Path(getattr(CONFIG,"MODELS_DIR",ROOT/"models"))
REPOS=Path(getattr(CONFIG,"REPOS_DIR",ROOT/"repos"))
ENV=Path(CONFIG.ENV_DIR)
PY=ENV/"bin"/"python"
OWW_REPO=REPOS/"openWakeWord"
WAKE=str(getattr(CONFIG,"WAKE_WORD","PINGO")).strip()
SEED=int(getattr(CONFIG,"RANDOM_SEED",42))
for p in (LOGS,CFG,CACHE,DATA,MODELS): p.mkdir(parents=True,exist_ok=True)

def sha256(path: Path)->str:
    h=hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

def write_json(path: Path,obj: Any)->None:
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,sort_keys=True)+"\n",encoding="utf-8")
    tmp.replace(path)

def read_jsonl(path: Path)->List[Dict[str,Any]]:
    out=[]
    with path.open(encoding="utf-8") as f:
        for n,line in enumerate(f,1):
            if line.strip():
                try: out.append(json.loads(line))
                except Exception as e: raise RuntimeError(f"Invalid JSONL {path}:{n}: {e}")
    return out

def run(cmd: Sequence[str], *, cwd: Optional[Path]=None, timeout: int=3600, env: Optional[Dict[str,str]]=None)->subprocess.CompletedProcess:
    e=os.environ.copy()
    paths=[str(OWW_REPO.resolve())] if OWW_REPO.exists() else []
    if e.get("PYTHONPATH"): paths.append(e["PYTHONPATH"])
    if paths: e["PYTHONPATH"]=os.pathsep.join(paths)
    if env: e.update({str(k):str(v) for k,v in env.items()})
    print("$"," ".join(map(str,cmd)))
    r=subprocess.run(list(map(str,cmd)),cwd=str((cwd or ROOT).resolve()),env=e,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE,timeout=timeout)
    if r.stdout.strip(): print(r.stdout[-12000:])
    if r.stderr.strip(): print("[stderr]\n"+r.stderr[-12000:])
    if r.returncode: raise RuntimeError(f"Command failed ({r.returncode}): {' '.join(map(str,cmd))}")
    return r

# Cell 29 — CPU latency and stability benchmark
import numpy as np, onnxruntime as ort, time
D=np.load(DATA/"features"/"training_tensors.npz");X=D["X"][:100].astype(np.float32)
s=ort.InferenceSession(str(MODELS/"pingo.onnx"),providers=["CPUExecutionProvider"]);name=s.get_inputs()[0].name
for _ in range(20):s.run(None,{name:X[:1]})
times=[]
for i in range(1000):
    x=X[i%len(X):i%len(X)+1];t=time.perf_counter();s.run(None,{name:x});times.append((time.perf_counter()-t)*1000)
a=np.asarray(times)
report={"runs":len(a),"mean_ms":float(a.mean()),"p50_ms":float(np.percentile(a,50)),"p95_ms":float(np.percentile(a,95)),"p99_ms":float(np.percentile(a,99)),"max_ms":float(a.max())}
write_json(CFG/"cell_29_cpu_benchmark.json",report);print(report)

In [ ]:
from __future__ import annotations
import csv, datetime as dt, hashlib, inspect, json, math, os, random, shutil, subprocess, sys, textwrap, wave
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

try:
    CONFIG
except NameError:
    class _FallbackConfig:
        TRAINING_ROOT="/content/pingo_training"
        LOGS_DIR="/content/pingo_training/logs"
        CONFIG_DIR="/content/pingo_training/config"
        CACHE_DIR="/content/pingo_training/cache"
        DATA_DIR="/content/pingo_training/data"
        MODELS_DIR="/content/pingo_training/models"
        REPOS_DIR="/content/pingo_training/repos"
        ENV_DIR="/content/pingo-env"
        WAKE_WORD="PINGO"
        RANDOM_SEED=42
    CONFIG=_FallbackConfig()

ROOT=Path(CONFIG.TRAINING_ROOT)
LOGS=Path(CONFIG.LOGS_DIR)
CFG=Path(CONFIG.CONFIG_DIR)
CACHE=Path(CONFIG.CACHE_DIR)
DATA=Path(getattr(CONFIG,"DATA_DIR",ROOT/"data"))
MODELS=Path(getattr(CONFIG,"MODELS_DIR",ROOT/"models"))
REPOS=Path(getattr(CONFIG,"REPOS_DIR",ROOT/"repos"))
ENV=Path(CONFIG.ENV_DIR)
PY=ENV/"bin"/"python"
OWW_REPO=REPOS/"openWakeWord"
WAKE=str(getattr(CONFIG,"WAKE_WORD","PINGO")).strip()
SEED=int(getattr(CONFIG,"RANDOM_SEED",42))
for p in (LOGS,CFG,CACHE,DATA,MODELS): p.mkdir(parents=True,exist_ok=True)

def sha256(path: Path)->str:
    h=hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""): h.update(b)
    return h.hexdigest()

def write_json(path: Path,obj: Any)->None:
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,sort_keys=True)+"\n",encoding="utf-8")
    tmp.replace(path)

def read_jsonl(path: Path)->List[Dict[str,Any]]:
    out=[]
    with path.open(encoding="utf-8") as f:
        for n,line in enumerate(f,1):
            if line.strip():
                try: out.append(json.loads(line))
                except Exception as e: raise RuntimeError(f"Invalid JSONL {path}:{n}: {e}")
    return out

def run(cmd: Sequence[str], *, cwd: Optional[Path]=None, timeout: int=3600, env: Optional[Dict[str,str]]=None)->subprocess.CompletedProcess:
    e=os.environ.copy()
    paths=[str(OWW_REPO.resolve())] if OWW_REPO.exists() else []
    if e.get("PYTHONPATH"): paths.append(e["PYTHONPATH"])
    if paths: e["PYTHONPATH"]=os.pathsep.join(paths)
    if env: e.update({str(k):str(v) for k,v in env.items()})
    print("$"," ".join(map(str,cmd)))
    r=subprocess.run(list(map(str,cmd)),cwd=str((cwd or ROOT).resolve()),env=e,text=True,stdout=subprocess.PIPE,stderr=subprocess.PIPE,timeout=timeout)
    if r.stdout.strip(): print(r.stdout[-12000:])
    if r.stderr.strip(): print("[stderr]\n"+r.stderr[-12000:])
    if r.returncode: raise RuntimeError(f"Command failed ({r.returncode}): {' '.join(map(str,cmd))}")
    return r

# Cell 30 — Final audit and archive
required=[
 DATA/"indexes"/"positive_all.jsonl",
 DATA/"indexes"/"hard_negative_all.jsonl",
 DATA/"indexes"/"training_all.jsonl",
 DATA/"features"/"training_tensors.npz",
 MODELS/"pingo_best.pt",
 MODELS/"pingo.onnx",
 CFG/"pingo_threshold.json",
 CFG/"cell_27_onnx_parity.json",
 CFG/"cell_29_cpu_benchmark.json",
 ROOT/"deployment"/"pingo_raspberry_pi.zip",
]
missing=[str(p) for p in required if not p.exists()]
audit={"stage":"Cell 30 — Final audit","created_at_utc":dt.datetime.now(dt.timezone.utc).isoformat(),
       "passed":not missing,"missing":missing,"artifacts":[{"path":str(p),"size_bytes":p.stat().st_size,"sha256":sha256(p)} for p in required if p.exists()]}
write_json(ROOT/"reports"/"final_training_audit.json",audit)
if missing:raise RuntimeError("Missing final artifacts:\\n"+"\\n".join(missing))
archive_root=ROOT/"release"/"pingo_training_release";shutil.rmtree(archive_root,ignore_errors=True);archive_root.mkdir(parents=True)
for p in required:
    dst=archive_root/p.name;shutil.copy2(p,dst)
shutil.copy2(ROOT/"reports"/"final_training_audit.json",archive_root/"final_training_audit.json")
zip_path=Path(shutil.make_archive(str(ROOT/"release"/"pingo_training_release"),"zip",archive_root))
print("="*88);print("CELLS 1–30 PIPELINE AUDIT PASSED");print("Release:",zip_path);print("Audit:",ROOT/"reports"/"final_training_audit.json");print("="*88)